In [ ]:
import hashlib
import json
import re
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    mean_squared_error,
    ndcg_score,
)

INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
OUTPUT_DIR = WORKING / "rr_pressure_100"
COMPARISON_DIR = WORKING / "rr_model_comparison"
CACHE_INPUT = None
SEED = 42
THREADS = 4
WICKET_VALUE = 4.0
FOLD_FRACTIONS = (0.26, 0.45, 0.63, 0.81)
ROLE_IDS = {"batter": "striker_id", "bowler": "bowler_id"}
ROLE_NAMES = {"batter": "striker_name", "bowler": "bowler_name"}
PRIMARY_SCENARIOS = (
    "match_demand", "close_finish", "limited_support",
    "strong_opposition", "sustained_attack", "form_or_stakes",
)

PERCENTILES = (0.75, 0.45)
RECENT_MATCHES = 5
ACCEPT_INFERRED_OTHER_AS_NO_BALL = False
INCLUDE_OTHER_T20_COMPETITIONS = False
DERIVATION = {
    "rating_reference_events": 60, "rating_reference_matches": 3,
    "rating_shrink_events": 120, "rating_shrink_matches": 5,
    "bowler_run_weight": 0.65, "batter_run_weight": 0.60,
    "variation_min_context_events": 12, "form_min_events": 30,
    "form_slump_run_equivalent_gap": 0.15,
    "pressure_candidate_threshold": 65, "pressure_min_weight_coverage": 0.70,
    "collapse_wickets_in_12_legal_balls": 2,
    "attack_runs_in_6_bowler_events": 12,
    "attack_boundaries_in_6_bowler_events": 2,
    "response_opportunities": 6, "trajectory_min_matches": 5,
    "matchup_min_prior_events": 12,
}
SOURCES = {
    "ball_by_ball": {
        "match_id", "innings", "innings_ball_num", "striker_id", "bowler_id",
        "delivery_style", "stroke", "extras_type", "runs",
    },
    "matches": {"match_id", "match_date", "comp_id", "win_team_id"},
    "competitions": {"comp_id", "comp_name", "match_format", "season"},
    "teams": {"comp_id", "team_id", "team_name"},
    "playing_eleven": {"match_id", "team_id", "player_id", "is_played"},
    "competition_team_player_mapping": {
        "comp_id", "team_id", "player_id", "player_name",
    },
}
TOP_K = 10

DATA_CODE = "35b9bd4a05a7086cf96c42655a52bf3e8edf283d6534d529d4b9dbd879f5a350"

SCENARIO_MINIMUM = (18, 3)
CHECKLIST = {
    "raw_inputs": "Read the six raw files and build pre-ball features.",
    "scenario_data": "Build the existing 50 solo scenarios per role.",
    "forecasts": "Benchmark CatBoost, LightGBM, XGBoost and their ensemble on the same folds.",
    "exports": "Check solo, combination and repeat scoring and write every top 10.",
}

ROLE_SCENARIO_COUNTS = {"batter": 50, "bowler": 50}
TOTAL_SCENARIOS = sum(ROLE_SCENARIO_COUNTS.values())
RATE_SCENARIO = "raise_chase_requirement"

DIRECT_SCENARIOS = {
    "recovery_after_slow_powerplay", "death_over_acceleration",
    "new_ball_after_early_wickets", "acceleration_after_dot_buildup",
    "rebuild_after_wicket", "rising_chase_rate", "nineteenth_over",
}


MIN_OVERALL_ATTEMPTS = 60
MIN_OVERALL_MATCHES = 6

CUMULATIVE_FAMILIES = {
    "batter": {
        "Pacing and innings construction": [1, 8, 14, 21, 22, 24, 25, 26],
        "Chase finish conversion": [2, 16, 17, 18, 19, 20, 32],
        "Wicket preservation and tail management": [3, 9, 11, 13, 23, 27, 28, 29, 30, 31],
        "Scoring recovery and acceleration": [5, 7, 12, 33, 34, 35, 36],
        "Adapting to bowling matchups": [4, 10, 15, 37, 38, 39, 40, 41, 42, 43, 44, 45],
        "Batting through form slumps and knockouts": [6, 46, 47, 48, 49, 50],
    },
    "bowler": {
        "Target defence and required-rate squeeze": [1, 17, 24, 25, 26, 27, 33],
        "Closing overs and last-ball execution": [2, 7, 13, 16, 18, 19, 20, 21, 22, 37],
        "Strike bowling and partnership disruption": [8, 12, 23, 31, 32],
        "Spell management and damage control": [3, 5, 9, 11, 14, 15, 28, 29, 30, 34, 35, 36, 38],
        "Containing batter threats and handedness": [4, 10, 39, 40, 41, 42, 43, 44, 45],
        "Bowling through form slumps and knockouts": [6, 46, 47, 48, 49, 50],
    },
}

CUMULATIVE_CONTEXT_FIELDS = [
    "x_team_runs_before", "x_wickets_lost_before", "x_legal_balls_used",
    "x_balls_remaining", "x_current_run_rate", "x_runs_required", "x_required_run_rate",
    "x_score_minus_trajectory_proxy", "x_historical_chase_success_probability",
    "x_striker_runs_before", "x_striker_nonwide_deliveries_before",
    "x_partnership_runs_before", "x_partnership_legal_balls_before",
    "x_legal_balls_since_wicket", "x_consecutive_team_dot_events",
    "x_runs_previous_6_legal_ball_window", "x_wickets_previous_12_legal_ball_window",
    "x_remaining_top3_strength_score100_proxy", "x_other_bowling_top3_strength_proxy",
    "x_batting_hand", "x_bowling_arm", "x_bowling_style", "x_pace_or_spin",
    "x_phase", "x_competition_group", "x_venue_id", "x_knockout", "innings",
]
CUMULATIVE_OPPOSITION_FIELDS = {
    "batter": [
        "x_batter_opponent_ability10", "x_batter_opponent_form",
        "x_batter_opponent_live_threat", "x_bowler_mean_speed_prior",
        "x_bowler_effective_repertoire_categories_proxy",
        "x_bowler_previous_6_events_boundaries",
    ],
    "bowler": [
        "x_bowler_opponent_ability10", "x_bowler_opponent_form",
        "x_bowler_opponent_live_threat", "x_batter_effective_repertoire_categories_proxy",
        "x_bowler_legal_balls_before",
    ],
}
CUMULATIVE_OUTCOMES = {
    "batter": ("y_batter_runs", "y_striker_dismissed"),
    "bowler": ("y_bowler_runs", "y_bowler_wicket"),
}

CUMULATIVE_RECIPES = {'batter': [{'number': 1,
             'key': 'match_demand',
             'metric': 'Innings-adjusted scoring',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 2,
             'key': 'close_finish',
             'metric': 'Chase progress',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 3,
             'key': 'limited_support',
             'metric': 'Innings-adjusted scoring',
             'runs': 50,
             'wicket': 50,
             'discipline': 0},
            {'number': 4,
             'key': 'strong_opposition',
             'metric': 'Context-adjusted bat runs',
             'runs': 65,
             'wicket': 35,
             'discipline': 0},
            {'number': 5,
             'key': 'sustained_attack',
             'metric': 'Scoring recovery',
             'runs': 70,
             'wicket': 30,
             'discipline': 0},
            {'number': 6,
             'key': 'form_or_stakes',
             'metric': 'Innings-adjusted scoring',
             'runs': 65,
             'wicket': 35,
             'discipline': 0},
            {'number': 7,
             'key': 'recovery_after_slow_powerplay',
             'metric': 'Scoring recovery',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 8,
             'key': 'death_over_acceleration',
             'metric': 'Innings-adjusted scoring',
             'runs': 85,
             'wicket': 15,
             'discipline': 0},
            {'number': 9,
             'key': 'new_ball_after_early_wickets',
             'metric': 'Innings-adjusted scoring',
             'runs': 55,
             'wicket': 45,
             'discipline': 0},
            {'number': 10,
             'key': 'switch_pace_spin',
             'metric': 'Context-adjusted bat runs',
             'runs': 70,
             'wicket': 30,
             'discipline': 0},
            {'number': 11,
             'key': 'set_with_fresh_partner',
             'metric': 'Innings-adjusted scoring',
             'runs': 65,
             'wicket': 35,
             'discipline': 0},
            {'number': 12,
             'key': 'acceleration_after_dot_buildup',
             'metric': 'Scoring recovery',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 13,
             'key': 'rebuild_after_wicket',
             'metric': 'Scoring recovery',
             'runs': 55,
             'wicket': 45,
             'discipline': 0},
            {'number': 14,
             'key': 'rising_chase_rate',
             'metric': 'Chase progress',
             'runs': 80,
             'wicket': 20,
             'discipline': 0},
            {'number': 15,
             'key': 'fresh_bowler_post_powerplay',
             'metric': 'Context-adjusted bat runs',
             'runs': 70,
             'wicket': 30,
             'discipline': 0},
            {'number': 16,
             'key': 'steep_chase_with_two_to_four_overs_left',
             'metric': 'Chase progress',
             'runs': 85,
             'wicket': 15,
             'discipline': 0},
            {'number': 17,
             'key': 'reachable_last_over_chase',
             'metric': 'Chase progress',
             'runs': 80,
             'wicket': 20,
             'discipline': 0},
            {'number': 18,
             'key': 'last_over_starts_needing_13_or_more',
             'metric': 'Chase progress',
             'runs': 85,
             'wicket': 15,
             'discipline': 0},
            {'number': 19,
             'key': 'boundary_needed_off_the_last_ball',
             'metric': 'Chase progress',
             'runs': 100,
             'wicket': 0,
             'discipline': 0},
            {'number': 20,
             'key': 'low_runs_needed_with_the_last_wicket',
             'metric': 'Chase progress',
             'runs': 45,
             'wicket': 55,
             'discipline': 0},
            {'number': 21,
             'key': 'large_chase_at_the_opening',
             'metric': 'Chase progress',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 22,
             'key': 'comfortable_chase_turning_against_the_batting_side',
             'metric': 'Chase progress',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 23,
             'key': 'ahead_of_the_rate_but_running_out_of_wickets',
             'metric': 'Chase progress',
             'runs': 45,
             'wicket': 55,
             'discipline': 0},
            {'number': 24,
             'key': 'steep_required_rate_jump_in_the_middle_overs',
             'metric': 'Chase progress',
             'runs': 85,
             'wicket': 15,
             'discipline': 0},
            {'number': 25,
             'key': 'first_innings_well_below_its_scoring_trajectory',
             'metric': 'Context-adjusted bat runs',
             'runs': 70,
             'wicket': 30,
             'discipline': 0},
            {'number': 26,
             'key': 'death_overs_with_wickets_in_hand_but_a_scoring_deficit',
             'metric': 'Context-adjusted bat runs',
             'runs': 85,
             'wicket': 15,
             'discipline': 0},
            {'number': 27,
             'key': 'new_batter_walking_into_a_middle_overs_collapse',
             'metric': 'Innings-adjusted scoring',
             'runs': 50,
             'wicket': 50,
             'discipline': 0},
            {'number': 28,
             'key': 'established_batter_loses_the_other_established_batter',
             'metric': 'Scoring recovery',
             'runs': 55,
             'wicket': 45,
             'discipline': 0},
            {'number': 29,
             'key': 'chasing_with_the_tail_and_no_deeper_support',
             'metric': 'Chase progress',
             'runs': 45,
             'wicket': 55,
             'discipline': 0},
            {'number': 30,
             'key': 'last_batting_pair_still_needs_a_meaningful_chase',
             'metric': 'Chase progress',
             'runs': 45,
             'wicket': 55,
             'discipline': 0},
            {'number': 31,
             'key': 'double_strike_before_the_innings_settles',
             'metric': 'Innings-adjusted scoring',
             'runs': 45,
             'wicket': 55,
             'discipline': 0},
            {'number': 32,
             'key': 'new_finisher_exposed_late_in_a_demanding_chase',
             'metric': 'Chase progress',
             'runs': 85,
             'wicket': 15,
             'discipline': 0},
            {'number': 33,
             'key': 'team_scoring_squeeze_after_the_powerplay',
             'metric': 'Scoring recovery',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 34,
             'key': 'dots_at_the_end_of_a_tight_chase',
             'metric': 'Scoring recovery',
             'runs': 80,
             'wicket': 20,
             'discipline': 0},
            {'number': 35,
             'key': 'two_over_drought_after_an_early_wicket',
             'metric': 'Scoring recovery',
             'runs': 60,
             'wicket': 40,
             'discipline': 0},
            {'number': 36,
             'key': 'established_batter_stuck_below_a_run_a_ball',
             'metric': 'Scoring recovery',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 37,
             'key': 'high_ability_pace_in_a_demanding_innings',
             'metric': 'Context-adjusted bat runs',
             'runs': 70,
             'wicket': 30,
             'discipline': 0},
            {'number': 38,
             'key': 'high_ability_off_spin_in_a_demanding_innings',
             'metric': 'Context-adjusted bat runs',
             'runs': 70,
             'wicket': 30,
             'discipline': 0},
            {'number': 39,
             'key': 'high_ability_wrist_spin_in_a_demanding_innings',
             'metric': 'Context-adjusted bat runs',
             'runs': 70,
             'wicket': 30,
             'discipline': 0},
            {'number': 40,
             'key': 'high_ability_left_arm_orthodox_under_pressure',
             'metric': 'Context-adjusted bat runs',
             'runs': 70,
             'wicket': 30,
             'discipline': 0},
            {'number': 41,
             'key': 'high_velocity_pace_after_an_early_wicket',
             'metric': 'Context-adjusted bat runs',
             'runs': 60,
             'wicket': 40,
             'discipline': 0},
            {'number': 42,
             'key': 'proven_bowler_also_dangerous_in_this_spell',
             'metric': 'Context-adjusted bat runs',
             'runs': 65,
             'wicket': 35,
             'discipline': 0},
            {'number': 43,
             'key': 'facing_a_bowler_who_has_repeatedly_dismissed_you',
             'metric': 'Context-adjusted bat runs',
             'runs': 55,
             'wicket': 45,
             'discipline': 0},
            {'number': 44,
             'key': 'facing_a_style_that_has_suppressed_your_scoring',
             'metric': 'Context-adjusted bat runs',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 45,
             'key': 'variable_repertoire_bowler_in_an_urgent_chase',
             'metric': 'Chase progress',
             'runs': 80,
             'wicket': 20,
             'discipline': 0},
            {'number': 46,
             'key': 'personal_batting_slump_with_a_demanding_chase',
             'metric': 'Chase progress',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 47,
             'key': 'scoring_burden_in_the_knockout_powerplay',
             'metric': 'Innings-adjusted scoring',
             'runs': 60,
             'wicket': 40,
             'discipline': 0},
            {'number': 48,
             'key': 'knockout_chase_at_the_death',
             'metric': 'Chase progress',
             'runs': 85,
             'wicket': 15,
             'discipline': 0},
            {'number': 49,
             'key': 'knockout_first_innings_recovery_after_a_collapse',
             'metric': 'Scoring recovery',
             'runs': 50,
             'wicket': 50,
             'discipline': 0},
            {'number': 50,
             'key': 'knockout_batter_facing_a_proven_bowler_in_form',
             'metric': 'Context-adjusted bat runs',
             'runs': 65,
             'wicket': 35,
             'discipline': 0}],
 'bowler': [{'number': 1,
             'key': 'match_demand',
             'metric': 'Innings-adjusted run control',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 2,
             'key': 'close_finish',
             'metric': 'Chase squeeze',
             'runs': 75,
             'wicket': 20,
             'discipline': 5},
            {'number': 3,
             'key': 'limited_support',
             'metric': 'Innings-adjusted run control',
             'runs': 65,
             'wicket': 35,
             'discipline': 0},
            {'number': 4,
             'key': 'strong_opposition',
             'metric': 'Run suppression',
             'runs': 60,
             'wicket': 40,
             'discipline': 0},
            {'number': 5,
             'key': 'sustained_attack',
             'metric': 'Control after a setback',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 6,
             'key': 'form_or_stakes',
             'metric': 'Innings-adjusted run control',
             'runs': 70,
             'wicket': 30,
             'discipline': 0},
            {'number': 7,
             'key': 'first_innings_death',
             'metric': 'Run suppression',
             'runs': 80,
             'wicket': 20,
             'discipline': 0},
            {'number': 8,
             'key': 'settled_partnership',
             'metric': 'Run suppression',
             'runs': 45,
             'wicket': 55,
             'discipline': 0},
            {'number': 9,
             'key': 'return_spell',
             'metric': 'Control after a setback',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 10,
             'key': 'handedness_switch',
             'metric': 'Run suppression',
             'runs': 60,
             'wicket': 30,
             'discipline': 10},
            {'number': 11,
             'key': 'after_illegal_delivery',
             'metric': 'Run suppression',
             'runs': 55,
             'wicket': 15,
             'discipline': 30},
            {'number': 12,
             'key': 'after_own_wicket',
             'metric': 'Run suppression',
             'runs': 60,
             'wicket': 40,
             'discipline': 0},
            {'number': 13,
             'key': 'finish_tight_over',
             'metric': 'Run suppression',
             'runs': 70,
             'wicket': 20,
             'discipline': 10},
            {'number': 14,
             'key': 'final_quota',
             'metric': 'Innings-adjusted run control',
             'runs': 65,
             'wicket': 35,
             'discipline': 0},
            {'number': 15,
             'key': 'start_after_expensive_over',
             'metric': 'Control after a setback',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 16,
             'key': 'nineteenth_over',
             'metric': 'Innings-adjusted run control',
             'runs': 70,
             'wicket': 20,
             'discipline': 10},
            {'number': 17,
             'key': 'raise_chase_requirement',
             'metric': 'Chase squeeze',
             'runs': 85,
             'wicket': 15,
             'discipline': 0},
            {'number': 18,
             'key': 'defending_7–12_at_the_start_of_the_last_over',
             'metric': 'Chase squeeze',
             'runs': 70,
             'wicket': 20,
             'discipline': 10},
            {'number': 19,
             'key': 'defending_six_or_fewer_in_the_last_over',
             'metric': 'Chase squeeze',
             'runs': 60,
             'wicket': 30,
             'discipline': 10},
            {'number': 20,
             'key': 'last_legal_ball_with_the_match_within_reach',
             'metric': 'Chase squeeze',
             'runs': 70,
             'wicket': 15,
             'discipline': 15},
            {'number': 21,
             'key': 'defending_a_narrow_cushion_in_overs_17–18',
             'metric': 'Chase squeeze',
             'runs': 75,
             'wicket': 20,
             'discipline': 5},
            {'number': 22,
             'key': 'nineteenth_over_to_a_set_striker_in_a_balanced_chase',
             'metric': 'Chase squeeze',
             'runs': 65,
             'wicket': 25,
             'discipline': 10},
            {'number': 23,
             'key': 'only_one_wicket_needed_but_the_chase_is_live',
             'metric': 'Chase squeeze',
             'runs': 45,
             'wicket': 55,
             'discipline': 0},
            {'number': 24,
             'key': 'small_target_and_no_early_defensive_cushion',
             'metric': 'Chase squeeze',
             'runs': 70,
             'wicket': 30,
             'discipline': 0},
            {'number': 25,
             'key': 'chase_shifting_decisively_toward_the_batters',
             'metric': 'Chase squeeze',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 26,
             'key': 'below_seven_required_with_batting_depth_intact',
             'metric': 'Chase squeeze',
             'runs': 70,
             'wicket': 30,
             'discipline': 0},
            {'number': 27,
             'key': 'required_rate_falls_sharply_over_two_overs',
             'metric': 'Chase squeeze',
             'runs': 80,
             'wicket': 20,
             'discipline': 0},
            {'number': 28,
             'key': 'first_innings_accelerating_above_trajectory_in_the_middle',
             'metric': 'Run suppression',
             'runs': 65,
             'wicket': 35,
             'discipline': 0},
            {'number': 29,
             'key': 'explosive_opposition_powerplay',
             'metric': 'Run suppression',
             'runs': 60,
             'wicket': 40,
             'discipline': 0},
            {'number': 30,
             'key': 'few_credible_bowling_alternatives_left',
             'metric': 'Innings-adjusted run control',
             'runs': 65,
             'wicket': 35,
             'discipline': 0},
            {'number': 31,
             'key': 'first_innings_wicket_drought_while_runs_build',
             'metric': 'Run suppression',
             'runs': 45,
             'wicket': 55,
             'discipline': 0},
            {'number': 32,
             'key': 'protect_the_hard_won_wicket_against_a_strong_replacement',
             'metric': 'Run suppression',
             'runs': 55,
             'wicket': 45,
             'discipline': 0},
            {'number': 33,
             'key': 'final_quota_under_an_urgent_chase_defense',
             'metric': 'Chase squeeze',
             'runs': 70,
             'wicket': 20,
             'discipline': 10},
            {'number': 34,
             'key': 'own_spell_after_an_expensive_start',
             'metric': 'Control after a setback',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 35,
             'key': 'following_your_own_conceded_boundary_in_a_live_chase',
             'metric': 'Chase squeeze',
             'runs': 70,
             'wicket': 20,
             'discipline': 10},
            {'number': 36,
             'key': 'taking_over_after_a_multi_over_surge',
             'metric': 'Control after a setback',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 37,
             'key': 'protecting_a_low_cost_over_against_a_strong_finisher',
             'metric': 'Run suppression',
             'runs': 75,
             'wicket': 15,
             'discipline': 10},
            {'number': 38,
             'key': 'bowling_after_a_runaway_boundary_over_starts',
             'metric': 'Control after a setback',
             'runs': 65,
             'wicket': 35,
             'discipline': 0},
            {'number': 39,
             'key': 'bowling_to_a_proven,_in_form_hitter_already_scoring_freely',
             'metric': 'Run suppression',
             'runs': 60,
             'wicket': 40,
             'discipline': 0},
            {'number': 40,
             'key': 'defending_late_against_a_dangerous_left_hander',
             'metric': 'Chase squeeze',
             'runs': 70,
             'wicket': 25,
             'discipline': 5},
            {'number': 41,
             'key': 'defending_late_against_a_dangerous_right_hander',
             'metric': 'Chase squeeze',
             'runs': 70,
             'wicket': 25,
             'discipline': 5},
            {'number': 42,
             'key': 'bowling_to_an_opponent_who_has_dominated_you',
             'metric': 'Run suppression',
             'runs': 65,
             'wicket': 35,
             'discipline': 0},
            {'number': 43,
             'key': 'unfavorable_handedness_against_a_quality_batter',
             'metric': 'Run suppression',
             'runs': 70,
             'wicket': 30,
             'discipline': 0},
            {'number': 44,
             'key': 'bowling_to_a_two_sided_batting_threat',
             'metric': 'Run suppression',
             'runs': 60,
             'wicket': 40,
             'discipline': 0},
            {'number': 45,
             'key': 'high_range_shotmaker_attacking_a_defendable_chase',
             'metric': 'Chase squeeze',
             'runs': 75,
             'wicket': 25,
             'discipline': 0},
            {'number': 46,
             'key': 'bowling_slump_while_the_defense_is_still_in_play',
             'metric': 'Chase squeeze',
             'runs': 70,
             'wicket': 25,
             'discipline': 5},
            {'number': 47,
             'key': 'knockout_powerplay_getting_away_from_the_defense',
             'metric': 'Run suppression',
             'runs': 55,
             'wicket': 45,
             'discipline': 0},
            {'number': 48,
             'key': 'knockout_chase_alive_in_the_death_overs',
             'metric': 'Chase squeeze',
             'runs': 70,
             'wicket': 20,
             'discipline': 10},
            {'number': 49,
             'key': 'knockout_first_innings_with_a_strong_finish_looming',
             'metric': 'Run suppression',
             'runs': 70,
             'wicket': 25,
             'discipline': 5},
            {'number': 50,
             'key': 'knockout_pressure_against_an_in_form_quality_striker',
             'metric': 'Run suppression',
             'runs': 60,
             'wicket': 40,
             'discipline': 0}]}

CUMULATIVE_SCALES = {"runs": 0.75, "wickets": 0.10, "discipline": 0.065}

BASE_MODELS = ("catboost", "lightgbm", "xgboost")
RANKING_MODELS = (*BASE_MODELS, "ensemble")
ENSEMBLE_WEIGHTS = {"catboost": 1 / 3, "lightgbm": 1 / 3, "xgboost": 1 / 3}
GPU_MODE = "auto"
SUBMISSION_MODEL = "ensemble"
MODEL_ROUNDS = 110
COMBO_CAP = 8.0
REPEAT_CAP = 6.0
REPEAT_MATCH_MIN_ATTEMPTS = 4
REPEAT_MATCH_MIN_EDGE = 0.05
REPEAT_SATURATION = 5.0

COMBINATIONS = {
    "batter": [
        ("demand_against_quality", "Match demand against strong bowling", (1, 4), "Innings-adjusted scoring", 70, 30, 0, 2),
        ("collapse_against_quality", "Limited batting support against strong bowling", (3, 4), "Innings-adjusted scoring", 55, 45, 0, 2),
        ("wickets_and_scoring_squeeze", "Collapse alongside a scoring squeeze", (3, 5), "Innings-adjusted scoring", 55, 45, 0, 2),
        ("close_chase_quality", "Close chase against a strong bowler", (2, 4), "Chase progress", 80, 20, 0, 2),
        ("early_loss_quality", "Early wicket and a high-quality opposition bowler", (9, 4), "Innings-adjusted scoring", 55, 45, 0, 1),
        ("slow_powerplay_rate", "Slow powerplay followed by a rising chase requirement", (7, 14), "Chase progress", 80, 20, 0, 1),
        ("fresh_partner_rebuild", "Set striker rebuilding a stand with a fresh partner", (11, 13), "Innings-adjusted scoring", 55, 45, 0, 1),
        ("steep_close_chase", "Steep chase late in a competitive finish", (2, 16), "Chase progress", 85, 15, 0, 1),
        ("chase_tail_quality", "Tail support and strong opposition in a demanding chase", (29, 4), "Chase progress", 45, 55, 0, 1),
        ("pace_style_transition", "Pace/spin transition into high-quality pace", (10, 37), "Context-adjusted bat runs", 70, 30, 0, 1),
        ("knockout_death_quality", "Live knockout finish against strong bowling", (48, 4), "Chase progress", 85, 15, 0, 1),
        ("collapse_demand_quality", "Match demand, limited support and strong opposition", (1, 3, 4), "Innings-adjusted scoring", 55, 45, 0, 2),
    ],
    "bowler": [
        ("demand_against_quality", "Match demand against strong batting", (1, 4), "Innings-adjusted run control", 65, 35, 0, 2),
        ("close_chase_quality", "Close defence against a strong batter", (2, 4), "Chase squeeze", 70, 20, 10, 2),
        ("support_under_attack", "Limited options with the batting side attacking", (3, 5), "Innings-adjusted run control", 65, 35, 0, 2),
        ("strong_set_partnership", "Settled partnership with a strong striker", (4, 8), "Run suppression", 45, 55, 0, 2),
        ("penultimate_close", "Pressure 19th over in a close chase", (16, 2), "Chase squeeze", 70, 20, 10, 2),
        ("quota_late_defence", "Final quota over while defending a live chase", (14, 33), "Chase squeeze", 70, 20, 10, 1),
        ("costly_takeover_attack", "Taking over after a costly over while under attack", (15, 5), "Innings-adjusted run control", 75, 25, 0, 1),
        ("powerplay_boundary_surge", "Explosive powerplay following a boundary surge", (29, 38), "Run suppression", 60, 40, 0, 1),
        ("extra_close_finish", "Responding to an illegal ball in a close chase", (11, 2), "Chase squeeze", 55, 15, 30, 1),
        ("strong_left_hander_finish", "Dangerous left-hander in a genuinely close finish", (40, 2), "Chase squeeze", 70, 25, 5, 1),
        ("knockout_death_quality", "Live knockout finish against strong batting", (48, 4), "Chase squeeze", 70, 20, 10, 1),
        ("limited_options_quality_attack", "Limited options, strong batting and sustained attack", (3, 4, 5), "Innings-adjusted run control", 60, 40, 0, 2),
    ],
}

In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def signature(payload):
    message = json.dumps(payload, sort_keys=True).encode()
    return hashlib.sha256(message).hexdigest()


def write_json(path, data):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(data, indent=2, allow_nan=False))
    temporary.replace(path)


def write_parquet(path, data):
    temporary = path.with_suffix(".tmp")
    data.to_parquet(temporary, index=False, compression="snappy")
    temporary.replace(path)


def write_csv(path, data):
    temporary = path.with_suffix(".tmp")
    data.to_csv(temporary, index=False)
    temporary.replace(path)


def discover_raw(root):
    assert root.is_dir(), f"Input directory not found: {root}"
    roles = {name: [] for name in SOURCES}
    for path in sorted(root.rglob("*")):
        if (not path.is_file() or path.stat().st_size == 0
                or not path.name.lower().endswith((".csv", ".csv.gz"))):
            continue
        cols = set(pd.read_csv(path, nrows=0).columns)
        matching = [key for key, fields in SOURCES.items() if fields <= cols]
        if matching:
            key = max(matching, key=lambda k: len(SOURCES[k]))
            roles[key].append(path)
    missing = [key for key, paths in roles.items() if not paths]
    duplicates = {key: paths for key, paths in roles.items() if len(paths) > 1}
    assert not missing, f"Attach the original CSVs. Missing: {missing}"
    assert not duplicates, f"Attach one version of each raw file: {duplicates}"
    return {key: paths[0] for key, paths in roles.items()}


def raw_manifest(paths):
    return {
        name: {"file": str(path), "sha256": file_hash(path)}
        for name, path in paths.items()
    }


def eligible_cache(component, cache_key):
    filename = f"{component}_receipt.json"
    candidates = [OUTPUT_DIR / filename]
    cache_root = INPUT_ROOT if CACHE_INPUT is None else Path(CACHE_INPUT)
    candidates.extend(sorted(cache_root.rglob(filename)))
    for receipt in dict.fromkeys(candidates):
        if not receipt.is_file():
            continue
        meta = json.loads(receipt.read_text())
        if meta["signature"] != cache_key:
            continue
        directory = receipt.parent
        ready = all(
            (directory / name).is_file()
            and file_hash(directory / name) == digest
            for name, digest in meta["files"].items()
        )
        if ready:
            return directory
    return None


def save_receipt(stage, key, files, metadata):
    values = dict(
        signature=key,
        files={name: file_hash(OUTPUT_DIR / name) for name in files},
        **metadata,
    )
    write_json(OUTPUT_DIR / f"{stage}_receipt.json", values)

In [ ]:
def build_audited_state(raw):
    balls = raw["ball_by_ball"].copy()
    matches = raw["matches"].copy()
    competitions = raw["competitions"].copy()
    roster = raw["playing_eleven"].copy()
    mapping = raw["competition_team_player_mapping"].copy()
    names = mapping[["player_id", "player_name"]].drop_duplicates()
    assert names.player_id.is_unique, "Resolve conflicting names per ID before proceeding."
    assert matches.match_id.is_unique and competitions.comp_id.is_unique
    for col in ["innings", "overid", "over_ball_num", "innings_ball_num", "runs", "extras1_runs", "extras_2_runs"]:
        balls[col] = pd.to_numeric(balls[col], errors="raise").astype(int)
    assert not balls.duplicated(["match_id", "innings", "innings_ball_num"]).any()
    match_info = matches.merge(competitions, on="comp_id", how="left", validate="many_to_one")
    match_info["date"] = pd.to_datetime(match_info.match_date, format="%Y-%m-%d", errors="raise")
    match_info["competition_group"] = np.select(
        [match_info.comp_name.str.startswith("IPL "), match_info.match_format.eq("T20I")],
        ["IPL", "T20I"], default="Other T20")
    d = balls.merge(match_info, on="match_id", how="left", validate="many_to_one")
    assert len(d) == len(balls) and d.date.notna().all() and d.comp_name.notna().all()
    d = d.sort_values(["date", "match_id", "innings", "innings_ball_num"]).reset_index(drop=True)
    d["row_id"] = d.match_id + ":" + d.innings.astype(str) + ":" + d.innings_ball_num.astype(str)
    assert d.row_id.is_unique

    d["dismissal_clean"] = d.dismissal_type.replace({"Obstracting the field":"Obstructing the field"})
    allowed_dismissals = {"Bowled", "Caught", "LBW", "RunOut", "Stump", "Hit Wicket", "Retire Out", "Obstructing the field"}
    credited_dismissals = {"Bowled", "Caught", "LBW", "Stump", "Hit Wicket"}
    d["y_team_runs"] = d.runs + d.extras1_runs + d.extras_2_runs
    d["y_batter_runs"] = d.runs
    d["y_team_wicket"] = d.is_wicket.eq("true").astype(int)
    d["y_striker_dismissed"] = (d.y_team_wicket.eq(1) & d.dismissed_player_id.eq(d.striker_id)).astype(int)
    d["y_bowler_wicket"] = (d.y_team_wicket.eq(1) & d.dismissal_clean.isin(credited_dismissals)).astype(int)
    d["y_boundary"] = (d.is_four.eq("true") | d.is_six.eq("true")).astype(int)
    d["y_legal_delivery_candidate"] = (~d.extras_type.isin(["wide", "Overstep", "other", "Penalty"])).astype(int)
    d["y_nonwide_faced"] = (~d.extras_type.isin(["wide", "Penalty"])).astype(int)
    d["y_bowler_runs"] = d.runs + np.where(d.extras_type.isin(["wide", "Overstep", "other"]), d.extras1_runs, 0)

    reason_sets = defaultdict(set)
    def flag_matches(mask, reason):
        for match_id in d.loc[mask, "match_id"].unique():
            reason_sets[match_id].add(reason)

    flag_matches(d.extras_type.eq("other") & (not ACCEPT_INFERRED_OTHER_AS_NO_BALL), "unresolved_other_extras")
    flag_matches(~d.extras_type.isin(["None","wide","Overstep","other","byes","legbyes"]), "penalty_or_unknown_extras")
    flag_matches(d.extras_type.eq("None") & (d.extras1_runs.ne(0)|d.extras_2_runs.ne(0)), "extras_label_contradiction")
    flag_matches(d.extras_type.isin(["wide","byes","legbyes"]) & d.runs.ne(0), "batter_runs_on_nonbat_extras")
    flag_matches(d.extras_2_runs.ne(0) & (~d.extras_type.isin(["Overstep","other"]) | d.runs.ne(0)), "unresolved_secondary_extras")
    flag_matches(d.extras_type.isin(["Overstep","other"]) & d.extras1_runs.ne(1), "unexpected_noball_penalty")
    flag_matches(d.y_team_wicket.eq(1) & (d.dismissed_player_id.eq("") | ~d.dismissal_clean.isin(allowed_dismissals)), "incomplete_wicket")
    flag_matches(d.y_team_wicket.eq(0) & d.dismissed_player_id.ne(""), "dismissed_id_without_wicket")
    flag_matches(d.y_team_wicket.eq(1) & ~d.dismissed_player_id.isin(names.player_id), "unknown_dismissed_player")
    flag_matches(d[["runs","extras1_runs","extras_2_runs"]].lt(0).any(axis=1), "negative_score_component")
    flag_matches((d.is_four.eq("true") & d.runs.ne(4)) | (d.is_six.eq("true") & d.runs.ne(6)), "boundary_flag_contradiction")
    revised = matches.first_inn_revised_overs.ne("") | matches.second_inn_revised_overs.ne("") | matches.revised_target.ne("")
    for mid in matches.loc[revised,"match_id"]:
        reason_sets[mid].add("revised_allocation_or_target_timing_unverified")
    for mid in matches.loc[matches.win_team_id.eq(""),"match_id"]:
        reason_sets[mid].add("no_decisive_result")
    flag_matches(d.innings.gt(2), "extra_innings_separate_analysis")

    innings_audit = d.groupby(["match_id","innings"], as_index=False).agg(
        delivery_records=("row_id","size"), sequence_min=("innings_ball_num","min"), sequence_max=("innings_ball_num","max"),
        team_runs=("y_team_runs","sum"), team_wickets=("y_team_wicket","sum"), legal_balls=("y_legal_delivery_candidate","sum"),
        batting_team_id=("batting_team_id","first"), batting_teams=("batting_team_id","nunique"), bowling_teams=("bowling_team_id","nunique"))
    for row in innings_audit.itertuples():
        if row.sequence_min != 1 or row.sequence_max != row.delivery_records or row.batting_teams != 1 or row.bowling_teams != 1:
            reason_sets[row.match_id].add("sequence_or_team_inconsistency")
        if row.innings in [1,2] and (row.legal_balls>120 or row.team_wickets>10):
            reason_sets[row.match_id].add("ordinary_innings_limit_inconsistent")
    over_audit = d.groupby(["match_id","innings","overid"],as_index=False).agg(legal_balls=("y_legal_delivery_candidate","sum"))
    over_audit["last_recorded_over"] = over_audit.groupby(["match_id","innings"]).overid.transform("max")
    bad_overs = over_audit.loc[(over_audit.overid.lt(over_audit.last_recorded_over)&over_audit.legal_balls.ne(6)) | over_audit.legal_balls.gt(6)]
    for mid in bad_overs.match_id.unique():
        reason_sets[mid].add("over_count_requires_review")
    first = innings_audit.loc[innings_audit.innings.eq(1)].set_index("match_id")
    second = innings_audit.loc[innings_audit.innings.eq(2)].set_index("match_id")
    for row in match_info.itertuples():
        mid = row.match_id
        if mid not in first.index or mid not in second.index:
            reason_sets[mid].add("missing_ordinary_innings")
            continue
        a,b = first.loc[mid],second.loc[mid]
        if a.team_wickets<10 and a.legal_balls != 120:
            reason_sets[mid].add("incomplete_first_innings_or_unrecorded_reduction")
        if row.win_team_id == b.batting_team_id:
            if b.team_runs < a.team_runs+1:
                reason_sets[mid].add("chase_result_score_mismatch")
        elif row.win_team_id == a.batting_team_id:
            margin = re.search(r"won by\s+(\d+)\s+runs?\b",row.result,re.I)
            if margin is None or a.team_runs-b.team_runs != int(margin.group(1)):
                reason_sets[mid].add("run_margin_score_mismatch")
            if b.team_wickets<10 and b.legal_balls != 120:
                reason_sets[mid].add("incomplete_defence_or_unrecorded_reduction")
        elif row.win_team_id:
            reason_sets[mid].add("winner_not_in_recorded_innings")

    match_audit = match_info[["match_id","match_date","comp_name","competition_group","result"]].copy()
    match_audit["delivery_rows"] = match_audit.match_id.map(d.groupby("match_id").size()).fillna(0).astype(int)
    match_audit["accounting_exclusion_reasons"] = match_audit.match_id.map(lambda k:" | ".join(sorted(reason_sets[k])))
    match_audit["accounting_eligible"] = match_audit.accounting_exclusion_reasons.eq("")
    match_audit["in_primary_scope"] = match_audit.competition_group.isin(["IPL","T20I"]) | INCLUDE_OTHER_T20_COMPETITIONS
    match_audit["model_eligible"] = match_audit.accounting_eligible & match_audit.in_primary_scope
    safe_ids = set(match_audit.loc[match_audit.accounting_eligible,"match_id"])
    model_ids = set(match_audit.loc[match_audit.model_eligible,"match_id"])
    d["state_accounting_eligible"] = d.match_id.isin(safe_ids)
    d["primary_model_eligible"] = d.match_id.isin(model_ids)

    ig = ["match_id","innings"]
    def before_cumsum(frame, keys, column):
        return frame.groupby(keys,sort=False)[column].cumsum()-frame[column]

    d["x_team_runs_before"] = before_cumsum(d,ig,"y_team_runs")
    d["x_wickets_lost_before"] = before_cumsum(d,ig,"y_team_wicket")
    d["x_legal_balls_used"] = before_cumsum(d,ig,"y_legal_delivery_candidate")
    d["x_balls_remaining"] = (120-d.x_legal_balls_used).clip(lower=0)
    d["x_wickets_remaining"] = 10-d.x_wickets_lost_before
    d["x_current_run_rate"] = (6*d.x_team_runs_before/d.x_legal_balls_used.replace(0,np.nan))
    d["x_innings"] = d.innings
    d["x_phase"] = np.select([d.x_legal_balls_used.lt(36),d.x_legal_balls_used.lt(90)], ["powerplay","middle"],default="death")
    d["x_competition_group"] = d.competition_group
    d["x_venue_id"] = d.venue_id
    d["x_venue_country"] = d.venue_country
    d["x_knockout"] = pd.to_numeric(d.knock_out)
    d["x_target"] = np.where(d.innings.eq(2),d.match_id.map(first.team_runs)+1,np.nan)
    d.loc[~d.state_accounting_eligible,"x_target"] = np.nan
    d["x_runs_required"] = (d.x_target-d.x_team_runs_before).clip(lower=0)
    d["x_required_run_rate"] = 6*d.x_runs_required/d.x_balls_remaining.replace(0,np.nan)
    d["x_required_minus_current_rate"] = d.x_required_run_rate-d.x_current_run_rate
    d["x_striker_runs_before"] = before_cumsum(d,ig+["striker_id"],"y_batter_runs")
    d["x_striker_nonwide_deliveries_before"] = before_cumsum(d,ig+["striker_id"],"y_nonwide_faced")
    d["x_bowler_runs_before"] = before_cumsum(d,ig+["bowler_id"],"y_bowler_runs")
    d["x_bowler_wickets_before"] = before_cumsum(d,ig+["bowler_id"],"y_bowler_wicket")
    d["x_bowler_legal_balls_before"] = before_cumsum(d,ig+["bowler_id"],"y_legal_delivery_candidate")
    d["x_bowler_quota_balls_remaining_proxy"] = (24-d.x_bowler_legal_balls_before).clip(lower=0)
    d["x_batting_hand"] = d.batting_type
    d["x_bowling_arm"] = np.where(d.bowling_type.str.contains("left",case=False),"left","right")
    d["x_pace_or_spin"] = d.pace_or_spin
    d["x_bowling_style"] = d.bowling_type.str.lower().str.strip()
    d["x_style_matchup"] = d.x_batting_hand+" vs "+d.x_bowling_arm+" "+d.x_pace_or_spin

    non_runs=np.zeros(len(d),dtype=int)
    non_faced=np.zeros(len(d),dtype=int)
    partner_runs=np.zeros(len(d),dtype=int)
    partner_legal=np.zeros(len(d),dtype=int)
    since_wicket=np.zeros(len(d),dtype=int)
    consecutive_dots=np.zeros(len(d),dtype=int)
    last6runs=np.zeros(len(d),dtype=int)
    last6wickets=np.zeros(len(d),dtype=int)
    last6boundaries=np.zeros(len(d),dtype=int)
    remaining_pools={}

    canonical_rows=[]
    roster_resolution_rows=[]
    roster_keys=["match_id","team_id","player_id"]
    for key,group in roster.groupby(roster_keys,sort=False):
        item=dict(zip(roster_keys,key)); conflicts=[]
        for column in [c for c in roster if c not in roster_keys]:
            values=sorted(set(group[column])-{ "" })
            item[column]=values[0] if len(values)==1 else ""
            if len(values)>1:
                conflicts.append(column)
        canonical_rows.append(item)
        if len(group)>1:
            roster_resolution_rows.append({**dict(zip(roster_keys,key)),"raw_rows":len(group),
                 "conflicting_fields":" | ".join(conflicts),
                 "action":"consistent fields retained; conflicting fields blanked"})
    canonical_roster=pd.DataFrame(canonical_rows)
    roster_resolution=pd.DataFrame(roster_resolution_rows)
    assert not canonical_roster.duplicated(roster_keys).any()
    structural_fields={"is_played","is_captain","is_wk","is_impact","is_impact_off"}
    unresolved_structural_teams=set()
    for row in roster_resolution.itertuples():
        if structural_fields.intersection(row.conflicting_fields.split(" | ")):
            unresolved_structural_teams.add((row.match_id,row.team_id))

    played=canonical_roster.loc[canonical_roster.is_played.eq("true")].copy()
    roster_conflicts=roster.loc[roster.duplicated(["match_id","team_id","player_id"],keep=False)].copy()
    starting=played.loc[played.is_impact.ne("1")].drop_duplicates(["match_id","team_id","player_id"])
    initial_pools=starting.groupby(["match_id","team_id"]).player_id.agg(set).to_dict()
    conflict_teams=set(unresolved_structural_teams)
    impact_teams=set(map(tuple,played.loc[played.is_impact.eq("1")|played.is_impact_off.eq("1"),["match_id","team_id"]].drop_duplicates().to_numpy()))
    resource_pool_valid=np.zeros(len(d),dtype=bool)
    resource_is_proxy=np.zeros(len(d),dtype=bool)

    for key,g in d.groupby(ig,sort=False):
        scores=defaultdict(int);faced=defaultdict(int);dismissed=set()
        pr=pl=since=dot=0
        run_prefix=np.r_[0,g.y_team_runs.cumsum().to_numpy()]
        wicket_prefix=np.r_[0,g.y_team_wicket.cumsum().to_numpy()]
        boundary_prefix=np.r_[0,g.y_boundary.cumsum().to_numpy()]
        legal_before=g.x_legal_balls_used.to_numpy()
        for j,row in enumerate(g.itertuples()):
            i=row.Index
            non_runs[i]=scores[row.non_striker_id];non_faced[i]=faced[row.non_striker_id]
            partner_runs[i]=pr;partner_legal[i]=pl;since_wicket[i]=since;consecutive_dots[i]=dot
            start=np.searchsorted(legal_before,max(0,legal_before[j]-6),side="left")
            last6runs[i]=run_prefix[j]-run_prefix[start]
            last6wickets[i]=wicket_prefix[j]-wicket_prefix[start]
            last6boundaries[i]=boundary_prefix[j]-boundary_prefix[start]
            teamkey=(row.match_id,row.batting_team_id)
            pool=initial_pools.get(teamkey,set())
            resource_pool_valid[i]=(len(pool)==11 and teamkey not in conflict_teams and row.striker_id in pool and row.non_striker_id in pool)
            resource_is_proxy[i]=teamkey in impact_teams
            remaining_pools[i]=pool-dismissed-{row.striker_id,row.non_striker_id}
            scores[row.striker_id]+=row.y_batter_runs
            faced[row.striker_id]+=row.y_nonwide_faced
            if row.y_team_wicket:
                dismissed.add(row.dismissed_player_id);pr=pl=since=0
            else:
                pr+=row.y_team_runs;pl+=row.y_legal_delivery_candidate;since+=row.y_legal_delivery_candidate
            dot=dot+1 if row.y_team_runs==0 else 0

    d["x_non_striker_runs_before"]=non_runs
    d["x_non_striker_nonwide_deliveries_before"]=non_faced
    d["x_partnership_runs_before"]=partner_runs
    d["x_partnership_legal_balls_before"]=partner_legal
    d["x_legal_balls_since_wicket"]=since_wicket
    d["x_consecutive_team_dot_events"]=consecutive_dots
    d["x_runs_previous_6_legal_ball_window"]=last6runs
    d["x_wickets_previous_6_legal_ball_window"]=last6wickets
    d["x_boundaries_previous_6_legal_ball_window"]=last6boundaries
    d["x_resource_pool_valid"]=resource_pool_valid
    d["x_remaining_pool_substitution_uncertain"]=resource_is_proxy
    for label,name in [("y_bowler_runs","runs"),("y_boundary","boundaries")]:
        d[f"x_bowler_previous_6_events_{name}"]=d.groupby(ig+["bowler_id"],sort=False)[label].transform(lambda z:z.shift().rolling(6,min_periods=1).sum()).fillna(0)
    flag_matches(d.innings.isin([1,2]) & (d.x_legal_balls_used.ge(120)|d.x_wickets_lost_before.ge(10)), "delivery_after_innings_limit")
    flag_matches(d.innings.eq(2) & d.x_target.notna() & d.x_team_runs_before.ge(d.x_target), "delivery_after_chase_won")
    flag_matches(d.x_bowler_legal_balls_before.ge(24), "bowler_quota_requires_review")
    match_audit["accounting_exclusion_reasons"] = match_audit.match_id.map(lambda k:" | ".join(sorted(reason_sets[k])))
    match_audit["accounting_eligible"] = match_audit.accounting_exclusion_reasons.eq("")
    match_audit["model_eligible"] = match_audit.accounting_eligible & match_audit.in_primary_scope
    safe_ids=set(match_audit.loc[match_audit.accounting_eligible,"match_id"])
    model_ids=set(match_audit.loc[match_audit.model_eligible,"match_id"])
    d["state_accounting_eligible"]=d.match_id.isin(safe_ids)
    d["primary_model_eligible"]=d.match_id.isin(model_ids)
    d.loc[~d.state_accounting_eligible,["x_target","x_runs_required","x_required_run_rate","x_required_minus_current_rate"]]=np.nan

    history_source=d.loc[d.primary_model_eligible].copy()
    CTX=["competition_group","x_phase","x_batting_hand","x_pace_or_spin"]
    context_stats=defaultdict(lambda:np.zeros(4,dtype=float))  # events, batter runs, bowler runs, striker dismissals
    global_stats=np.zeros(4,dtype=float)

    def new_history():
        return {"events":0,"opportunity":0,"runs":0,"wickets":0,"boundaries":0,"matches":0,
                "recent":deque(maxlen=RECENT_MATCHES),"repertoire":defaultdict(int),
                "repertoire_performance":defaultdict(lambda:np.zeros(3,dtype=float)),
                "context_events":0,"run_residual_sum":0.,"wicket_residual_sum":0.,"last_date":None}

    bat_history=defaultdict(new_history)
    bowl_history=defaultdict(new_history)
    bat_style_history=defaultdict(lambda:np.zeros(4,dtype=float))
    bowl_hand_history=defaultdict(lambda:np.zeros(4,dtype=float))
    pair_history=defaultdict(lambda:np.zeros(4,dtype=float))
    bat_feature_rows=[];bowl_feature_rows=[];context_rows=[];matchup_rows=[]
    appearance_rows={"batter":[],"bowler":[]}
    repertoire_rows=[]

    def ratio(a,b):
        return a/b if b>0 else np.nan

    def history_features(h,role):
        recent=np.sum(np.asarray(list(h["recent"]),dtype=float),axis=0) if h["recent"] else np.zeros(4)
        counts=np.asarray(list(h["repertoire"].values()),dtype=float)
        probs=counts/counts.sum() if counts.size and counts.sum() else np.array([])
        ans={"prior_events":h["events"],"prior_matches":h["matches"],"prior_opportunities":h["opportunity"],
             "prior_runs_per_event":ratio(h["runs"],h["events"]),
             "prior_wicket_rate":ratio(h["wickets"],h["events"]),
             "prior_boundary_rate":ratio(h["boundaries"],h["events"]),
             "last5_matches_available":len(h["recent"]),"last5_events":recent[0],
             "last5_runs_per_event":ratio(recent[2],recent[0]),"last5_wicket_rate":ratio(recent[3],recent[0]),
             "prior_repertoire_categories":len(counts),
             "prior_repertoire_entropy":float(-(probs*np.log(probs)).sum()) if len(probs) else np.nan,
             "prior_repertoire_label_coverage":ratio(counts.sum(),h["events"]),
             "prior_repertoire_dominant_share":float(probs.max()) if len(probs) else np.nan,
             "prior_context_events":h["context_events"],
             "prior_context_run_residual":ratio(h["run_residual_sum"],h["context_events"]),
             "prior_context_wicket_residual":ratio(h["wicket_residual_sum"],h["context_events"]),
             "history_missing":int(h["events"]==0)}
        if role=="bowler":
            ans["prior_economy"]=6*ratio(h["runs"],h["opportunity"])
            ans["last5_economy"]=6*ratio(recent[2],recent[1])
        else:
            ans["prior_runs_per_nonwide_event"]=ratio(h["runs"],h["opportunity"])
            ans["last5_runs_per_nonwide_event"]=ratio(recent[2],recent[1])
        ans["recent_minus_long_run_rate"]=ans["last5_runs_per_event"]-ans["prior_runs_per_event"]
        return ans

    def update_role(day,role):
        pidcol="striker_id" if role=="batter" else "bowler_id"
        runscol="y_batter_runs" if role=="batter" else "y_bowler_runs"
        oppcol="y_nonwide_faced" if role=="batter" else "y_legal_delivery_candidate"
        wicketcol="y_striker_dismissed" if role=="batter" else "y_bowler_wicket"
        stylecol="stroke" if role=="batter" else "delivery_style"
        store=bat_history if role=="batter" else bowl_history
        for (mid,pid),g in day.groupby(["match_id",pidcol],sort=False):
            h=store[pid]
            ev=len(g);op=int(g[oppcol].sum());ru=int(g[runscol].sum());wi=int(g[wicketcol].sum())
            h["events"]+=ev;h["opportunity"]+=op;h["runs"]+=ru;h["wickets"]+=wi
            h["boundaries"]+=int(g.y_boundary.sum());h["matches"]+=1
            h["recent"].append((ev,op,ru,wi));h["last_date"]=g.date.iloc[0]
            for label,n in g.loc[~g[stylecol].isin(["","None"]),stylecol].value_counts().items():
                h["repertoire"][label]+=int(n)
            for label,category in g.loc[~g[stylecol].isin(["","None"])].groupby(stylecol,sort=False):
                h["repertoire_performance"][label]+=np.array([len(category),category[runscol].sum(),category[wicketcol].sum()])
            expected_col="audit_expected_batter_runs" if role=="batter" else "audit_expected_bowler_runs"
            usable=g[expected_col].notna()
            h["context_events"]+=int(usable.sum())
            h["run_residual_sum"]+=float((g.loc[usable,runscol]-g.loc[usable,expected_col]).sum())
            expected_wicket="audit_expected_striker_wicket" if role=="batter" else "audit_expected_bowler_wicket"
            h["wicket_residual_sum"]+=float((g.loc[usable,wicketcol]-g.loc[usable,expected_wicket]).sum())
            appearance_rows[role].append({"date":g.date.iloc[0],"match_id":mid,"player_id":pid,
                                         "events":ev,"opportunities":op,"runs":ru,"wickets":wi})

    context_bowler_wickets=defaultdict(float)
    for date,current in d.groupby("date",sort=True):
        today_source=history_source.loc[history_source.date.eq(date)].copy()
        required_bat=set(current.striker_id)|set(current.non_striker_id)
        for i in current.index:
            required_bat.update(remaining_pools[i])
        for pid in sorted(required_bat):
            h=bat_history[pid]
            bat_feature_rows.append({"date":date,"player_id":pid,"history_latest_date":h["last_date"],**history_features(h,"batter")})
            for label,stats in h["repertoire_performance"].items():
                repertoire_rows.append({"date":date,"player_id":pid,"role":"batter","category":label,
                    "prior_events":stats[0],"prior_runs_per_event":ratio(stats[1],stats[0]),
                    "prior_striker_dismissal_or_bowler_wicket_rate":ratio(stats[2],stats[0])})
        for pid in sorted(set(current.bowler_id)):
            h=bowl_history[pid]
            bowl_feature_rows.append({"date":date,"player_id":pid,"history_latest_date":h["last_date"],**history_features(h,"bowler")})
            for label,stats in h["repertoire_performance"].items():
                repertoire_rows.append({"date":date,"player_id":pid,"role":"bowler","category":label,
                    "prior_events":stats[0],"prior_runs_per_event":ratio(stats[1],stats[0]),
                    "prior_striker_dismissal_or_bowler_wicket_rate":ratio(stats[2],stats[0])})
        for key in current[CTX].drop_duplicates().itertuples(index=False,name=None):
            values=context_stats[key]
            context_rows.append({"date":date,**dict(zip(CTX,key)),"x_context_prior_events":values[0],
                 "x_context_expected_batter_runs":ratio(values[1],values[0]),
                 "x_context_expected_bowler_runs":ratio(values[2],values[0]),
                 "x_context_expected_striker_wicket":ratio(values[3],values[0]),
                 "x_context_expected_bowler_wicket":ratio(context_bowler_wickets[key],values[0])})
        for key in current[["striker_id","bowler_id","x_bowling_arm","x_pace_or_spin","x_batting_hand"]].drop_duplicates().itertuples(index=False,name=None):
            sid,bid,arm,pace,hand=key
            bs=bat_style_history[(sid,arm,pace)]
            bh=bowl_hand_history[(bid,hand)]
            pair=pair_history[(sid,bid)]
            matchup_rows.append({"date":date,"striker_id":sid,"bowler_id":bid,"x_bowling_arm":arm,"x_pace_or_spin":pace,"x_batting_hand":hand,
                "x_batter_style_prior_events":bs[0],"x_batter_style_prior_runs_per_event":ratio(bs[1],bs[0]),
                "x_batter_style_prior_dismissal_rate":ratio(bs[2],bs[0]),
                "x_bowler_hand_prior_events":bh[0],"x_bowler_hand_prior_runs_per_event":ratio(bh[1],bh[0]),
                "x_bowler_hand_prior_wicket_rate":ratio(bh[2],bh[0]),
                "x_pair_prior_events":pair[0],"x_pair_prior_batter_runs_per_event":ratio(pair[1],pair[0]),
                "x_pair_prior_striker_dismissal_rate":ratio(pair[2],pair[0])})
        if today_source.empty:
            continue
        contexts=[tuple(x) for x in today_source[CTX].to_numpy()]
        today_source["audit_expected_batter_runs"]=[ratio(context_stats[k][1],context_stats[k][0]) for k in contexts]
        today_source["audit_expected_bowler_runs"]=[ratio(context_stats[k][2],context_stats[k][0]) for k in contexts]
        today_source["audit_expected_striker_wicket"]=[ratio(context_stats[k][3],context_stats[k][0]) for k in contexts]
        today_source["audit_expected_bowler_wicket"]=[ratio(context_bowler_wickets[k],context_stats[k][0]) for k in contexts]
        update_role(today_source,"batter");update_role(today_source,"bowler")
        for key,g in today_source.groupby(CTX,sort=False):
            context_stats[key]+=np.array([len(g),g.y_batter_runs.sum(),g.y_bowler_runs.sum(),g.y_striker_dismissed.sum()])
            context_bowler_wickets[key]+=g.y_bowler_wicket.sum()
        for (pid,arm,pace),g in today_source.groupby(["striker_id","x_bowling_arm","x_pace_or_spin"],sort=False):
            bat_style_history[(pid,arm,pace)]+=np.array([len(g),g.y_batter_runs.sum(),g.y_striker_dismissed.sum(),0])
        for (pid,hand),g in today_source.groupby(["bowler_id","x_batting_hand"],sort=False):
            bowl_hand_history[(pid,hand)]+=np.array([len(g),g.y_bowler_runs.sum(),g.y_bowler_wicket.sum(),0])
        for (sid,bid),g in today_source.groupby(["striker_id","bowler_id"],sort=False):
            pair_history[(sid,bid)]+=np.array([len(g),g.y_batter_runs.sum(),g.y_striker_dismissed.sum(),0])

    bat_history_table=pd.DataFrame(bat_feature_rows)
    bowl_history_table=pd.DataFrame(bowl_feature_rows)
    context_history_table=pd.DataFrame(context_rows)
    matchup_history_table=pd.DataFrame(matchup_rows)
    for table in [bat_history_table,bowl_history_table]:
        known=table.history_latest_date.notna()
        assert (table.loc[known,"history_latest_date"]<table.loc[known,"date"]).all(), "History date leakage detected."

    def merge_history(frame,table,role,id_column):
        renamed=table.rename(columns={"player_id":id_column,**{c:f"x_{role}_{c}" for c in table if c not in ["date","player_id","history_latest_date"]},
                                      "history_latest_date":f"audit_{role}_history_latest_date"})
        result=frame.merge(renamed,on=["date",id_column],how="left",validate="many_to_one")
        assert len(result)==len(frame)
        return result
    state_row_order=d.row_id.tolist()
    d=merge_history(d,bat_history_table,"batter","striker_id")
    d=merge_history(d,bowl_history_table,"bowler","bowler_id")
    d=merge_history(d,bat_history_table,"partner","non_striker_id")
    d=d.merge(context_history_table,on=["date"]+CTX,how="left",validate="many_to_one")
    d=d.merge(matchup_history_table,on=["date","striker_id","bowler_id","x_bowling_arm","x_pace_or_spin","x_batting_hand"],how="left",validate="many_to_one")
    assert len(d)==len(balls) and d.row_id.is_unique

    support_lookup={(row.date,row.player_id):(row.prior_runs_per_event,row.prior_matches)
                    for row in bat_history_table.itertuples()}
    support_count=[];support_known=[];support_mean=[];support_top3=[]
    assert d.row_id.tolist()==state_row_order
    for row in d.itertuples():
        pool=remaining_pools[row.Index]
        values=[support_lookup.get((row.date,pid),(np.nan,0))[0] for pid in pool]
        observed=[v for v in values if np.isfinite(v)]
        valid=row.x_resource_pool_valid
        support_count.append(len(pool) if valid else np.nan)
        support_known.append(len(observed) if valid else np.nan)
        support_mean.append(float(np.mean(observed)) if valid and observed else np.nan)
        support_top3.append(float(np.mean(sorted(observed,reverse=True)[:3])) if valid and observed else np.nan)
    d["x_remaining_batters_pool_count"]=support_count
    d["x_remaining_batters_with_history"]=support_known
    d["x_remaining_batting_mean_runs_per_event_proxy"]=support_mean
    d["x_remaining_batting_top3_runs_per_event_proxy"]=support_top3
    d["x_remaining_support_history_fraction"]=d.x_remaining_batters_with_history/d.x_remaining_batters_pool_count.replace(0,np.nan)

    first_innings=d.loc[d.primary_model_eligible&d.innings.eq(1)].groupby(["date","match_id","competition_group"],as_index=False).agg(total=("y_team_runs","sum"))
    par_rows=[]
    for group,dates in d.groupby("competition_group"):
        prior_sum=prior_count=0
        for date in sorted(dates.date.unique()):
            par_rows.append({"date":date,"competition_group":group,"x_prior_first_innings_count":prior_count,
                             "x_prior_first_innings_mean_total":ratio(prior_sum,prior_count)})
            today=first_innings.loc[first_innings.date.eq(date)&first_innings.competition_group.eq(group)]
            prior_sum+=today.total.sum();prior_count+=len(today)
    d=d.merge(pd.DataFrame(par_rows),on=["date","competition_group"],how="left",validate="many_to_one")
    d["x_linear_par_score_proxy"]=d.x_prior_first_innings_mean_total*d.x_legal_balls_used/120
    d["x_score_minus_linear_par_proxy"]=d.x_team_runs_before-d.x_linear_par_score_proxy

    def new_quality_state():
        return {"events":0,"matches":0,"context_events":0,"run_residual":0.,"wicket_residual":0.,
                "recent":deque(maxlen=RECENT_MATCHES),"variations":defaultdict(lambda:np.zeros(3)),
                "non_striker_dismissals":0}

    qhistory={"batter":defaultdict(new_quality_state),"bowler":defaultdict(new_quality_state)}
    quality_records={"batter":[],"bowler":[]}
    trajectory_stats=defaultdict(lambda:np.zeros(3))
    trajectory_broad=defaultdict(lambda:np.zeros(3))
    chase_stats=defaultdict(lambda:np.zeros(2))
    chase_broad=defaultdict(lambda:np.zeros(2))
    chase_global=np.zeros(2)
    chase_reference_last_date=None
    trajectory_records=[];chase_records=[]
    first_total_lookup=first.team_runs.to_dict()
    winner_lookup=matches.set_index("match_id").win_team_id.to_dict()

    def percentile_against(value,reference):
        if not np.isfinite(value) or len(reference)==0:
            return np.nan
        return 50*(np.searchsorted(reference,value,side="left")+np.searchsorted(reference,value,side="right"))/len(reference)

    def strength_snapshots(store,role,date,required):
        known={pid:h for pid,h in store.items() if h["context_events"]>0}
        refs={pid:h for pid,h in known.items() if h["events"]>=DERIVATION["rating_reference_events"]
              and h["matches"]>=DERIVATION["rating_reference_matches"]}
        fallback=len(refs)<5
        if fallback:
            refs=known
        sign=1 if role=="batter" else -1
        def run_skill(h): return sign*h["run_residual"]/h["context_events"]
        def wicket_skill(h): return -sign*h["wicket_residual"]/h["context_events"]
        run_ref=np.sort([run_skill(h) for h in refs.values()])
        wicket_ref=np.sort([wicket_skill(h) for h in refs.values()])
        weight=DERIVATION[role+"_run_weight"]
        for pid in sorted(required):
            h=store[pid]; n=h["context_events"]
            reliability=(n/(n+DERIVATION["rating_shrink_events"])
                         *h["matches"]/(h["matches"]+DERIVATION["rating_shrink_matches"]))
            if n:
                rs=run_skill(h);ws=wicket_skill(h)
                raw_score=weight*percentile_against(rs,run_ref)+(1-weight)*percentile_against(ws,wicket_ref)
                score=50+reliability*(raw_score-50)
                life_equiv=rs+10*ws
            else:
                rs=ws=score=life_equiv=np.nan
            recent=np.sum(np.asarray(list(h["recent"])),axis=0) if h["recent"] else np.zeros(4)
            recent_equiv=sign*ratio(recent[0],recent[2])-10*sign*ratio(recent[1],recent[2])
            form_known=h["matches"]>=3 and recent[3]>=DERIVATION["form_min_events"] and recent[2]>0
            gap=recent_equiv-life_equiv if form_known else np.nan
            supported=[v for v in h["variations"].values() if v[0]>=DERIVATION["variation_min_context_events"]]
            effective=sum(sign*v[1]/v[0]-10*sign*v[2]/v[0]>0 for v in supported)
            quality_records[role].append({"date":date,"player_id":pid,
                 "strength_score100_prior":score,"strength_reliability":reliability,
                 "strength_reference_players":len(refs),"strength_reference_fallback":fallback,
                 "strength_context_events":n,"strength_prior_matches":h["matches"],
                 "run_skill_residual_prior":rs,"wicket_skill_residual_prior":ws,
                 "form_context_gap_prior":gap,
                 "form_slump_candidate":float(gap < -DERIVATION["form_slump_run_equivalent_gap"]) if form_known else np.nan,
                 "supported_repertoire_categories":len(supported),"effective_repertoire_categories_proxy":effective,
                 "prior_non_striker_dismissals":h["non_striker_dismissals"]})

    def chase_key(row,with_wickets=True):
        if row.innings!=2 or not np.isfinite(row.x_runs_required) or row.x_balls_remaining<=0:
            return None
        key=(row.competition_group,int(np.digitize(row.x_balls_remaining,[12,30,60,90])),
             int(np.digitize(row.x_runs_required/row.x_balls_remaining,[.75,1.,1.25,1.5,1.75,2.,2.5,3.])))
        return key+(int(np.digitize(row.x_wickets_remaining,[2,5,8])),) if with_wickets else key

    for date,day in d.groupby("date",sort=True):
        needed_bat=set(day.striker_id)|set(day.non_striker_id)
        needed_bowl=set(day.bowler_id)
        for mid in day.match_id.unique():
            for teamid in day.loc[day.match_id.eq(mid),"bowling_team_id"].unique():
                needed_bowl.update(initial_pools.get((mid,teamid),set()))
        for i in day.index:
            needed_bat.update(remaining_pools[i])
        strength_snapshots(qhistory["batter"],"batter",date,needed_bat)
        strength_snapshots(qhistory["bowler"],"bowler",date,needed_bowl)
        for row in day.itertuples():
            bucket=int(np.digitize(row.x_wickets_lost_before,[3,6]))
            key=(row.competition_group,int(row.x_legal_balls_used),bucket)
            state=trajectory_stats[key];kind="wicket-adjusted"
            if state[0]<DERIVATION["trajectory_min_matches"]:
                state=trajectory_broad[key[:2]];kind="phase-only fallback"
            enough=state[0]>=DERIVATION["trajectory_min_matches"]
            trajectory_records.append({"row_id":row.row_id,
                "x_trajectory_reference_matches":state[0],"x_trajectory_reference_kind":kind,
                "x_trajectory_expected_score_proxy":ratio(state[1],state[0]) if enough else np.nan,
                "x_trajectory_expected_remaining_runs_proxy":ratio(state[2],state[0]) if enough else np.nan})
            ck=chase_key(row)
            if ck is not None:
                stats=chase_stats[ck];backoff=False
                if stats[0]<5:
                    stats=chase_broad[ck[:3]];backoff=True
                chase_records.append({"row_id":row.row_id,
                    "x_historical_chase_success_probability":(stats[1]+1)/(stats[0]+2) if stats[0]>0 else np.nan,
                    "x_historical_chase_reference_matches":stats[0],"x_historical_chase_broad_fallback":backoff,
                    "x_prior_global_chase_winrate":(chase_global[1]+1)/(chase_global[0]+2) if chase_global[0]>0 else np.nan,
                    "audit_chase_reference_latest_date":chase_reference_last_date})
        eligible=day.loc[day.primary_model_eligible]
        for role,pidcol,runcol,wicketcol,expectrun,expectwicket,label in [
            ("batter","striker_id","y_batter_runs","y_striker_dismissed","x_context_expected_batter_runs","x_context_expected_striker_wicket","stroke"),
            ("bowler","bowler_id","y_bowler_runs","y_bowler_wicket","x_context_expected_bowler_runs","x_context_expected_bowler_wicket","delivery_style")]:
            for (mid,pid),g in eligible.groupby(["match_id",pidcol],sort=False):
                h=qhistory[role][pid];use=g[expectrun].notna()&g[expectwicket].notna()
                rr=(g.loc[use,runcol]-g.loc[use,expectrun]).sum()
                wr=(g.loc[use,wicketcol]-g.loc[use,expectwicket]).sum()
                h["events"]+=len(g);h["matches"]+=1;h["context_events"]+=int(use.sum())
                h["run_residual"]+=rr;h["wicket_residual"]+=wr
                h["recent"].append((rr,wr,int(use.sum()),len(g)))
                for category,sub in g.loc[use&~g[label].isin(["","None"])].groupby(label,sort=False):
                    h["variations"][category]+=np.array([len(sub),(sub[runcol]-sub[expectrun]).sum(),(sub[wicketcol]-sub[expectwicket]).sum()])
        non_striker_out=eligible.loc[eligible.y_team_wicket.eq(1)&eligible.dismissed_player_id.ne(eligible.striker_id)]
        for pid,count in non_striker_out.dismissed_player_id.value_counts().items():
            qhistory["batter"][pid]["non_striker_dismissals"]+=int(count)
        unique_states=eligible.loc[eligible.innings.eq(1)].drop_duplicates(["match_id","x_legal_balls_used"])
        for row in unique_states.itertuples():
            key=(row.competition_group,int(row.x_legal_balls_used),int(np.digitize(row.x_wickets_lost_before,[3,6])))
            values=np.array([1,row.x_team_runs_before,first_total_lookup[row.match_id]-row.x_team_runs_before])
            trajectory_stats[key]+=values;trajectory_broad[key[:2]]+=values
        seen=set();seen_broad=set();seen_matches=set()
        for row in eligible.loc[eligible.innings.eq(2)].itertuples():
            key=chase_key(row)
            if key is None:continue
            won=int(winner_lookup[row.match_id]==row.batting_team_id)
            if (row.match_id,key) not in seen:
                chase_stats[key]+=np.array([1,won]);seen.add((row.match_id,key))
            if (row.match_id,key[:3]) not in seen_broad:
                chase_broad[key[:3]]+=np.array([1,won]);seen_broad.add((row.match_id,key[:3]))
            if row.match_id not in seen_matches:
                chase_global+=np.array([1,won]);seen_matches.add(row.match_id)
        if seen_matches:chase_reference_last_date=date

    strength_tables={role:pd.DataFrame(rows) for role,rows in quality_records.items()}
    for role,idcol in [("batter","striker_id"),("bowler","bowler_id"),("partner","non_striker_id")]:
        source=strength_tables["batter" if role=="partner" else role]
        source=source.rename(columns={"player_id":idcol,**{c:f"x_{role}_{c}" for c in source if c not in ["date","player_id"]}})
        d=d.merge(source,on=["date",idcol],how="left",validate="many_to_one")
    d=d.merge(pd.DataFrame(trajectory_records),on="row_id",how="left",validate="one_to_one")
    d=d.merge(pd.DataFrame(chase_records),on="row_id",how="left",validate="one_to_one")
    d["x_score_minus_trajectory_proxy"]=d.x_team_runs_before-d.x_trajectory_expected_score_proxy
    d["y_chase_won"]=np.where(d.innings.eq(2),d.win_team_id.eq(d.batting_team_id).astype(float),np.nan)
    assert len(d)==len(balls) and d.row_id.tolist()==state_row_order

    bat_strength={(row.date,row.player_id):row.strength_score100_prior for row in strength_tables["batter"].itertuples()}
    bowl_strength={(row.date,row.player_id):(row.strength_score100_prior,row.strength_context_events) for row in strength_tables["bowler"].itertuples()}
    remaining_strength=[];remaining_known=[];bowling_count=[];bowling_balls=[];bowling_quality=[];bowling_uncertain=[]
    bowling_history_coverage=[]
    wickets12=np.zeros(len(d))
    for (mid,inn),g in d.groupby(["match_id","innings"],sort=False):
        used=defaultdict(int)
        prefix=np.r_[0,g.y_team_wicket.cumsum().to_numpy()]
        clock=g.x_legal_balls_used.to_numpy()
        for j,row in enumerate(g.itertuples()):
            i=row.Index
            begin=np.searchsorted(clock,max(0,clock[j]-12),side="left")
            wickets12[i]=prefix[j]-prefix[begin]
            pool=remaining_pools[i]
            scores=[bat_strength.get((row.date,pid),np.nan) for pid in pool]
            scores=[v for v in scores if np.isfinite(v)]
            remaining_known.append((i,len(scores)))
            enough=row.x_resource_pool_valid and len(scores)>=max(1,len(pool)/2)
            remaining_strength.append((i,float(np.mean(sorted(scores,reverse=True)[:3])) if enough else np.nan))
            teamkey=(mid,row.bowling_team_id);own=initial_pools.get(teamkey,set())
            valid=len(own)==11 and teamkey not in conflict_teams and row.bowler_id in own
            candidates=[]
            known_options=0
            for pid in own-{row.bowler_id}:
                score,evidence=bowl_strength.get((row.date,pid),(np.nan,0))
                quota=max(0,24-used[pid])
                known_options+=int(evidence>0)
                if np.isfinite(score) and evidence>=24 and quota>0:
                    candidates.append((score,quota))
            bowling_count.append((i,len(candidates) if valid and known_options else np.nan))
            bowling_history_coverage.append((i,known_options/max(1,len(own-{row.bowler_id})) if valid else np.nan))
            bowling_balls.append((i,sum(v[1] for v in candidates) if valid else np.nan))
            bowling_quality.append((i,float(np.mean(sorted([v[0] for v in candidates],reverse=True)[:3])) if valid and candidates else np.nan))
            bowling_uncertain.append((i,teamkey in impact_teams or not valid))
            used[row.bowler_id]+=row.y_legal_delivery_candidate

    for col,items in [("x_remaining_top3_strength_score100_proxy",remaining_strength),
                      ("x_remaining_strength_players_known",remaining_known),
                      ("x_other_bowling_options_count_proxy",bowling_count),
                      ("x_other_bowling_quota_balls_proxy",bowling_balls),
                      ("x_other_bowling_top3_strength_proxy",bowling_quality),
                      ("x_other_bowling_history_coverage",bowling_history_coverage),
                      ("x_bowling_resources_uncertain",bowling_uncertain)]:
        d[col]=pd.Series(dict(items)).reindex(d.index)
    d["x_wickets_previous_12_legal_ball_window"]=wickets12
    d["x_collapse_candidate"]=d.x_wickets_previous_12_legal_ball_window.ge(DERIVATION["collapse_wickets_in_12_legal_balls"]).astype(int)
    d["x_bowler_under_attack_candidate"]=(d.x_bowler_previous_6_events_runs.ge(DERIVATION["attack_runs_in_6_bowler_events"])
            |d.x_bowler_previous_6_events_boundaries.ge(DERIVATION["attack_boundaries_in_6_bowler_events"])).astype(int)
    d["x_batter_entry_crisis_candidate"]=(d.x_collapse_candidate.eq(1)&d.x_striker_nonwide_deliveries_before.le(6)).astype(int)

    def mean_available(*parts):
        return pd.concat(parts,axis=1).mean(axis=1,skipna=True)
    chase=d.innings.eq(2)
    winp=d.x_historical_chase_success_probability
    d["x_batter_match_demand_score100"]=np.where(chase,100*(1-winp),(-d.x_score_minus_trajectory_proxy*3).clip(0,100))
    d["x_bowler_match_demand_score100"]=np.where(chase,100*winp,(d.x_score_minus_trajectory_proxy*3).clip(0,100))
    d["x_finish_tension_score100_proxy"]=np.where(chase,100*4*winp*(1-winp)*(1-d.x_balls_remaining/120),np.nan)
    empty_support=d.x_resource_pool_valid&d.x_remaining_batters_pool_count.eq(0)
    support_pressure=100-d.x_remaining_top3_strength_score100_proxy
    support_pressure=support_pressure.mask(empty_support,100)
    d["x_batter_resource_pressure_score100"]=mean_available(10*d.x_wickets_lost_before,support_pressure)
    d["x_bowler_resource_pressure_score100"]=mean_available(10*d.x_wickets_remaining,d.x_remaining_top3_strength_score100_proxy,
                                                                 100-d.x_other_bowling_top3_strength_proxy)
    d["x_batter_recent_shock_score100"]=(50*d.x_wickets_previous_12_legal_ball_window+5*d.x_consecutive_team_dot_events).clip(0,100)
    d["x_bowler_recent_shock_score100"]=mean_available((d.x_bowler_previous_6_events_runs/18*100).clip(0,100),
                                                                   (d.x_bowler_previous_6_events_boundaries/3*100).clip(0,100))

    pressure_component_columns={}
    for role,opponent in [("batter","bowler"),("bowler","batter")]:
        components=pd.DataFrame({
            "match":d[f"x_{role}_match_demand_score100"],"opponent":d[f"x_{opponent}_strength_score100_prior"],
            "resources":d[f"x_{role}_resource_pressure_score100"],"recent":d[f"x_{role}_recent_shock_score100"],
            "stakes":100*d.x_knockout})
        weights=pd.Series({"match":.35,"opponent":.25,"resources":.20,"recent":.15,"stakes":.05})
        coverage=components.notna().mul(weights,axis=1).sum(axis=1)
        base=components.mul(weights,axis=1).sum(axis=1)/coverage.replace(0,np.nan)
        d[f"x_{role}_pressure_weight_coverage"]=coverage
        d[f"x_{role}_pressure_context_score100_candidate"]=base
        form=d[f"x_{role}_form_slump_candidate"]
        d[f"x_{role}_pressure_with_form_score100_candidate"]=np.where(form.notna(),.9*base+10*form,base)
        d[f"x_{role}_high_pressure_candidate"]=((base>=DERIVATION["pressure_candidate_threshold"])
                  &(coverage>=DERIVATION["pressure_min_weight_coverage"])).astype(int)
        pressure_component_columns[role]=components.columns.tolist()

    episode_rows=[]
    eligible_states=d.loc[d.primary_model_eligible]
    for (mid,inn),inning in eligible_states.groupby(["match_id","innings"],sort=False):
        for role,pidcol,oppcol,runcol,expectcol in [
            ("batter","striker_id","y_nonwide_faced","y_batter_runs","x_context_expected_batter_runs"),
            ("bowler","bowler_id","y_legal_delivery_candidate","y_bowler_runs","x_context_expected_bowler_runs")]:
            for pid,g in inning.groupby(pidcol,sort=False):
                triggers=[]
                if g[f"x_{role}_form_slump_candidate"].iloc[0]==1:
                    triggers.append((0,"prior_form_slump"))
                triggercol="x_batter_entry_crisis_candidate" if role=="batter" else "x_bowler_under_attack_candidate"
                locations=np.flatnonzero(g[triggercol].to_numpy()==1)
                if len(locations):triggers.append((int(locations[0]),"collapse_entry" if role=="batter" else "sustained_attack"))
                last_selected_end=-1
                for start,reason in sorted(triggers):
                    opening=g.iloc[start];future=g.iloc[start:]
                    opportunities=future[oppcol].cumsum()
                    reached=np.flatnonzero(opportunities.to_numpy()>=DERIVATION["response_opportunities"])
                    endseq=int(future.iloc[int(reached[0])].innings_ball_num) if len(reached) else int(inning.innings_ball_num.max())
                    if role=="batter":
                        dismissals=inning.loc[inning.y_team_wicket.eq(1)&inning.dismissed_player_id.eq(pid)
                                              &inning.innings_ball_num.ge(opening.innings_ball_num)]
                        if len(dismissals):endseq=min(endseq,int(dismissals.innings_ball_num.min()))
                    response=future.loc[future.innings_ball_num.le(endseq)]
                    if response.empty:continue
                    observed_dismissal=int(role=="batter" and ((inning.y_team_wicket.eq(1))&inning.dismissed_player_id.eq(pid)
                                  &inning.innings_ball_num.between(int(opening.innings_ball_num),endseq)).any())
                    complete=int(response[oppcol].sum()>=DERIVATION["response_opportunities"] or observed_dismissal)
                    baseline_known=response[expectcol].notna().all()
                    actual=float(response[runcol].sum());expected=float(response[expectcol].sum()) if baseline_known else np.nan
                    wickets=int(response.y_bowler_wicket.sum()) if role=="bowler" else observed_dismissal
                    success=(int(actual>=expected and not observed_dismissal) if role=="batter" else int(actual<=expected or wickets>0)) if baseline_known and complete else np.nan
                    selected=int(opening.innings_ball_num>last_selected_end)
                    if selected:last_selected_end=endseq
                    episode_rows.append({"episode_id":f"{mid}:{inn}:{role}:{pid}:{reason}","date":opening.date,"match_id":mid,"innings":inn,
                        "role":role,"player_id":pid,"trigger":reason,"start_sequence":int(opening.innings_ball_num),"end_sequence":endseq,
                        "selected_nonoverlap":selected,"y_response_events":len(response),"y_response_opportunities":int(response[oppcol].sum()),
                        "y_response_runs":actual,"y_response_expected_runs":expected,"y_response_wickets_or_dismissal":wickets,
                        "y_response_complete":complete,"y_recovery_success_candidate":success,
                        "y_run_response_above_expected":actual-expected if role=="batter" else expected-actual})
    recovery_episodes=pd.DataFrame(episode_rows)
    assert len(recovery_episodes),"No recovery episodes found; inspect criteria."
    recovery_history_rows=[]
    recovery_store=defaultdict(lambda:np.zeros(3))
    for date,day in d.groupby("date",sort=True):
        for role,pidcol in [("batter","striker_id"),("bowler","bowler_id")]:
            for pid in day[pidcol].unique():
                h=recovery_store[(role,pid)]
                recovery_history_rows.append({"date":date,"role":role,"player_id":pid,
                    "prior_recovery_episodes":h[0],"prior_completed_recovery_episodes":h[1],
                    "prior_recovery_success_rate":(h[2]+1)/(h[1]+2) if h[1] else np.nan})
        episodes=recovery_episodes.loc[recovery_episodes.date.eq(date)&recovery_episodes.selected_nonoverlap.eq(1)]
        for (role,pid),g in episodes.groupby(["role","player_id"],sort=False):
            valid=g.y_recovery_success_candidate.notna()
            recovery_store[(role,pid)]+=np.array([len(g),valid.sum(),g.loc[valid,"y_recovery_success_candidate"].sum()])
    recovery_history_by_date=pd.DataFrame(recovery_history_rows)
    for role,pidcol in [("batter","striker_id"),("bowler","bowler_id")]:
        table=recovery_history_by_date.loc[recovery_history_by_date.role.eq(role)].drop(columns="role")
        table=table.rename(columns={"player_id":pidcol,**{c:f"x_{role}_{c}" for c in table if c not in ["date","player_id"]}})
        d=d.merge(table,on=["date",pidcol],how="left",validate="many_to_one")
    assert len(d)==len(balls) and d.row_id.tolist()==state_row_order
    return d, match_audit, strength_tables, recovery_episodes, canonical_roster

In [ ]:
def construct_micro_features(model_data, raw_balls):
    game=model_data.copy()
    game['date']=pd.to_datetime(game['date'])
    source=raw_balls.copy()
    for c in ['match_id','striker_id','non_striker_id','bowler_id']:
        source[c]=source[c].astype(str)
    for c in ['innings','innings_ball_num','runs']:
        source[c]=pd.to_numeric(source[c],errors='raise')
    source['row_id']=source.match_id+':'+source.innings.astype(int).astype(str)+':'+source.innings_ball_num.astype(int).astype(str)
    fields=['row_id','delivery_style','stroke']
    assert source.row_id.is_unique
    d=game.merge(source[fields],on='row_id',how='left',validate='one_to_one')
    if d.delivery_style.isna().any():raise ValueError('Raw ball source does not cover every retained delivery.')
    d=d.sort_values(['date','match_id','innings','innings_ball_num']).reset_index(drop=True)
    variations=sorted(set(d.delivery_style)-{'','None'})
    shots=sorted(set(d.stroke.dropna())-{'','None'})
    def slug(label):
        return re.sub(r'[^a-z0-9]+','_',label.lower()).strip('_')+'_'+hashlib.sha1(label.encode()).hexdigest()[:5]
    batting_against=defaultdict(lambda:defaultdict(lambda:np.zeros(3)))
    bowling_to_shot=defaultdict(lambda:defaultdict(lambda:np.zeros(3)))
    batter_shots=defaultdict(lambda:defaultdict(float))
    bowler_variations=defaultdict(lambda:defaultdict(float))
    bowler_variation_outcomes=defaultdict(lambda:defaultdict(lambda:np.zeros(3)))
    snapshots=[]
    for date,day in d.groupby('date',sort=True):
        required=set(day.striker_id)|set(day.non_striker_id)|set(day.bowler_id)
        for pid in required:
            record={'date':date,'player_id':pid}
            for label in variations:
                v=batting_against[pid][label];stem='vs_variation_'+slug(label)
                record[stem+'_events']=v[0]
                record[stem+'_runs']=v[1]/v[0] if v[0] else np.nan
                record[stem+'_dismissals']=v[2]/v[0] if v[0] else np.nan
                b=bowler_variation_outcomes[pid][label];stem='variation_'+slug(label)
                record[stem+'_events']=b[0]
                record[stem+'_runs_conceded']=b[1]/b[0] if b[0] else np.nan
                record[stem+'_wickets']=b[2]/b[0] if b[0] else np.nan
            for label in shots:
                v=bowling_to_shot[pid][label];stem='vs_shot_'+slug(label)
                record[stem+'_events']=v[0]
                record[stem+'_runs_conceded']=v[1]/v[0] if v[0] else np.nan
                record[stem+'_wickets']=v[2]/v[0] if v[0] else np.nan
                record['shot_'+slug(label)+'_events']=batter_shots[pid][label]
            snapshots.append(record)
        for row in day.itertuples():
            if row.delivery_style not in ['', 'None']:
                batting_against[row.striker_id][row.delivery_style]+=np.array([1,row.y_batter_runs,row.y_striker_dismissed])
                bowler_variations[row.bowler_id][row.delivery_style]+=1
                bowler_variation_outcomes[row.bowler_id][row.delivery_style]+=np.array([1,row.y_bowler_runs,row.y_bowler_wicket])
            if row.stroke not in ['', 'None'] and pd.notna(row.stroke):
                bowling_to_shot[row.bowler_id][row.stroke]+=np.array([1,row.y_bowler_runs,row.y_bowler_wicket])
                batter_shots[row.striker_id][row.stroke]+=1
    history=pd.DataFrame(snapshots)
    result=d[['row_id','date','striker_id','non_striker_id','bowler_id']].copy()
    for role,pidcol,include in [
        ('batter','striker_id',('vs_variation_','shot_')),
        ('partner','non_striker_id',('vs_variation_','shot_')),
        ('bowler','bowler_id',('variation_','vs_shot_')),
    ]:
        selected=[c for c in history if c.startswith(include)]
        h=history[['date','player_id']+selected].rename(columns={'player_id':pidcol,
            **{c:'x_micro_'+role+'_'+c for c in selected}})
        result=result.merge(h,on=['date',pidcol],how='left',validate='many_to_one')
    vulnerability=np.full(len(result),np.nan);dismissal_risk=np.full(len(result),np.nan);coverage=np.zeros(len(result))
    total_variations=np.zeros(len(result));weighted_runs=np.zeros(len(result));weighted_outs=np.zeros(len(result))
    for label in variations:
        key=slug(label)
        probability=result['x_micro_bowler_variation_'+key+'_events'].to_numpy(dtype=float)
        supported=result['x_micro_batter_vs_variation_'+key+'_events'].to_numpy(dtype=float)
        runs=result['x_micro_batter_vs_variation_'+key+'_runs'].to_numpy(dtype=float)
        outs=result['x_micro_batter_vs_variation_'+key+'_dismissals'].to_numpy(dtype=float)
        total_variations+=probability
        usable=(supported>=6)&np.isfinite(runs)
        coverage+=np.where(usable,probability,0)
        weighted_runs+=np.where(usable,probability*np.nan_to_num(runs),0)
        weighted_outs+=np.where(usable,probability*np.nan_to_num(outs),0)
    result['x_micro_repertoire_matchup_runs_proxy']=np.divide(weighted_runs,coverage,out=vulnerability,where=coverage>0)
    result['x_micro_repertoire_matchup_dismissal_proxy']=np.divide(weighted_outs,coverage,out=dismissal_risk,where=coverage>0)
    result['x_micro_repertoire_matchup_coverage']=np.divide(coverage,total_variations,out=np.zeros(len(result)),where=total_variations>0)
    known=np.zeros(len(result));shot_total=np.zeros(len(result));weighted=np.zeros(len(result))
    for label in shots:
        key=slug(label);count=result['x_micro_batter_shot_'+key+'_events'].to_numpy(dtype=float)
        n=result['x_micro_bowler_vs_shot_'+key+'_events'].to_numpy(dtype=float)
        runs=result['x_micro_bowler_vs_shot_'+key+'_runs_conceded'].to_numpy(dtype=float)
        shot_total+=count;use=(n>=6)&np.isfinite(runs)
        known+=np.where(use,count,0);weighted+=np.where(use,count*np.nan_to_num(runs),0)
    result['x_micro_shot_matchup_runs_proxy']=np.divide(weighted,known,out=np.full(len(result),np.nan),where=known>0)
    result['x_micro_shot_matchup_coverage']=np.divide(known,shot_total,out=np.zeros(len(result)),where=shot_total>0)
    seq=[]
    for key,inning in d.groupby(['match_id','innings'],sort=False):
        last_variations=defaultdict(lambda:deque(maxlen=6));last_shots=defaultdict(lambda:deque(maxlen=6))
        for row in inning.itertuples():
            bv=list(last_variations[row.bowler_id]);bs=list(last_shots[row.striker_id])
            seq.append({'row_id':row.row_id,
                'x_micro_previous_bowler_variation':bv[-1] if bv else '__NO_PRIOR_DELIVERY__',
                'x_micro_previous_bowler_variation_2':bv[-2] if len(bv)>1 else '__NO_PRIOR_DELIVERY__',
                'x_micro_last6_bowler_variation_changes':sum(a!=b for a,b in zip(bv,bv[1:])),
                'x_micro_previous_batter_shot':bs[-1] if bs else '__NO_PRIOR_DELIVERY__',
                'x_micro_last6_batter_shot_diversity':len(set(bs))})
            last_variations[row.bowler_id].append(row.delivery_style)
            last_shots[row.striker_id].append(row.stroke if pd.notna(row.stroke) else '__UNKNOWN__')
    result=result.merge(pd.DataFrame(seq),on='row_id',validate='one_to_one')
    result=result[['row_id']+[c for c in result if c.startswith('x_micro_')]]
    result=result.merge(d[['row_id','delivery_style','stroke']].rename(columns={'delivery_style':'audit_delivery_style','stroke':'audit_stroke'}),on='row_id',validate='one_to_one')
    assert len(result)==len(model_data) and result.row_id.is_unique
    metadata={'feature_columns':[c for c in result if c.startswith('x_micro_')],'audit_columns':['audit_delivery_style','audit_stroke'],'variations':variations,'shots':shots,
        'history_cutoff':'strictly earlier match dates; sequence features use strictly earlier deliveries',
        'rate_definition':'recorded delivery events; batting dismissals concern striker dismissal events',
        'source_matches':'only the audited match/row population in constructed_data_v2',
        'timing':'no current shot or delivery variation is used as its own pre-ball input'}
    return result,metadata

In [ ]:
def add_speed_history(game, source):
    speed = pd.to_numeric(source["bowl_speed"], errors="coerce")
    history = game[["date", "bowler_id"]].copy()
    history["speed"] = speed.where(speed.between(35, 175)).to_numpy()
    daily = history.groupby(["bowler_id", "date"], sort=True)["speed"].agg(
        speed_sum="sum", speed_observations="count"
    ).reset_index()
    keys = ["speed_sum", "speed_observations"]
    prior = daily.groupby("bowler_id", sort=False)[keys].cumsum() - daily[keys]
    daily["x_bowler_speed_observations_prior"] = prior["speed_observations"]
    daily["x_bowler_mean_speed_prior"] = prior["speed_sum"].div(
        prior["speed_observations"].where(prior["speed_observations"] >= 12)
    )
    return game.merge(
        daily.drop(columns=keys), on=["date", "bowler_id"],
        validate="many_to_one", sort=False,
    )


def normalized(series):
    return series.clip(0, 1)


def weighted_available(parts):
    data = pd.concat([x for x, _ in parts], axis=1)
    weights = np.asarray([w for _, w in parts], dtype=float)
    total = data.notna().mul(weights).sum(axis=1)
    result = data.mul(weights).sum(axis=1).div(total.where(total > 0))
    return normalized(result)


def add_specialist_inputs(game):
    chase = game["innings"].eq(2)
    usage = game["x_legal_balls_used"]
    balls_left = game["x_balls_remaining"]
    required_rate = game["x_required_run_rate"]
    historical = game["x_historical_chase_success_probability"]
    fallback = 1 - normalized((required_rate - 5.0) / 10.0)
    historical = historical.combine_first(fallback)
    trajectory_gap = game["x_score_minus_trajectory_proxy"].combine_first(
        game["x_team_runs_before"] - 8.0 * usage / 6.0
    )
    finish = (4 * historical * (1 - historical) * (1 - balls_left / 120))
    finish = finish.where(chase & balls_left.le(36))
    late_wickets = normalized(game["x_wickets_previous_12_legal_ball_window"] / 3)
    lost_wickets = normalized(game["x_wickets_lost_before"] / 8)
    support = game["x_remaining_top3_strength_score100_proxy"] / 100
    no_support = game["x_resource_pool_valid"].eq(True) & game[
        "x_remaining_batters_pool_count"
    ].eq(0)
    support = support.mask(no_support, 0.0)
    best_alternatives = game["x_other_bowling_top3_strength_proxy"] / 100
    raw_style = game["x_bowling_style"].str.lower()
    game["opposition_style_batter"] = np.select(
        [
            raw_style.str.contains(r"leg.break|chinaman", regex=True, na=False),
            raw_style.str.contains(r"off.break|offbreak", regex=True, na=False),
            raw_style.str.contains("orthodox", na=False),
            game["x_pace_or_spin"].str.casefold().eq("pace"),
        ],
        ["leg_spin", "off_spin", "left_arm_orthodox", "pace"],
        default="unknown",
    )
    game["opposition_style_bowler"] = game["x_batting_hand"].where(
        game["x_batting_hand"].isin(["RHB", "LHB"]), "unknown"
    )
    batter_threat = normalized(
        0.55 * game["x_striker_runs_before"] / 45
        + 0.45 * game["x_striker_nonwide_deliveries_before"] / 30
    )
    bowler_threat = normalized(
        0.50 + 0.17 * game["x_bowler_wickets_before"]
        + (1.2 * game["x_bowler_legal_balls_before"]
           - game["x_bowler_runs_before"]) / 24.0
    ).where(game["x_bowler_legal_balls_before"].gt(0))
    last_team_runs = game["x_runs_previous_6_legal_ball_window"]
    attack = normalized((last_team_runs - 5) / 13)
    bowler_under_fire = weighted_available([
        (normalized(game["x_bowler_previous_6_events_runs"] / 18), 0.6),
        (normalized(game["x_bowler_previous_6_events_boundaries"] / 3), 0.4),
    ])
    for role in ROLE_IDS:
        batting = role == "batter"
        opponent = "bowler" if batting else "batter"
        strength = game[f"x_{role}_strength_score100_prior"] / 10
        opponent_strength = game[f"x_{opponent}_strength_score100_prior"] / 10
        game[f"x_{role}_base_ability10"] = strength
        game[f"x_{role}_opponent_ability10"] = opponent_strength
        form = game[f"x_{role}_form_context_gap_prior"]
        opponent_form = game[f"x_{opponent}_form_context_gap_prior"]
        spell_threat = bowler_threat if batting else batter_threat
        game[f"x_{role}_opponent_form"] = opponent_form
        game[f"x_{role}_opponent_live_threat"] = spell_threat
        threat_now = weighted_available([
            (opponent_strength / 10, 0.65),
            (normalized(0.5 + opponent_form / 1.2), 0.2),
            (spell_threat, 0.15),
        ]).where(opponent_strength.notna())
        demand = 1 - historical if batting else historical
        first_innings_demand = normalized(
            (-trajectory_gap if batting else trajectory_gap) / 30
        )
        resources = (
            weighted_available([(lost_wickets, 0.35), (1-support, 0.40),
                                (late_wickets, 0.25)])
            if batting else weighted_available([
                (1-lost_wickets, 0.25), (support, 0.30),
                (1-best_alternatives, 0.30), (attack, 0.15),
            ])
        )
        momentum = (
            weighted_available([
                (normalized(game["x_consecutive_team_dot_events"] / 5), 0.40),
                (late_wickets, 0.30), (1-attack, 0.30),
            ]) if batting else weighted_available([
                (attack, 0.35), (bowler_under_fire, 0.40),
                (normalized(trajectory_gap / 35), 0.25),
            ])
        )
        form_slump = normalized(-form / 0.65)
        stakes = np.maximum(
            form_slump.fillna(0), 0.75 * game["x_knockout"].astype(float)
        ).where(form_slump.notna() | game["x_knockout"].eq(1))
        values = (demand.where(chase, first_innings_demand), finish,
                  resources, threat_now, momentum, stakes)
        for scenario, value in zip(PRIMARY_SCENARIOS, values):
            game[pressure_column(role, scenario)] = (value * 100).astype("float32")
    return game


def pressure_column(role, scenario):
    return f"x_pressure_{role}_{scenario}"


def data_audit(game, source):
    assert game["row_id"].is_unique and len(game) == len(source)
    assert game["row_id"].eq(source["row_id"].reset_index(drop=True)).all()
    assert game["x_balls_remaining"].between(1, 120).all()
    assert game["x_wickets_lost_before"].between(0, 9).all()
    chase = game["innings"].eq(2)
    assert game.loc[chase, "x_runs_required"].gt(0).all()
    assert game.loc[~chase, "x_target"].isna().all()
    assert game.groupby("match_id")["date"].nunique().eq(1).all()
    assert game["y_bowler_runs"].le(game["y_team_runs"]).all()
    history = game[["date", "bowler_id", "x_bowler_speed_observations_prior"]]
    assert history.groupby(["date", "bowler_id"])[
        "x_bowler_speed_observations_prior"
    ].nunique().le(1).all()
    for role in ROLE_IDS:
        assert game[f"x_{role}_base_ability10"].dropna().between(0, 10).all()
        for scenario in PRIMARY_SCENARIOS:
            values = game[pressure_column(role, scenario)].dropna()
            assert len(values) and values.between(0, 100.00001).all()


def construct_dataset(paths):
    raw = {
        name: pd.read_csv(path, dtype=str, keep_default_na=False)
        for name, path in paths.items()
    }
    all_states, audit, ability, episodes, roster = build_audited_state(raw)
    source = all_states.loc[all_states["primary_model_eligible"]].copy()
    source = source.reset_index(drop=True)
    assert len(source) > 0, "No matches passed the supplied-data accounting audit."
    identity = [
        "row_id", "date", "match_id", "comp_id", "innings",
        "innings_ball_num", "batting_team_id", "bowling_team_id",
        "striker_id", "striker_name", "non_striker_id", "non_striker_name",
        "bowler_id", "bowler_name",
    ]
    fields = [c for c in source if c.startswith(("x_", "y_"))]
    game = source[identity + fields].copy()
    mapping = raw["competition_team_player_mapping"].drop_duplicates("player_id")
    names = mapping.set_index("player_id")["player_name"].replace("", np.nan)
    for participant in ("striker", "non_striker", "bowler"):
        name = f"{participant}_name"
        game[name] = game[f"{participant}_id"].map(names).fillna(game[name])
    team_fields = ["comp_id", "team_id", "team_name"]
    teams = raw["teams"][team_fields].drop_duplicates()
    assert not teams.duplicated(["comp_id", "team_id"]).any()
    for side in ("batting", "bowling"):
        team_id = f"{side}_team_id"
        team_lookup = teams.rename(columns={"team_id": team_id, "team_name": f"{side}_team"})
        game = game.merge(
            team_lookup, on=["comp_id", team_id], how="left", validate="many_to_one"
        )
        game[f"{side}_team"] = game[f"{side}_team"].fillna(game[team_id])
    micro, micro_metadata = construct_micro_features(game, raw["ball_by_ball"])
    assert len(micro_metadata["feature_columns"]) > 0
    game = game.merge(micro, on="row_id", how="left", validate="one_to_one")
    game = add_speed_history(game, source)
    game = add_specialist_inputs(game)
    for role in ROLE_IDS:
        history = ability[role].copy()
        history["role"] = role
        history["ability10"] = history["strength_score100_prior"] / 10
        history["player_name"] = history["player_id"].map(names)
        write_parquet(OUTPUT_DIR / f"{role}_ability_by_date.parquet", history)
    for column in game.select_dtypes(include=["float64", "float32"]):
        game[column] = game[column].astype("float32")
    data_audit(game, source)
    write_csv(OUTPUT_DIR / "match_audit.csv", audit)
    write_parquet(OUTPUT_DIR / "recovery_episodes.parquet", episodes)
    write_parquet(OUTPUT_DIR / "canonical_roster.parquet", roster)
    write_json(OUTPUT_DIR / "micro_manifest.json", micro_metadata)
    write_csv(OUTPUT_DIR / "feature_manifest.csv", pd.DataFrame({
        "column": game.columns,
        "input": game.columns.str.startswith("x_"),
        "dtype": game.dtypes.astype(str).to_numpy(),
        "missing": game.isna().sum().to_numpy(),
    }))
    return game

In [ ]:
SCENARIO_CATALOG = {
    "batter": {
        "match_demand": {
            "scenario": "Match demand",
            "family": "Match demand",
            "rule": "Runs, balls, wickets and the normal trajectory versus the current state. Uses the notebook's frozen high-pressure cutoff.",
            "pressure": "The innings is moving away from the required or expected scoring trajectory.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "close_finish": {
            "scenario": "Close finish",
            "family": "Close finishes",
            "rule": "Late chase with competitive historical win odds. Uses the notebook's frozen high-pressure cutoff.",
            "pressure": "In a late, competitive chase, a single ball can materially change the match.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "limited_support": {
            "scenario": "Collapse or limited support",
            "family": "Wickets and support",
            "rule": "Wickets and batting or bowling alternatives available. Uses the notebook's frozen high-pressure cutoff.",
            "pressure": "Losing the striker is more costly when few credible batters remain.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "strong_opposition": {
            "scenario": "Strong opposition",
            "family": "Opposition and matchups",
            "rule": "Earlier opponent ability and form, plus current-match threat. Uses the notebook's frozen high-pressure cutoff.",
            "pressure": "A proven opponent in form or already threatening reduces the margin for error.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "sustained_attack": {
            "scenario": "Sustained attack",
            "family": "Momentum and recovery",
            "rule": "Recent dots or wickets for the batter, recent attacks for the bowler. Uses the notebook's frozen high-pressure cutoff.",
            "pressure": "Dot balls and wickets make it difficult to accelerate without risking another dismissal.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "form_or_stakes": {
            "scenario": "Personal form or knockout stakes",
            "family": "Personal form and stakes",
            "rule": "Recent underperformance or recorded knockout stakes. Uses the notebook's frozen high-pressure cutoff.",
            "pressure": "A measurable form slump or a recorded knockout creates a specific test of performance.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "recovery_after_slow_powerplay": {
            "scenario": "Recovering after a slow team powerplay",
            "family": "Momentum and recovery",
            "rule": "Over 7 onward when the team ended six overs at least six runs below the mean of earlier powerplays from the same competition group and innings; at least eight earlier innings.",
            "pressure": "The remaining overs have to make up for a team scoring shortfall.",
            "measure": "Batter run-rate lift versus pre-ball context. Track dismissals separately.",
            "source": "specialist",
            "prefix": "pace_"
        },
        "death_over_acceleration": {
            "scenario": "Accelerating in overs 17 to 20",
            "family": "Match demand",
            "rule": "Overs 17–20 with a first-innings trajectory deficit of at least 8, a chase at 9+ RPO or 25–75% earlier-state win odds, or a knockout. Any striker.",
            "pressure": "The batting side has four overs left, with a scoring shortfall, urgent chase or close finish.",
            "measure": "Batter run-rate lift versus pre-ball context. Track dismissals separately.",
            "source": "refined_specialist",
            "prefix": "pace_"
        },
        "new_ball_after_early_wickets": {
            "scenario": "Facing the new ball after early wickets",
            "family": "Wickets and support",
            "rule": "First five overs after at least one wicket has fallen; multiple wickets also qualify.",
            "pressure": "An early wicket forces the batting side to score while protecting its remaining resources.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "switch_pace_spin": {
            "scenario": "Switching between pace and spin",
            "family": "Opposition and matchups",
            "rule": "Bowling changes between pace and spin since the striker last faced. At least one original pressure index must also be high.",
            "pressure": "A striker facing an existing pressure state has to reset timing and shot selection against a different pace.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "set_with_fresh_partner": {
            "scenario": "Set batter with a fresh partner",
            "family": "Wickets and support",
            "rule": "Striker has faced at least 20 non-wide balls and partner fewer than six. At least one original pressure index must also be high.",
            "pressure": "Much of the scoring burden sits with the established batter while the partner settles.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "acceleration_after_dot_buildup": {
            "scenario": "Scoring after a dot-heavy 15-ball stretch",
            "family": "Momentum and recovery",
            "rule": "At least eight team dot balls among the striker's preceding 15 legal balls faced in the same innings. The following attempt is evaluated.",
            "pressure": "The striker has used a substantial share of recent balls without scoring.",
            "measure": "Batter run-rate lift versus pre-ball context. Track dismissals separately.",
            "source": "specialist",
            "prefix": "pace_"
        },
        "rebuild_after_wicket": {
            "scenario": "Established batter rebuilding after a wicket",
            "family": "Wickets and support",
            "rule": "Overs 7 to 16, after at least 12 non-wide balls faced and with no more than three legal balls elapsed since a team wicket.",
            "pressure": "The established striker needs to avoid compounding the wicket while keeping the innings moving.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "rising_chase_rate": {
            "scenario": "Batting when the required rate is rising",
            "family": "Match demand",
            "rule": "After the powerplay with at least 18 legal balls left, the required rate has risen by at least one run per over from 12 legal balls earlier.",
            "pressure": "A useful scoring pace two overs ago now falls short of what the chase needs.",
            "measure": "Batter runs against the pre-ball pace needed. Dismissal remains a separate cost.",
            "source": "specialist",
            "prefix": "pace_"
        },
        "fresh_bowler_post_powerplay": {
            "scenario": "Facing a fresh bowler introduced after the powerplay",
            "family": "Opposition and matchups",
            "rule": "The current bowler was introduced after 36 legal balls and has used fewer than six. At least one original pressure index must also be high.",
            "pressure": "A batter already under pressure must adjust to a bowler with little same-match evidence to read.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "steep_chase_with_two_to_four_overs_left": {
            "scenario": "Steep chase with two to four overs left",
            "family": "Close finishes",
            "rule": "12–24 legal balls left and at least two runs needed per ball.",
            "pressure": "Boundary scoring is needed with little time to recover a dot or dismissal.",
            "measure": "Batter runs against the pre-ball pace needed. Dismissal remains a separate cost.",
            "source": "shared_family",
            "prefix": "pace_"
        },
        "reachable_last_over_chase": {
            "scenario": "Reachable last-over chase",
            "family": "Close finishes",
            "rule": "In the final over, 7–12 required off six balls at its start; score all subsequent attempts in that over.",
            "pressure": "The chase is within reach, but a single dot or wicket can shift the balance.",
            "measure": "Batter runs against the pre-ball pace needed. Dismissal remains a separate cost.",
            "source": "shared_family",
            "prefix": "pace_"
        },
        "last_over_starts_needing_13_or_more": {
            "scenario": "Last over starts needing 13 or more",
            "family": "Close finishes",
            "rule": "In the final over after it started with 13+ needed off six.",
            "pressure": "The batter is likely to need multiple boundaries against a defense expecting them.",
            "measure": "Batter runs against the pre-ball pace needed. Dismissal remains a separate cost.",
            "source": "shared_family",
            "prefix": "pace_"
        },
        "boundary_needed_off_the_last_ball": {
            "scenario": "Boundary needed off the last ball",
            "family": "Close finishes",
            "rule": "One legal ball left and 4–6 runs required.",
            "pressure": "The innings depends on one boundary-capable opportunity.",
            "measure": "Batter runs against the pre-ball pace needed. Dismissal remains a separate cost.",
            "source": "shared_family",
            "prefix": "pace_"
        },
        "low_runs_needed_with_the_last_wicket": {
            "scenario": "Low runs needed with the last wicket",
            "family": "Close finishes",
            "rule": "1–6 runs to win, up to 12 legal balls left, and exactly one wicket left.",
            "pressure": "Even routine scoring must be attempted with no batting cover.",
            "measure": "Batter runs against the pre-ball pace needed. Dismissal remains a separate cost.",
            "source": "shared_family",
            "prefix": "pace_"
        },
        "large_chase_at_the_opening": {
            "scenario": "Large chase at the opening",
            "family": "Match demand",
            "rule": "First 12 legal balls of a chase with a target of 190 or higher.",
            "pressure": "The target demands an aggressive tempo from the opening overs.",
            "measure": "Batter runs against the pre-ball pace needed. Dismissal remains a separate cost.",
            "source": "shared_family",
            "prefix": "pace_"
        },
        "comfortable_chase_turning_against_the_batting_side": {
            "scenario": "Comfortable chase turning against the batting side",
            "family": "Match demand",
            "rule": "After the powerplay with 18+ balls left, earlier-state win odds moved from at least 65% 12 balls ago to 50% or lower.",
            "pressure": "A once-manageable chase has deteriorated and now requires a change in approach.",
            "measure": "Batter runs against the pre-ball pace needed. Dismissal remains a separate cost.",
            "source": "shared_family",
            "prefix": "pace_"
        },
        "ahead_of_the_rate_but_running_out_of_wickets": {
            "scenario": "Ahead of the rate but running out of wickets",
            "family": "Match demand",
            "rule": "Chase after six overs, 2–3 wickets left, and current rate at least 1 RPO above required rate.",
            "pressure": "The scoring pace is manageable, but protecting the few remaining wickets is critical.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "steep_required_rate_jump_in_the_middle_overs": {
            "scenario": "Steep required-rate jump in the middle overs",
            "family": "Match demand",
            "rule": "Overs 7–16, required rate at least 10 RPO and up at least 2.5 from 12 legal balls earlier.",
            "pressure": "The batter has to change scoring gears before the end of the innings.",
            "measure": "Batter runs against the pre-ball pace needed. Dismissal remains a separate cost.",
            "source": "shared_family",
            "prefix": "pace_"
        },
        "first_innings_well_below_its_scoring_trajectory": {
            "scenario": "First innings well below its scoring trajectory",
            "family": "Match demand",
            "rule": "Overs 10–16 with a score at least 15 runs below the earlier same-context trajectory.",
            "pressure": "The batting side has limited overs to recover a meaningful shortfall.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "death_overs_with_wickets_in_hand_but_a_scoring_deficit": {
            "scenario": "Death overs with wickets in hand but a scoring deficit",
            "family": "Match demand",
            "rule": "First innings, overs 17–20, 6+ wickets left, and at least 12 runs behind trajectory.",
            "pressure": "Unused batting depth creates a clear obligation to accelerate immediately.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "new_batter_walking_into_a_middle_overs_collapse": {
            "scenario": "New batter walking into a middle-overs collapse",
            "family": "Wickets and support",
            "rule": "Overs 7–16, striker has faced at most 3 non-wide attempts and 2+ wickets fell in the previous 12 legal balls.",
            "pressure": "A new batter must score while preventing recent wickets from becoming a full collapse.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "established_batter_loses_the_other_established_batter": {
            "scenario": "Established batter loses the other established batter",
            "family": "Wickets and support",
            "rule": "On strike immediately following a team wicket, with the striker having faced 16+ non-wides, the previous other batter gone, and the preceding pair having faced 24+ legal balls.",
            "pressure": "A set batting pair was broken and the survivor now carries the scoring load.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "chasing_with_the_tail_and_no_deeper_support": {
            "scenario": "Chasing with the tail and no deeper support",
            "family": "Wickets and support",
            "rule": "Post-powerplay chase at 8+ RPO with 2–4 wickets left and at least two rated remaining batters with a top-three ability proxy at most 45/100.",
            "pressure": "The striker must sustain the rate because a wicket exposes historically weak support.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "last_batting_pair_still_needs_a_meaningful_chase": {
            "scenario": "Last batting pair still needs a meaningful chase",
            "family": "Wickets and support",
            "rule": "Exactly one wicket left with at least 7 runs required and between 6 and 30 legal balls remaining.",
            "pressure": "The pair must keep scoring without any replacement batter available.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "double_strike_before_the_innings_settles": {
            "scenario": "Double strike before the innings settles",
            "family": "Wickets and support",
            "rule": "First three overs with two or more team wickets already down.",
            "pressure": "Batters must break the early wicket run while facing most of the powerplay still to come.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "new_finisher_exposed_late_in_a_demanding_chase": {
            "scenario": "New finisher exposed late in a demanding chase",
            "family": "Wickets and support",
            "rule": "Overs 19–20 with at least 9 RPO needed and striker has faced at most two non-wide attempts.",
            "pressure": "There is almost no time to assess the bowler before taking scoring risks.",
            "measure": "Batter runs against the pre-ball pace needed. Dismissal remains a separate cost.",
            "source": "shared_family",
            "prefix": "pace_"
        },
        "team_scoring_squeeze_after_the_powerplay": {
            "scenario": "Team scoring squeeze after the powerplay",
            "family": "Momentum and recovery",
            "rule": "Overs 7–16 with 10 runs or fewer scored over the previous 12 legal balls and either a first-innings deficit of 8+ or an 8+ RPO chase.",
            "pressure": "The whole side has been stalled and the striker needs to release the pressure.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "dots_at_the_end_of_a_tight_chase": {
            "scenario": "Dots at the end of a tight chase",
            "family": "Momentum and recovery",
            "rule": "At least three preceding team dots, 6–24 legal balls left and at least 7 RPO needed.",
            "pressure": "Recent scoreless balls have reduced the number of remaining scoring opportunities.",
            "measure": "Batter runs against the pre-ball pace needed. Dismissal remains a separate cost.",
            "source": "shared_family",
            "prefix": "pace_"
        },
        "two_over_drought_after_an_early_wicket": {
            "scenario": "Two-over drought after an early wicket",
            "family": "Momentum and recovery",
            "rule": "After at least 12 legal balls and within the first six overs, with a wicket down and at most 10 runs added over the prior 12 legal balls.",
            "pressure": "The early wicket has been followed by a measurable scoring freeze despite powerplay opportunity.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "established_batter_stuck_below_a_run_a_ball": {
            "scenario": "Established batter stuck below a run a ball",
            "family": "Momentum and recovery",
            "rule": "Striker has faced 18+ non-wides at under one run per attempt, with 9+ RPO required in a chase or an 8+ first-innings trajectory deficit.",
            "pressure": "An established striker is using deliveries the team now needs to score from.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "high_ability_pace_in_a_demanding_innings": {
            "scenario": "High-ability pace in a demanding innings",
            "family": "Opposition and matchups",
            "rule": "Recorded pace, opponent ability at least 6.2/10 and at least one original pressure index is high.",
            "pressure": "The batter must score against historically stronger pace while the match state already requires it.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "high_ability_off_spin_in_a_demanding_innings": {
            "scenario": "High-ability off-spin in a demanding innings",
            "family": "Opposition and matchups",
            "rule": "Recorded right-arm off-break, opponent ability at least 6.2/10 and at least one original pressure index is high.",
            "pressure": "Performance against a well-rated off-spinner tests a distinct turn and scoring-angle matchup.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "high_ability_wrist_spin_in_a_demanding_innings": {
            "scenario": "High-ability wrist spin in a demanding innings",
            "family": "Opposition and matchups",
            "rule": "Recorded right-arm leg-break or left-arm wrist spin, opponent ability at least 6.2/10, and at least one original pressure index is high.",
            "pressure": "The batter needs to score against a strong wrist spinner while reading turn and variation.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "high_ability_left_arm_orthodox_under_pressure": {
            "scenario": "High-ability left-arm orthodox under pressure",
            "family": "Opposition and matchups",
            "rule": "Recorded slow left-arm orthodox, opponent ability at least 6.2/10 and at least one original pressure index is high.",
            "pressure": "The batter must adapt to this angle-and-turn matchup against a proven spinner.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "high_velocity_pace_after_an_early_wicket": {
            "scenario": "High-velocity pace after an early wicket",
            "family": "Opposition and matchups",
            "rule": "First six overs with a wicket already lost against a pace bowler averaging at least 138 km/h from 20+ prior measured balls.",
            "pressure": "Less reaction time increases the difficulty of restoring the innings after an early wicket.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "proven_bowler_also_dangerous_in_this_spell": {
            "scenario": "Proven bowler also dangerous in this spell",
            "family": "Opposition and matchups",
            "rule": "Opponent has 6.2+ ability, recent form gap at least +0.15 and current-match threat at least 0.65.",
            "pressure": "The bowler combines long-run quality, recent form and evidence of a threatening spell.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "facing_a_bowler_who_has_repeatedly_dismissed_you": {
            "scenario": "Facing a bowler who has repeatedly dismissed you",
            "family": "Opposition and matchups",
            "rule": "At least 18 earlier head-to-head attempts, 10% or higher striker dismissal rate and an original pressure index high.",
            "pressure": "The batter has a measured matchup risk before the delivery.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "facing_a_style_that_has_suppressed_your_scoring": {
            "scenario": "Facing a style that has suppressed your scoring",
            "family": "Opposition and matchups",
            "rule": "At least 30 prior attempts against the recorded style, style scoring at most 80% of personal career scoring per attempt, and a chase at 8+ RPO or an 8+ first-innings deficit.",
            "pressure": "Match demand forces scoring against a style that has historically constrained the individual.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "variable_repertoire_bowler_in_an_urgent_chase": {
            "scenario": "Variable-repertoire bowler in an urgent chase",
            "family": "Opposition and matchups",
            "rule": "After the powerplay in a chase at 9+ RPO, the current bowler has 4+ effective prior variation categories and recorded category coverage of at least 40%.",
            "pressure": "The batter must score quickly against an opponent with multiple documented options.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "personal_batting_slump_with_a_demanding_chase": {
            "scenario": "Personal batting slump with a demanding chase",
            "family": "Personal form and stakes",
            "rule": "Recorded pre-match batting form slump, at least three recent matches available, and a post-powerplay chase at 9+ RPO.",
            "pressure": "The batter has to meet a clear scoring requirement despite recent underperformance.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "scoring_burden_in_the_knockout_powerplay": {
            "scenario": "Scoring burden in the knockout powerplay",
            "family": "Personal form and stakes",
            "rule": "Recorded knockout, first six overs, with a wicket down or a chase requiring 9+ RPO.",
            "pressure": "Early mistakes carry elimination stakes while the innings already has a material constraint.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "knockout_chase_at_the_death": {
            "scenario": "Knockout chase at the death",
            "family": "Personal form and stakes",
            "rule": "Recorded knockout, overs 17–20 and chase odds between 20% and 80% based on earlier states.",
            "pressure": "The match is still live and there are too few balls left to recover from poor execution.",
            "measure": "Batter runs against the pre-ball pace needed. Dismissal remains a separate cost.",
            "source": "shared_family",
            "prefix": "pace_"
        },
        "knockout_first_innings_recovery_after_a_collapse": {
            "scenario": "Knockout first-innings recovery after a collapse",
            "family": "Personal form and stakes",
            "rule": "Recorded knockout, first innings after six overs and 2+ wickets in the prior 12 legal balls.",
            "pressure": "The batter must rebuild a first-innings total after a collapse in an elimination match.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "knockout_batter_facing_a_proven_bowler_in_form": {
            "scenario": "Knockout batter facing a proven bowler in form",
            "family": "Personal form and stakes",
            "rule": "Recorded knockout, bowler rated at least 6.2/10 with a recent positive form gap of at least +0.15, and a high original match-demand or limited-support index.",
            "pressure": "Elimination stakes and the adverse match state compound the difficulty of the matchup.",
            "measure": "Runs and dismissal versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        }
    },
    "bowler": {
        "match_demand": {
            "scenario": "Match demand",
            "family": "Match demand",
            "rule": "Runs, balls, wickets and the normal trajectory versus the current state. Uses the notebook's frozen high-pressure cutoff.",
            "pressure": "The chasing side or first-innings batters are scoring faster than the fielding side can afford.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "close_finish": {
            "scenario": "Close finish",
            "family": "Close finishes",
            "rule": "Late chase with competitive historical win odds. Uses the notebook's frozen high-pressure cutoff.",
            "pressure": "In a late, competitive chase, a single ball can materially change the match.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "limited_support": {
            "scenario": "Collapse or limited support",
            "family": "Wickets and support",
            "rule": "Wickets and batting or bowling alternatives available. Uses the notebook's frozen high-pressure cutoff.",
            "pressure": "With few credible bowling alternatives, the current bowler has more responsibility for the innings.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "strong_opposition": {
            "scenario": "Strong opposition",
            "family": "Opposition and matchups",
            "rule": "Earlier opponent ability and form, plus current-match threat. Uses the notebook's frozen high-pressure cutoff.",
            "pressure": "A proven opponent in form or already threatening reduces the margin for error.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "sustained_attack": {
            "scenario": "Sustained attack",
            "family": "Momentum and recovery",
            "rule": "Recent dots or wickets for the batter, recent attacks for the bowler. Uses the notebook's frozen high-pressure cutoff.",
            "pressure": "A scoring surge can remove the fielding side's remaining defensive margin.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "form_or_stakes": {
            "scenario": "Personal form or knockout stakes",
            "family": "Personal form and stakes",
            "rule": "Recent underperformance or recorded knockout stakes. Uses the notebook's frozen high-pressure cutoff.",
            "pressure": "A measurable form slump or a recorded knockout creates a specific test of performance.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "first_innings_death": {
            "scenario": "First-innings death-over containment",
            "family": "Match demand",
            "rule": "At least 96 legal balls used and the batting side has five or more wickets left. At least one original pressure index must also be high.",
            "pressure": "The batting side has enough wickets to attack with little concern about being bowled out.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "settled_partnership": {
            "scenario": "Bowling to a settled partnership",
            "family": "Opposition and matchups",
            "rule": "The current pair has added at least 30 runs and faced at least 24 legal balls. At least one original pressure index must also be high.",
            "pressure": "Two batters who have already built an innings together are harder to contain or separate.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "return_spell": {
            "scenario": "Returning for a new spell",
            "family": "Momentum and recovery",
            "rule": "First two deliveries on return after an 18-legal-ball gap, after the powerplay. At least one original pressure index must also be high.",
            "pressure": "The bowler has to regain rhythm immediately while the match is already under pressure.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "handedness_switch": {
            "scenario": "Adjusting to a left/right switch",
            "family": "Opposition and matchups",
            "rule": "The striker's recorded hand differs from the previous ball in the same bowler's over. At least one original pressure index must also be high.",
            "pressure": "The bowler must adjust line and field angles mid-over while already under pressure.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "after_illegal_delivery": {
            "scenario": "Recovering after a wide or no-ball",
            "family": "Momentum and recovery",
            "rule": "The same bowler's immediately previous attempt in the over was illegal. At least one original pressure index must also be high.",
            "pressure": "An extra run and replacement ball reduce the bowler's room for another mistake.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "after_own_wicket": {
            "scenario": "Building on your own wicket in the same over",
            "family": "Momentum and recovery",
            "rule": "A wicket was credited to this bowler one or two delivery attempts earlier in this over. At least one original pressure index must also be high.",
            "pressure": "The bowler has a short chance to capitalize on a wicket while the innings remains under pressure.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "finish_tight_over": {
            "scenario": "Closing out a tight over",
            "family": "Momentum and recovery",
            "rule": "The over has used four or five legal balls and the bowler has conceded at most six runs in it. At least one original pressure index must also be high.",
            "pressure": "A late boundary can undo a valuable low-cost over in an already pressured spell.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "final_quota": {
            "scenario": "Bowling the final quota over",
            "family": "Wickets and support",
            "rule": "The bowler has already delivered at least 18 legal balls in this innings. At least one original pressure index must also be high.",
            "pressure": "The bowler is using the last part of their usual four-over allocation under pressure.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "start_after_expensive_over": {
            "scenario": "Taking over after a costly wicketless over",
            "family": "Momentum and recovery",
            "rule": "First two legal-ball positions of the over after another bowler's over cost the team at least 10 runs with no wicket. At least one original pressure index must also be high.",
            "pressure": "The new bowler inherits an innings that has just accelerated without losing a wicket.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "specialist",
            "prefix": ""
        },
        "nineteenth_over": {
            "scenario": "Bowling the 19th over",
            "family": "Close finishes",
            "rule": "Over 19 with 4+ batting wickets left and a first-innings score within 5 runs of trajectory or ahead, a chase at 7–14 RPO or 25–75% earlier-state win odds, or a knockout.",
            "pressure": "The penultimate over leaves only six more legal balls to repair a costly spell.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "refined_specialist",
            "prefix": ""
        },
        "raise_chase_requirement": {
            "scenario": "Raising the required rate from a manageable chase",
            "family": "Match demand",
            "rule": "Second innings, 12 or more legal balls remaining, and a pre-ball required rate from six to ten runs per over. At least one original pressure index must also be high.",
            "pressure": "If the chase is manageable, beating the required pace can force the batters into higher-risk scoring.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "specialist",
            "prefix": "rate_"
        },
        "defending_7–12_at_the_start_of_the_last_over": {
            "scenario": "Defending 7–12 at the start of the last over",
            "family": "Close finishes",
            "rule": "Chase opened the final over needing 7–12 off six. Evaluate each subsequent attempt in that over.",
            "pressure": "The over is defendable, but one boundary or extra can give away the margin.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "defending_six_or_fewer_in_the_last_over": {
            "scenario": "Defending six or fewer in the last over",
            "family": "Close finishes",
            "rule": "Chase opened the final over needing 1–6 off six. Evaluate subsequent attempts in that over.",
            "pressure": "The opposition can win through routine scoring, so a wicket or sequence of dots is especially valuable.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "last_legal_ball_with_the_match_within_reach": {
            "scenario": "Last legal ball with the match within reach",
            "family": "Close finishes",
            "rule": "One legal ball remains and the chasing team needs 1–6 runs.",
            "pressure": "The bowler has one ball to manage a live result, and an extra may concede the match or another attempt.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "defending_a_narrow_cushion_in_overs_17–18": {
            "scenario": "Defending a narrow cushion in overs 17–18",
            "family": "Close finishes",
            "rule": "First two death overs and chase odds are 35–65% based on earlier match states.",
            "pressure": "The bowler can shift a balanced chase before the designated final two overs.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "nineteenth_over_to_a_set_striker_in_a_balanced_chase": {
            "scenario": "Nineteenth over to a set striker in a balanced chase",
            "family": "Close finishes",
            "rule": "Over 19, striker has faced 16+ non-wides and chase win odds are 30–70% based on earlier states.",
            "pressure": "An established batter is in position to take control of the penultimate over.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "only_one_wicket_needed_but_the_chase_is_live": {
            "scenario": "Only one wicket needed but the chase is live",
            "family": "Close finishes",
            "rule": "Exactly one batting wicket left, 7–24 legal balls remaining and 4–20 runs needed.",
            "pressure": "Taking the wicket can finish the match, but ordinary scoring can still beat the defense.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "small_target_and_no_early_defensive_cushion": {
            "scenario": "Small target and no early defensive cushion",
            "family": "Match demand",
            "rule": "Powerplay of a chase of 145 or fewer with required rate at most 7.5 RPO.",
            "pressure": "A low target leaves little room to concede early boundaries before the chase becomes comfortable.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "chase_shifting_decisively_toward_the_batters": {
            "scenario": "Chase shifting decisively toward the batters",
            "family": "Match demand",
            "rule": "Post-powerplay with 18+ legal balls left, earlier-state chase odds rise from at most 50% 12 balls ago to at least 65%.",
            "pressure": "The defense needs to reverse momentum before the batting side can cruise.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "below_seven_required_with_batting_depth_intact": {
            "scenario": "Below seven required with batting depth intact",
            "family": "Match demand",
            "rule": "Overs 7–16, 5 to under 7 RPO required and at least six batting wickets available.",
            "pressure": "Batters can play low-risk cricket, making sustained dots or wickets necessary to change the chase.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "required_rate_falls_sharply_over_two_overs": {
            "scenario": "Required rate falls sharply over two overs",
            "family": "Match demand",
            "rule": "After the powerplay with 18+ balls left, chase required rate fell by at least 2 RPO in 12 balls and is now 10 or less.",
            "pressure": "The previous defensive plan has failed to hold the scoring pace.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "first_innings_accelerating_above_trajectory_in_the_middle": {
            "scenario": "First innings accelerating above trajectory in the middle",
            "family": "Match demand",
            "rule": "Overs 7–16 of the first innings with a score at least 15 runs above the earlier same-context trajectory.",
            "pressure": "A substantial scoring overshoot risks setting an unusually difficult target if left unchecked.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "explosive_opposition_powerplay": {
            "scenario": "Explosive opposition powerplay",
            "family": "Match demand",
            "rule": "From the start of over 5 to the end of over 6, batting team scoring at 10+ RPO with at most one wicket lost.",
            "pressure": "Batters in control can keep attacking because the fielding side has yet to take enough wickets.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "few_credible_bowling_alternatives_left": {
            "scenario": "Few credible bowling alternatives left",
            "family": "Wickets and support",
            "rule": "12+ legal balls remain, at most two quota-eligible alternatives, alternate top-three ability at most 50/100, and 60%+ known-history coverage, with a high original pressure index.",
            "pressure": "The captain has limited credible replacements, increasing the current bowler's responsibility.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "first_innings_wicket_drought_while_runs_build": {
            "scenario": "First innings wicket drought while runs build",
            "family": "Wickets and support",
            "rule": "From over 9 to 16, no team wicket in the previous 36 legal balls, 7+ batting wickets left and scoring at 8+ RPO.",
            "pressure": "The fielding side needs a breakthrough before the set batters can exploit the death overs.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "protect_the_hard_won_wicket_against_a_strong_replacement": {
            "scenario": "Protect the hard-won wicket against a strong replacement",
            "family": "Wickets and support",
            "rule": "Within two legal balls after a team wicket, striker has faced at most two non-wides, is rated 6.5+/10, and a high original pressure index is active.",
            "pressure": "The new batter is historically strong, but the defense has a brief chance before they settle.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "final_quota_under_an_urgent_chase_defense": {
            "scenario": "Final quota under an urgent chase defense",
            "family": "Wickets and support",
            "rule": "Bowler has used 18+ legal balls, 7–24 remain in the match, and the chase needs 7–12 RPO.",
            "pressure": "A primary bowler has few deliveries left to protect a defendable target.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "own_spell_after_an_expensive_start": {
            "scenario": "Own spell after an expensive start",
            "family": "Momentum and recovery",
            "rule": "Bowler has completed 6–12 legal balls, has conceded at least 12 runs per six legal balls so far and taken no credited wicket.",
            "pressure": "Prior overs have consumed the defense's margin and the bowler has to recover within the remaining quota.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "following_your_own_conceded_boundary_in_a_live_chase": {
            "scenario": "Following your own conceded boundary in a live chase",
            "family": "Momentum and recovery",
            "rule": "Immediately following a boundary on the same bowler's previous attempt in the current over, with at most 24 balls left and chase odds at 30–75%.",
            "pressure": "The boundary made an already competitive chase more fragile. The response ball must restore control.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "taking_over_after_a_multi_over_surge": {
            "scenario": "Taking over after a multi-over surge",
            "family": "Momentum and recovery",
            "rule": "First two legal-ball positions of an over, after 24 legal balls; prior 12 legal balls cost 24+ runs with no wicket.",
            "pressure": "A prolonged team scoring surge calls for immediate control from the incoming bowler.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "protecting_a_low_cost_over_against_a_strong_finisher": {
            "scenario": "Protecting a low-cost over against a strong finisher",
            "family": "Momentum and recovery",
            "rule": "At least four legal balls into a death over, the bowler has conceded at most 4 in the current over, and striker is 6.5+ quality with at least 1.5 live runs per non-wide across 12+ faced.",
            "pressure": "One late boundary can erase the bowler's progress against a batter already hitting freely.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "bowling_after_a_runaway_boundary_over_starts": {
            "scenario": "Bowling after a runaway boundary over starts",
            "family": "Momentum and recovery",
            "rule": "First six overs after 6+ legal balls, among the preceding six legal balls the team allowed at least three boundaries and 15+ runs.",
            "pressure": "The matchup needs a rapid plan change under powerplay fielding constraints.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "bowling_to_a_proven,_in_form_hitter_already_scoring_freely": {
            "scenario": "Bowling to a proven, in-form hitter already scoring freely",
            "family": "Opposition and matchups",
            "rule": "Striker rated 6.5+/10 with recent form gap of at least +0.15, and 12+ non-wides at 1.5 or more live runs per non-wide.",
            "pressure": "The opponent combines long-run ability, recent form and live control of the innings.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "defending_late_against_a_dangerous_left_hander": {
            "scenario": "Defending late against a dangerous left-hander",
            "family": "Opposition and matchups",
            "rule": "Recorded left-handed striker, 6.5+ ability or 12+ non-wides at 1.5+ live runs per attempt, overs 17–20, with 7–14 RPO needed.",
            "pressure": "A capable left-hander changes the defensive angles in a chase where each delivery matters.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "defending_late_against_a_dangerous_right_hander": {
            "scenario": "Defending late against a dangerous right-hander",
            "family": "Opposition and matchups",
            "rule": "Recorded right-handed striker, 6.5+ ability or 12+ non-wides at 1.5+ live runs per attempt, overs 17–20, with 7–14 RPO needed.",
            "pressure": "This is the complementary handedness matchup at the same defensive stakes, for measuring versatility.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "bowling_to_an_opponent_who_has_dominated_you": {
            "scenario": "Bowling to an opponent who has dominated you",
            "family": "Opposition and matchups",
            "rule": "At least 18 earlier head-to-head attempts, striker has scored at least 1.65 runs per attempt, and an original bowler pressure index is high.",
            "pressure": "The batter has an evidence-backed advantage against this specific bowler.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "unfavorable_handedness_against_a_quality_batter": {
            "scenario": "Unfavorable handedness against a quality batter",
            "family": "Opposition and matchups",
            "rule": "At least 30 prior attempts against that batting hand, runs conceded per attempt at least 15% worse than own long-run rate, and striker rated 6.5+/10.",
            "pressure": "A proven opponent can exploit a measured weakness in the bowler's usual matchup.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "bowling_to_a_two_sided_batting_threat": {
            "scenario": "Bowling to a two-sided batting threat",
            "family": "Opposition and matchups",
            "rule": "The pair contains both hands and each batter has 6+/10 prior ability. Current pair has faced 18+ balls and the bowler faces a high original pressure index.",
            "pressure": "The fielding side faces two proven threats with different scoring angles and little easy matchup relief.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "high_range_shotmaker_attacking_a_defendable_chase": {
            "scenario": "High-range shotmaker attacking a defendable chase",
            "family": "Opposition and matchups",
            "rule": "Middle or death overs, chase at 7–12 RPO, striker has 4+ effective historical shot categories with 40%+ historical labels and 6.5+ ability.",
            "pressure": "A proven batter with a wide recorded shot range can exploit a single-line defensive plan.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "bowling_slump_while_the_defense_is_still_in_play": {
            "scenario": "Bowling slump while the defense is still in play",
            "family": "Personal form and stakes",
            "rule": "Pre-match bowling form-slump indicator, 3+ recent matches, and a post-powerplay chase at 7–12 RPO.",
            "pressure": "The bowler must contain a plausible chase despite persistent recent underperformance.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "knockout_powerplay_getting_away_from_the_defense": {
            "scenario": "Knockout powerplay getting away from the defense",
            "family": "Personal form and stakes",
            "rule": "Recorded knockout, 12–35 legal balls used, batting rate at least 9 RPO and at most one wicket lost.",
            "pressure": "An elimination-game powerplay is getting away without a breakthrough.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "knockout_chase_alive_in_the_death_overs": {
            "scenario": "Knockout chase alive in the death overs",
            "family": "Personal form and stakes",
            "rule": "Recorded knockout, overs 17–20 and chase odds between 20% and 80% based on earlier states.",
            "pressure": "Each remaining delivery can decide an elimination match.",
            "measure": "Team runs saved against required pace and credited wickets. Track change in chase required rate.",
            "source": "shared_family",
            "prefix": "chase_"
        },
        "knockout_first_innings_with_a_strong_finish_looming": {
            "scenario": "Knockout first innings with a strong finish looming",
            "family": "Personal form and stakes",
            "rule": "Recorded knockout, first-innings death overs with 5+ batting wickets remaining and a score within 5 runs of trajectory or ahead.",
            "pressure": "The batting side has enough resources to accelerate into a potentially difficult elimination-match target.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        },
        "knockout_pressure_against_an_in_form_quality_striker": {
            "scenario": "Knockout pressure against an in-form quality striker",
            "family": "Personal form and stakes",
            "rule": "Recorded knockout, striker rated at least 6.5/10 with a form gap of at least +0.15, and an original match-demand or close-finish index high.",
            "pressure": "Opponent quality matters more because the match state and elimination stakes are already adverse.",
            "measure": "Runs conceded incl. extras and credited wickets versus comparable pre-ball expectation.",
            "source": "shared_family",
            "prefix": ""
        }
    }
}

TARGET_PREFIX = {
    key: scenario["prefix"]
    for catalog in SCENARIO_CATALOG.values()
    for key, scenario in catalog.items()
}


def make_legacy_bands(game, warmup_mask):
    original_bands = {}
    derived_bands = {}
    thresholds = []
    for role in ROLE_IDS:
        available = role_available(game, role)
        original, cutoffs = make_pressure_bands(game, role, warmup_mask & available)
        high = original.eq("pressure").any(axis=1)
        ordinary = ~high & original.eq("ordinary").any(axis=1)
        columns = {}
        for key in ADDED_SCENARIOS[role]:
            triggered = game[f"x_scenario_{key}"]
            pressure = triggered if key in DIRECT_SCENARIOS else triggered & high
            control = ordinary & (triggered if key == RATE_SCENARIO else ~triggered)
            assert not (pressure & control).any(), (role, key)
            columns[key] = np.select(
                [pressure, control], ["pressure", "ordinary"], default="middle",
            )
        derived_bands[role] = pd.concat(
            [original, pd.DataFrame(columns, index=game.index)], axis=1,
        )
        original_bands[role] = original
        thresholds.extend(cutoffs)
    return original_bands, derived_bands, thresholds


def build_expanded_triggers(game, original_bands):
    assert game.index.is_unique
    d = game.sort_values(["date", "match_id", "innings", "innings_ball_num"])
    primary = PRIMARY_SCENARIOS
    legacy = {
        (role, key): original_bands[role][key].reindex(d.index).eq("pressure")
        for role in ROLE_IDS
        for key in primary
    }
    def x(field):
        return d["x_" + field]

    used = x("legal_balls_used")
    left = x("balls_remaining")
    wr = x("wickets_remaining")
    lost = x("wickets_lost_before")
    rrr = x("required_run_rate")
    need = x("runs_required")
    prob = x("historical_chase_success_probability")
    score = x("team_runs_before")
    striker = x("striker_nonwide_deliveries_before")
    partner = x("non_striker_nonwide_deliveries_before")
    since = x("legal_balls_since_wicket")
    paired = x("partnership_legal_balls_before")
    chase = d.innings.eq(2) & rrr.gt(0) & need.gt(0)
    first = d.innings.eq(1)
    death = used.ge(96)
    middle = used.between(36, 95)
    power = used.lt(36)
    local_gap = x("score_minus_trajectory_proxy").where(x("trajectory_reference_matches").ge(12))
    known_prob = prob.where(x("historical_chase_reference_matches").ge(12))
    recent_runs = x("runs_previous_6_legal_ball_window").where(used.ge(6))
    recent_boundaries = x("boundaries_previous_6_legal_ball_window").where(used.ge(6))
    batter_good = x("batter_base_ability10").ge(6.5)
    bowler_good = x("bowler_base_ability10").ge(6.2)
    bowled = x("bowler_legal_balls_before")
    knockout = x("knockout").eq(1)
    dots = x("consecutive_team_dot_events")
    position = used.mod(6)
    style = x("bowling_style")
    pace = x("pace_or_spin").eq("Pace")
    same_innings = ["match_id", "innings"]
    recent_wickets = x("wickets_previous_12_legal_ball_window").where(used.ge(12))
    known_bat_support = x("remaining_strength_players_known").ge(2)
    bat_support = x("remaining_top3_strength_score100_proxy")
    bowl_support = x("other_bowling_top3_strength_proxy")
    bowl_options = x("other_bowling_options_count_proxy")
    bowling_coverage = x("other_bowling_history_coverage").ge(0.60)
    hot_batter = striker.ge(12) & x("striker_runs_before").div(striker).ge(1.5)
    bowl_threat = x("batter_opponent_live_threat")
    bat_pressure = pd.concat([legacy["batter", key] for key in primary], axis=1).any(axis=1)
    bowl_pressure = pd.concat([legacy["bowler", key] for key in primary], axis=1).any(axis=1)

    state_cols = ["x_team_runs_before", "x_required_run_rate", "x_historical_chase_success_probability", "x_historical_chase_reference_matches"]
    state = (
        d[same_innings + ["x_legal_balls_used", *state_cols]]
        .drop_duplicates(same_innings + ["x_legal_balls_used"])
        .set_index(same_innings + ["x_legal_balls_used"])
    )
    def exact_prior_state(balls):
        keys = pd.MultiIndex.from_arrays([d.match_id, d.innings, used - balls])
        return state.reindex(keys).set_axis(d.index)

    prior = exact_prior_state(12)
    rrr_rise = rrr - prior.x_required_run_rate.where(prior.x_required_run_rate.gt(0))
    prior_prob = prior.x_historical_chase_success_probability.where(
        prior.x_historical_chase_reference_matches.ge(12)
    )
    runs12 = score - prior.x_team_runs_before
    delivery_groups = d.groupby(same_innings, sort=False)
    hand_key = ["match_id", "innings", "striker_id"]
    observed_hand = d[hand_key + ["x_batting_hand"]].copy()
    observed_hand["known_hand"] = x("batting_hand").where(x("batting_hand").isin(["RHB", "LHB"]))
    observed_hand["known_hand"] = observed_hand.groupby(hand_key).known_hand.ffill()
    striker_hand_history = observed_hand.known_hand
    last_hand = {}
    partner_hand = []
    for frame_row, hand in zip(d[same_innings + ["striker_id", "non_striker_id"]].itertuples(index=False, name=None), striker_hand_history):
        match, innings, striker_id, partner_id = frame_row
        partner_hand.append(last_hand.get((match, innings, partner_id)))
        if pd.notna(hand):
            last_hand[(match, innings, striker_id)] = hand
    partner_batting_hand = pd.Series(partner_hand, index=d.index)
    previous_wicket = delivery_groups.y_team_wicket.shift().eq(1)
    previous_non_striker = delivery_groups.non_striker_id.shift()
    previous_bowler = delivery_groups.bowler_id.shift()
    previous_ball_boundary = delivery_groups.y_boundary.shift().eq(1)
    own_over_runs = d.groupby(
        [d.match_id, d.innings, used.floordiv(6), d.bowler_id], sort=False
    ).y_bowler_runs.cumsum() - d.y_bowler_runs
    previous_pair_wickets = x("bowler_wickets_before")
    previous_spell_runs = x("bowler_runs_before")
    last_over = left.le(6)
    first_two_of_over = position.le(1)
    masks = {role: {} for role in ROLE_IDS}
    masks['batter']['steep_chase_with_two_to_four_overs_left'] = (
        chase & left.between(12, 24) & need.ge(2 * left)
    )
    masks['batter']['reachable_last_over_chase'] = (
        chase & last_over & exact_prior_state(position).x_required_run_rate.between(7, 12)
    )
    masks['batter']['last_over_starts_needing_13_or_more'] = (
        chase & last_over & exact_prior_state(position).x_required_run_rate.ge(13)
    )
    masks['batter']['boundary_needed_off_the_last_ball'] = (
        chase & left.eq(1) & need.between(4, 6)
    )
    masks['batter']['low_runs_needed_with_the_last_wicket'] = (
        chase & need.between(1, 6) & left.le(12) & wr.eq(1)
    )
    masks['batter']['large_chase_at_the_opening'] = (
        chase & used.lt(12) & x("target").ge(190)
    )
    masks['batter']['comfortable_chase_turning_against_the_batting_side'] = (
        chase & used.ge(36) & left.ge(18) & prior_prob.ge(.65) & known_prob.le(.5)
    )
    masks['batter']['ahead_of_the_rate_but_running_out_of_wickets'] = (
        chase & used.ge(36) & wr.between(2, 3) & x("current_run_rate").sub(rrr).ge(1)
    )
    masks['batter']['steep_required_rate_jump_in_the_middle_overs'] = (
        chase & middle & rrr.ge(10) & rrr_rise.ge(2.5)
    )
    masks['batter']['first_innings_well_below_its_scoring_trajectory'] = (
        first & used.between(54, 95) & local_gap.le(-15)
    )
    masks['batter']['death_overs_with_wickets_in_hand_but_a_scoring_deficit'] = (
        first & death & wr.ge(6) & local_gap.le(-12)
    )
    masks['batter']['new_batter_walking_into_a_middle_overs_collapse'] = (
        middle & striker.le(3) & recent_wickets.ge(2)
    )
    masks['batter']['established_batter_loses_the_other_established_batter'] = (
        previous_wicket & previous_non_striker.notna() & previous_non_striker.ne(d.non_striker_id)
            & previous_non_striker.ne(d.striker_id) & striker.ge(16)
            & delivery_groups.x_partnership_legal_balls_before.shift().ge(24)
    )
    masks['batter']['chasing_with_the_tail_and_no_deeper_support'] = (
        chase & used.ge(36) & rrr.ge(8) & wr.between(2, 4) & known_bat_support & bat_support.le(45)
    )
    masks['batter']['last_batting_pair_still_needs_a_meaningful_chase'] = (
        chase & wr.eq(1) & need.ge(7) & left.between(6, 30)
    )
    masks['batter']['double_strike_before_the_innings_settles'] = (
        used.lt(18) & lost.ge(2)
    )
    masks['batter']['new_finisher_exposed_late_in_a_demanding_chase'] = (
        chase & used.ge(108) & striker.le(2) & rrr.ge(9)
    )
    masks['batter']['team_scoring_squeeze_after_the_powerplay'] = (
        middle & runs12.le(10) & ((first & local_gap.le(-8)) | (chase & rrr.ge(8)))
    )
    masks['batter']['dots_at_the_end_of_a_tight_chase'] = (
        chase & dots.ge(3) & left.between(6, 24) & rrr.ge(7)
    )
    masks['batter']['two_over_drought_after_an_early_wicket'] = (
        power & lost.ge(1) & runs12.le(10)
    )
    masks['batter']['established_batter_stuck_below_a_run_a_ball'] = (
        striker.ge(18) & x("striker_runs_before").lt(striker)
            & ((chase & rrr.ge(9)) | (first & local_gap.le(-8)))
    )
    masks['batter']['high_ability_pace_in_a_demanding_innings'] = (
        bat_pressure & pace & bowler_good
    )
    masks['batter']['high_ability_off_spin_in_a_demanding_innings'] = (
        bat_pressure & style.eq("right-arm offbreak") & bowler_good
    )
    masks['batter']['high_ability_wrist_spin_in_a_demanding_innings'] = (
        bat_pressure & style.isin(["right-arm leg-break", "left-arm chinaman"]) & bowler_good
    )
    masks['batter']['high_ability_left_arm_orthodox_under_pressure'] = (
        bat_pressure & style.eq("slow left-arm orthodox") & bowler_good
    )
    masks['batter']['high_velocity_pace_after_an_early_wicket'] = (
        power & lost.ge(1) & pace & x("bowler_speed_observations_prior").ge(20)
            & x("bowler_mean_speed_prior").ge(138)
    )
    masks['batter']['proven_bowler_also_dangerous_in_this_spell'] = (
        bowler_good & x("batter_opponent_form").ge(.15) & bowl_threat.ge(.65)
    )
    masks['batter']['facing_a_bowler_who_has_repeatedly_dismissed_you'] = (
        bat_pressure & x("pair_prior_events").ge(18) & x("pair_prior_striker_dismissal_rate").ge(.10)
    )
    masks['batter']['facing_a_style_that_has_suppressed_your_scoring'] = (
        x("batter_style_prior_events").ge(30)
            & x("batter_style_prior_runs_per_event").le(.8 * x("batter_prior_runs_per_event"))
            & ((chase & rrr.ge(8)) | (first & local_gap.le(-8)))
    )
    masks['batter']['variable_repertoire_bowler_in_an_urgent_chase'] = (
        chase & used.ge(36) & rrr.ge(9) & x("bowler_effective_repertoire_categories_proxy").ge(4)
            & x("bowler_prior_repertoire_label_coverage").ge(.4)
    )
    masks['batter']['personal_batting_slump_with_a_demanding_chase'] = (
        chase & used.ge(36) & rrr.ge(9) & x("batter_form_slump_candidate").eq(1)
            & x("batter_last5_matches_available").ge(3)
    )
    masks['batter']['scoring_burden_in_the_knockout_powerplay'] = (
        knockout & power & (lost.ge(1) | (chase & rrr.ge(9)))
    )
    masks['batter']['knockout_chase_at_the_death'] = (
        knockout & chase & death & known_prob.between(.2, .8)
    )
    masks['batter']['knockout_first_innings_recovery_after_a_collapse'] = (
        knockout & first & used.ge(36) & recent_wickets.ge(2)
    )
    masks['batter']['knockout_batter_facing_a_proven_bowler_in_form'] = (
        knockout & bowler_good & x("batter_opponent_form").ge(.15)
            & (legacy["batter", "match_demand"] | legacy["batter", "limited_support"])
    )
    final_open_rrr = exact_prior_state(position).x_required_run_rate
    masks['bowler']['defending_7–12_at_the_start_of_the_last_over'] = (
        chase & last_over & final_open_rrr.between(7, 12)
    )
    masks['bowler']['defending_six_or_fewer_in_the_last_over'] = (
        chase & last_over & final_open_rrr.between(1, 6)
    )
    masks['bowler']['last_legal_ball_with_the_match_within_reach'] = (
        chase & left.eq(1) & need.between(1, 6)
    )
    masks['bowler']['defending_a_narrow_cushion_in_overs_17–18'] = (
        chase & used.between(96, 107) & known_prob.between(.35, .65)
    )
    masks['bowler']['nineteenth_over_to_a_set_striker_in_a_balanced_chase'] = (
        chase & used.between(108, 113) & striker.ge(16) & known_prob.between(.30, .70)
    )
    masks['bowler']['only_one_wicket_needed_but_the_chase_is_live'] = (
        chase & wr.eq(1) & left.between(7, 24) & need.between(4, 20)
    )
    masks['bowler']['small_target_and_no_early_defensive_cushion'] = (
        chase & power & x("target").le(145) & rrr.le(7.5)
    )
    masks['bowler']['chase_shifting_decisively_toward_the_batters'] = (
        chase & used.ge(36) & left.ge(18) & prior_prob.le(.50) & known_prob.ge(.65)
    )
    masks['bowler']['below_seven_required_with_batting_depth_intact'] = (
        chase & middle & rrr.ge(5) & rrr.lt(7) & wr.ge(6)
    )
    masks['bowler']['required_rate_falls_sharply_over_two_overs'] = (
        chase & used.ge(36) & left.ge(18) & rrr_rise.le(-2) & rrr.le(10)
    )
    masks['bowler']['first_innings_accelerating_above_trajectory_in_the_middle'] = (
        first & middle & local_gap.ge(15)
    )
    masks['bowler']['explosive_opposition_powerplay'] = (
        used.between(24, 35) & x("current_run_rate").ge(10) & lost.le(1)
    )
    masks['bowler']['few_credible_bowling_alternatives_left'] = (
        bowl_pressure & left.ge(12) & bowl_options.le(2) & bowl_support.le(50) & bowling_coverage
    )
    masks['bowler']['first_innings_wicket_drought_while_runs_build'] = (
        first & used.between(48, 95) & since.ge(36) & wr.ge(7) & x("current_run_rate").ge(8)
    )
    masks['bowler']['protect_the_hard_won_wicket_against_a_strong_replacement'] = (
        bowl_pressure & since.le(2) & lost.ge(1) & striker.le(2) & batter_good
    )
    masks['bowler']['final_quota_under_an_urgent_chase_defense'] = (
        chase & bowled.ge(18) & left.between(7, 24) & rrr.between(7, 12)
    )
    masks['bowler']['own_spell_after_an_expensive_start'] = (
        bowled.between(6, 12) & previous_spell_runs.ge(2 * bowled) & previous_pair_wickets.eq(0)
    )
    masks['bowler']['following_your_own_conceded_boundary_in_a_live_chase'] = (
        chase & left.le(24) & position.gt(0) & previous_bowler.eq(d.bowler_id)
             & previous_ball_boundary & known_prob.between(.30, .75)
    )
    masks['bowler']['taking_over_after_a_multi_over_surge'] = (
        used.ge(24) & first_two_of_over & runs12.ge(24) & recent_wickets.eq(0)
    )
    masks['bowler']['protecting_a_low_cost_over_against_a_strong_finisher'] = (
        death & position.ge(4) & own_over_runs.le(4)
             & batter_good & hot_batter
    )
    masks['bowler']['bowling_after_a_runaway_boundary_over_starts'] = (
        power & recent_boundaries.ge(3) & recent_runs.ge(15)
    )
    masks['bowler']['bowling_to_a_proven,_in_form_hitter_already_scoring_freely'] = (
        batter_good & x("bowler_opponent_form").ge(.15) & hot_batter
    )
    masks['bowler']['defending_late_against_a_dangerous_left_hander'] = (
        chase & death & rrr.between(7, 14) & x("batting_hand").eq("LHB") & (batter_good | hot_batter)
    )
    masks['bowler']['defending_late_against_a_dangerous_right_hander'] = (
        chase & death & rrr.between(7, 14) & x("batting_hand").eq("RHB") & (batter_good | hot_batter)
    )
    masks['bowler']['bowling_to_an_opponent_who_has_dominated_you'] = (
        bowl_pressure & x("pair_prior_events").ge(18) & x("pair_prior_batter_runs_per_event").ge(1.65)
    )
    masks['bowler']['unfavorable_handedness_against_a_quality_batter'] = (
        batter_good & x("bowler_hand_prior_events").ge(30)
             & x("bowler_hand_prior_runs_per_event").ge(1.15 * x("bowler_prior_runs_per_event"))
    )
    masks['bowler']['bowling_to_a_two_sided_batting_threat'] = (
        bowl_pressure & paired.ge(18)
             & d["x_partner_strength_score100_prior"].ge(60) & x("batter_base_ability10").ge(6)
             & partner_batting_hand.notna() & x("batting_hand").ne(partner_batting_hand)
    )
    masks['bowler']['high_range_shotmaker_attacking_a_defendable_chase'] = (
        chase & used.ge(36) & rrr.between(7, 12) & batter_good
             & x("batter_effective_repertoire_categories_proxy").ge(4)
             & x("batter_prior_repertoire_label_coverage").ge(.40)
    )
    masks['bowler']['bowling_slump_while_the_defense_is_still_in_play'] = (
        chase & used.ge(36) & rrr.between(7, 12) & x("bowler_form_slump_candidate").eq(1)
             & x("bowler_last5_matches_available").ge(3)
    )
    masks['bowler']['knockout_powerplay_getting_away_from_the_defense'] = (
        knockout & used.between(12, 35) & x("current_run_rate").ge(9) & lost.le(1)
    )
    masks['bowler']['knockout_chase_alive_in_the_death_overs'] = (
        knockout & chase & death & known_prob.between(.20, .80)
    )
    masks['bowler']['knockout_first_innings_with_a_strong_finish_looming'] = (
        knockout & first & death & wr.ge(5) & local_gap.ge(-5)
    )
    masks['bowler']['knockout_pressure_against_an_in_form_quality_striker'] = (
        knockout & batter_good & x("bowler_opponent_form").ge(.15)
             & (legacy["bowler", "match_demand"] | legacy["bowler", "close_finish"])
    )
    masks["batter"]["death_over_acceleration"] = (
        (first & local_gap.le(-8))
        | (chase & (rrr.ge(9) | known_prob.between(0.25, 0.75)))
        | knockout
    )
    masks["bowler"]["nineteenth_over"] = (
        (first & local_gap.ge(-5) & wr.ge(4))
        | (chase & (known_prob.between(0.25, 0.75) | rrr.between(7, 14)))
        | knockout
    )
    return {
        role: {
            name: mask.reindex(game.index).fillna(False).astype(bool)
            for name, mask in entries.items()
        }
        for role, entries in masks.items()
    }


STATE_COLUMNS = [
    "row_id", "match_id", "innings", "innings_ball_num", "striker_id",
    "bowler_id", "x_wickets_lost_before", "x_legal_balls_used",
    "x_wickets_remaining", "x_striker_nonwide_deliveries_before",
    "x_non_striker_nonwide_deliveries_before", "x_partnership_runs_before",
    "x_partnership_legal_balls_before", "x_batting_hand", "x_pace_or_spin",
    "y_legal_delivery_candidate", "y_team_runs", "y_team_wicket",
    "y_bowler_runs", "y_bowler_wicket", "x_required_run_rate",
    "x_balls_remaining", "x_bowler_legal_balls_before", "date", "x_competition_group", "x_team_runs_before", "x_legal_balls_since_wicket",
]
EXISTING_LABELS = {
    "match_demand": "Match demand",
    "close_finish": "Close finish",
    "limited_support": "Collapse or limited support",
    "strong_opposition": "Strong opposition",
    "sustained_attack": "Sustained attack",
    "form_or_stakes": "Personal form or knockout stakes",
}
ADDED_SCENARIOS = {
    "batter": {
        "recovery_after_slow_powerplay": (
            "Recovering after a slow team powerplay",
            "Over 7 onward when the team ended six overs at least six runs below the mean of earlier powerplays from the same competition group and innings; at least eight earlier innings.",
        ),
        "death_over_acceleration": (
            "Accelerating in overs 17 to 20",
            "Any batter in overs 17 to 20; prior deliveries faced are not an eligibility condition.",
        ),
        "new_ball_after_early_wickets": (
            "Facing the new ball after early wickets",
            "First five overs after at least one wicket has fallen; multiple wickets also qualify.",
        ),
        "switch_pace_spin": (
            "Switching between pace and spin",
            "Bowling changes between pace and spin since the striker last faced.",
        ),
        "set_with_fresh_partner": (
            "Set batter with a fresh partner",
            "Striker has faced at least 20 non-wide balls and partner fewer than six.",
        ),
        "acceleration_after_dot_buildup": (
            "Scoring after a dot-heavy 15-ball stretch",
            "At least eight team dot balls among the striker's preceding 15 legal balls faced in the same innings. The following attempt is evaluated.",
        ),
        "rebuild_after_wicket": (
            "Established batter rebuilding after a wicket",
            "Overs 7 to 16, after at least 12 non-wide balls faced and with no more than three legal balls elapsed since a team wicket.",
        ),
        "rising_chase_rate": (
            "Batting when the required rate is rising",
            "After the powerplay with at least 18 legal balls left, the required rate "
            "has risen by at least one run per over from 12 legal balls earlier.",
        ),
        "fresh_bowler_post_powerplay": (
            "Facing a fresh bowler introduced after the powerplay",
            "The current bowler was introduced after 36 legal balls and has used fewer than six.",
        ),
    },
    "bowler": {
        "first_innings_death": (
            "First-innings death-over containment",
            "At least 96 legal balls used and the batting side has five or more wickets left.",
        ),
        "settled_partnership": (
            "Bowling to a settled partnership",
            "The current pair has added at least 30 runs and faced at least 24 legal balls.",
        ),
        "return_spell": (
            "Returning for a new spell",
            "First two deliveries on return after an 18-legal-ball gap, after the powerplay.",
        ),
        "handedness_switch": (
            "Adjusting to a left/right switch",
            "The striker's recorded hand differs from the previous ball in the same bowler's over.",
        ),
        "after_illegal_delivery": (
            "Recovering after a wide or no-ball",
            "The same bowler's immediately previous attempt in the over was illegal.",
        ),
        "after_own_wicket": (
            "Building on your own wicket in the same over",
            "A wicket was credited to this bowler one or two delivery attempts earlier in this over.",
        ),
        "finish_tight_over": (
            "Closing out a tight over",
            "The over has used four or five legal balls and the bowler has conceded at most six runs in it.",
        ),
        "final_quota": (
            "Bowling the final quota over",
            "The bowler has already delivered at least 18 legal balls in this innings.",
        ),
        "start_after_expensive_over": (
            "Taking over after a costly wicketless over",
            "First two legal-ball positions of the over after another bowler's over cost the team "
            "at least 10 runs with no wicket.",
        ),
        "nineteenth_over": (
            "Bowling the 19th over",
            "Every attempt in over 19 of either innings, from 108 to 113 legal balls completed, including wides and no-balls.",
        ),
        "raise_chase_requirement": (
            "Raising the required rate from a manageable chase",
            "Second innings, 12 or more legal balls remaining, and a pre-ball required rate from six to ten runs per over.",
        ),
    },
}
def build_triggers(state):
    sort_keys = ["match_id", "innings", "innings_ball_num"]
    state = state.sort_values(sort_keys)
    keys = ["match_id", "innings"]
    balls = state["x_legal_balls_used"]
    striker_balls = state["x_striker_nonwide_deliveries_before"]
    partner_balls = state["x_non_striker_nonwide_deliveries_before"]
    over = balls // 6
    same_over = state.groupby([*keys, over, "bowler_id"], sort=False)
    bowler = state.groupby(keys + ["bowler_id"], sort=False)
    batter = state.groupby(keys + ["striker_id"], sort=False)
    previous_hand = same_over["x_batting_hand"].shift()
    previous_style = batter["x_pace_or_spin"].shift()
    last_bowler_clock = bowler["x_legal_balls_used"].shift()
    last_attempt_legal = bowler["y_legal_delivery_candidate"].shift()
    returning = (balls - last_bowler_clock - last_attempt_legal).ge(18)
    second_attempt = returning.groupby(
        [state["match_id"], state["innings"], state["bowler_id"]], sort=False
    ).shift(fill_value=False)
    hand = state["x_batting_hand"]
    style = state["x_pace_or_spin"]
    past_rate = previous_required_rate(state, balls)
    earlier_over = previous_over_state(state, over)
    powerplay_gap = prior_powerplay_gap(state, balls)
    prior_dots = earlier_dot_count(state)
    over_position = balls % 6
    over_runs_before = same_over["y_bowler_runs"].cumsum() - state["y_bowler_runs"]
    return pd.DataFrame({
        "row_id": state["row_id"],
        "recovery_after_slow_powerplay": balls.ge(36) & powerplay_gap.le(-6),
        "death_over_acceleration": balls.between(96, 119),
        "new_ball_after_early_wickets": (
            balls.lt(30) & state["x_wickets_lost_before"].ge(1)
        ),
        "switch_pace_spin": (
            style.isin(["Pace", "Spin"])
            & previous_style.isin(["Pace", "Spin"]) & style.ne(previous_style)
        ),
        "set_with_fresh_partner": striker_balls.ge(20) & partner_balls.lt(6),
        "acceleration_after_dot_buildup": prior_dots.ge(8),
        "rebuild_after_wicket": (
            balls.between(36, 95) & striker_balls.ge(12)
            & state["x_wickets_lost_before"].ge(1)
            & state["x_legal_balls_since_wicket"].between(0, 3)
        ),
        "rising_chase_rate": (
            state["innings"].eq(2) & balls.ge(36)
            & state["x_balls_remaining"].ge(18)
            & (state["x_required_run_rate"] - past_rate).ge(1)
        ),
        "fresh_bowler_post_powerplay": (
            bowler["x_legal_balls_used"].transform("first").ge(36)
            & state["x_bowler_legal_balls_before"].lt(6)
        ),
        "first_innings_death": (
            state["innings"].eq(1) & balls.ge(96)
            & state["x_wickets_remaining"].ge(5)
        ),
        "settled_partnership": (
            state["x_partnership_runs_before"].ge(30)
            & state["x_partnership_legal_balls_before"].ge(24)
        ),
        "return_spell": (returning | second_attempt) & balls.ge(36),
        "handedness_switch": (
            hand.isin(["RHB", "LHB"])
            & previous_hand.isin(["RHB", "LHB"]) & hand.ne(previous_hand)
        ),
        "after_illegal_delivery": (
            same_over["y_legal_delivery_candidate"].shift().eq(0)
        ),
        "after_own_wicket": (
            same_over["y_bowler_wicket"].shift(1).eq(1)
            | same_over["y_bowler_wicket"].shift(2).eq(1)
        ),
        "finish_tight_over": over_position.ge(4) & over_runs_before.le(6),
        "final_quota": state["x_bowler_legal_balls_before"].between(18, 23),
        "start_after_expensive_over": (
            over_position.lt(2)
            & earlier_over["runs"].ge(10) & earlier_over["wickets"].eq(0)
            & earlier_over["bowler"].ne(state["bowler_id"])
        ),
        "nineteenth_over": balls.between(108, 113),
        "raise_chase_requirement": (
            state["innings"].eq(2) & state["x_balls_remaining"].ge(12)
            & state["x_required_run_rate"].between(6, 10)
        ),
    })


def previous_required_rate(state, balls):
    keys = ["match_id", "innings", "x_legal_balls_used"]
    by_clock = state.drop_duplicates(keys).set_index(keys)["x_required_run_rate"]
    lookup = pd.MultiIndex.from_arrays(
        [state["match_id"], state["innings"], balls - 12], names=keys
    )
    return pd.Series(by_clock.reindex(lookup).to_numpy(), index=state.index)


def previous_over_state(state, over):
    groups = state.groupby(["match_id", "innings", over], sort=False)
    totals = groups.agg(
        runs=("y_team_runs", "sum"), wickets=("y_team_wicket", "sum"),
        bowler=("bowler_id", "last"),
    )
    lookup = pd.MultiIndex.from_arrays(
        [state["match_id"], state["innings"], over - 1], names=totals.index.names
    )
    previous = totals.reindex(lookup)
    previous.index = state.index
    return previous




def prior_powerplay_gap(state, balls):
    keys = ["match_id", "innings"]
    context = ["x_competition_group", "innings"]
    start = state.loc[balls.eq(36), [
        "match_id", "innings", "date", "x_competition_group", "x_team_runs_before",
    ]].drop_duplicates(keys)
    daily = start.groupby(context + ["date"], as_index=False).agg(
        pp_sum=("x_team_runs_before", "sum"), pp_matches=("match_id", "size"),
    ).sort_values("date", kind="stable")
    numeric = ["pp_sum", "pp_matches"]
    previous = daily.groupby(context, sort=False)[numeric].cumsum() - daily[numeric]
    daily["reference"] = previous["pp_sum"].div(
        previous["pp_matches"].where(previous["pp_matches"].ge(8))
    )
    start = start.merge(daily[context + ["date", "reference"]], on=context + ["date"],
                        how="left", validate="many_to_one")
    gaps = (start["x_team_runs_before"] - start["reference"]).set_axis(
        pd.MultiIndex.from_frame(start[keys])
    )
    result = gaps.reindex(pd.MultiIndex.from_frame(state[keys]))
    return pd.Series(result.to_numpy(), index=state.index).where(balls.ge(36))


def earlier_dot_count(state):
    keys = ["match_id", "innings", "striker_id"]
    legal = state["y_legal_delivery_candidate"].astype(int)
    after = legal.groupby([state[key] for key in keys], sort=False).cumsum()
    prior = after - legal
    earlier = state.loc[legal.eq(1), keys].copy()
    earlier["faced"] = after.loc[legal.eq(1)]
    earlier["dot"] = state.loc[legal.eq(1), "y_team_runs"].eq(0).astype(int)
    earlier["dot_count"] = earlier.groupby(keys, sort=False)["dot"].transform(
        lambda sequence: sequence.rolling(15, min_periods=15).sum()
    )
    lookup = pd.MultiIndex.from_arrays([state[key] for key in keys] + [prior])
    counts = earlier.set_index(keys + ["faced"])["dot_count"].reindex(lookup)
    return pd.Series(counts.to_numpy(), index=state.index)


def chase_rate_outcomes(game):
    live = (
        game["innings"].eq(2)
        & game["x_balls_remaining"].gt(0)
        & game["x_runs_required"].gt(0)
    )
    valid = live & game["x_balls_remaining"].ge(2)
    before_balls = game["x_balls_remaining"]
    legal = game["y_legal_delivery_candidate"]
    before_need = game["x_runs_required"]
    after_need = (before_need - game["y_team_runs"]).clip(lower=0)
    after_balls = before_balls - legal
    before = (6 * before_need / before_balls).where(live)
    chasing_wickets_left = game["x_wickets_remaining"] - game["y_team_wicket"]
    still_defined = valid & (chasing_wickets_left.gt(0) | after_need.eq(0))
    after = (6 * after_need.div(after_balls.where(after_balls.gt(0)))).where(still_defined)
    required_per_ball = before_need.div(before_balls.where(live))
    equivalent = (legal * required_per_ball - game["y_team_runs"]).where(live)
    batting_surplus = (game["y_batter_runs"] - legal * required_per_ball).where(live)
    return pd.DataFrame({
        "required_rate_before": before,
        "required_rate_after": after,
        "required_rate_change": after - before,
        "bowling_runs_saved_vs_required_pace": equivalent,
        "batting_runs_above_required_pace": batting_surplus,
    })


def scenario_target_prefix(key):
    return TARGET_PREFIX[key]


def apply_scenario_target(frame, key):
    prefix = scenario_target_prefix(key)
    if not prefix:
        return frame
    result = frame.copy()
    for field in ("observed_utility", "context_expected", "context_residual"):
        result[field] = result[f"{prefix}{field}"]
    return result


def scenario_definitions(role):
    return [
        (key, entry["scenario"], entry["rule"], entry["source"])
        for key, entry in SCENARIO_CATALOG[role].items()
    ]


def role_available(game, role):
    if role == "batter":
        return game["y_nonwide_faced"].eq(1)
    return game["row_id"].notna()


def compile_game_scenarios(game):
    triggers = build_triggers(game[STATE_COLUMNS])
    assert triggers["row_id"].is_unique and len(triggers) == len(game)
    names = [c for c in triggers if c != "row_id"]
    game = game.merge(
        triggers.rename(columns={key: f"x_scenario_{key}" for key in names}),
        on="row_id", how="left", validate="one_to_one",
    )
    assert game["row_id"].is_unique
    blocks = chronological_blocks(game)
    stage = np.full(len(game), -1, dtype=int)
    for fold, (start, end) in enumerate(blocks):
        stage[game["date"].between(start, end)] = fold
    assert (stage[game["date"].ge(blocks[0][0])] >= 0).all()
    originals, legacy_bands, thresholds = make_legacy_bands(
        game, game["date"].lt(blocks[0][0]),
    )
    masks = build_expanded_triggers(game, originals)
    bands_by_role = {}
    definitions, records, audits = [], [], []
    rates = chase_rate_outcomes(game)
    rate = rates["bowling_runs_saved_vs_required_pace"]
    chase_utility = rate + WICKET_VALUE * game["y_bowler_wicket"]
    for role in ROLE_IDS:
        original = originals[role]
        legacy = legacy_bands[role]
        available = role_available(game, role)
        original_high = original.eq("pressure").any(axis=1)
        ordinary_union = ~original_high & original.eq("ordinary").any(axis=1)
        labels = game[
            ["row_id", "date", "match_id", "innings", "x_phase",
             "x_competition_group", ROLE_IDS[role]]
        ].rename(columns={ROLE_IDS[role]: "player_id"})
        target = (
            game["y_batter_runs"] - WICKET_VALUE * game["y_striker_dismissed"]
            if role == "batter" else
            WICKET_VALUE * game["y_bowler_wicket"] - game["y_bowler_runs"]
        )
        targets = {"": target, "pace_": game["y_batter_runs"], "rate_": rate, "chase_": chase_utility}
        labels = labels.assign(role=role, fold=stage)
        band_columns = {}
        for number, (key, specification) in enumerate(SCENARIO_CATALOG[role].items(), 1):
            specialized_target = targets[specification["prefix"]]
            if specification["source"] == "shared_family":
                high = masks[role][key]
                ordinary = ordinary_union & ~high
                if specification["prefix"] == "chase_":
                    ordinary &= chase_utility.notna()
                if specification["prefix"] == "pace_":
                    ordinary &= rates["batting_runs_above_required_pace"].notna()
                states = ["pressure", "ordinary", "middle"]
                band = pd.Categorical.from_codes(
                    np.select([high, ordinary], [0, 1], default=2), categories=states,
                )
            else:
                band = pd.Categorical(legacy[key])
                if specification["source"] == "refined_specialist":
                    drop_pressure = legacy[key].eq("pressure") & ~masks[role][key]
                    band[drop_pressure.to_numpy()] = "middle"
            band = pd.Series(band, index=game.index)
            band_columns[key] = band
            definitions.append({
                "role": role, "scenario_number": number, "scenario_id": key,
                "scenario": specification["scenario"], "source": specification["source"],
                "family": specification["family"], "pressure_rule": specification["rule"],
                "pressure_mechanism": specification["pressure"], "target": specification["measure"],
            })
            measurable = available & band.isin(["pressure", "ordinary"])
            assert specialized_target.loc[measurable].notna().all(), (role, key)
            part = labels.loc[measurable].assign(
                scenario_id=key, band=band.loc[measurable].astype(str),
                observed_utility=specialized_target.loc[measurable],
            )
            for name in ("role", "scenario_id", "band", "x_phase", "x_competition_group"):
                part[name] = part[name].astype("category")
            records.append(part)
            counts = part.groupby(["fold", "band"], observed=True).agg(
                rows=("row_id", "size"), matches=("match_id", "nunique"),
                players=("player_id", "nunique"),
            ).reindex(
                pd.MultiIndex.from_product(
                    [range(-1, len(blocks)), ["pressure", "ordinary"]], names=["fold", "band"],
                ), fill_value=0,
            ).reset_index().assign(role=role, scenario_id=key)
            audits.append(counts)
        bands_by_role[role] = pd.DataFrame(band_columns, index=game.index)
    membership = pd.concat(records, ignore_index=True)
    assert not membership.duplicated(["role", "scenario_id", "row_id"]).any()
    definitions = pd.DataFrame(definitions)
    assert definitions.groupby("role").size().to_dict() == ROLE_SCENARIO_COUNTS
    write_parquet(OUTPUT_DIR / "scenario_membership.parquet", membership)
    write_csv(OUTPUT_DIR / "scenario_data_counts.csv", pd.concat(audits, ignore_index=True))
    write_csv(OUTPUT_DIR / "scenario_definitions.csv", definitions)
    write_csv(OUTPUT_DIR / "original_pressure_thresholds.csv", pd.DataFrame(thresholds))
    return game, bands_by_role, blocks

In [ ]:
def chronological_blocks(game):
    dates = np.sort(game["date"].unique())
    starts = [int(len(dates) * value) for value in FOLD_FRACTIONS]
    assert 0 < starts[0] and len(set(starts)) == len(starts) >= 3
    assert starts[-1] < len(dates)
    starts.append(len(dates))
    blocks = [
        (pd.Timestamp(dates[a]), pd.Timestamp(dates[b - 1]))
        for a, b in zip(starts[:-1], starts[1:])
    ]
    assert all(a[1] < b[0] for a, b in zip(blocks[:-1], blocks[1:]))
    return blocks


def make_pressure_bands(game, role, warmup):
    frame = pd.DataFrame(index=game.index)
    cutoffs = []
    groups = ["innings", "x_phase", "x_competition_group"]
    keys = pd.MultiIndex.from_frame(game[groups])
    for scenario in PRIMARY_SCENARIOS:
        field = pressure_column(role, scenario)
        past = game.loc[warmup, groups + [field]].dropna()
        assert len(past) >= 100, f"Too little earlier data for {role}/{scenario}"
        high, low = past[field].quantile(list(PERCENTILES)).to_list()
        grouped = past.groupby(groups)[field]
        thresholds = grouped.quantile(list(PERCENTILES)).unstack()
        thresholds.loc[grouped.size().lt(40), :] = np.nan
        mapped = thresholds.reindex(keys)
        highs = mapped[PERCENTILES[0]].fillna(high).to_numpy()
        lows = mapped[PERCENTILES[1]].fillna(low).to_numpy()
        paired = highs > lows
        highs = np.where(paired, highs, np.maximum(lows + 1e-4, high))
        values = game[field]
        frame[scenario] = np.select(
            [values.isna(), values.ge(highs), values.le(lows)],
            ["unknown", "pressure", "ordinary"], default="middle",
        )
        cutoffs.append({
            "role": role, "scenario": scenario, "fallback_high": high,
            "fallback_ordinary": low, "warmup_rows": len(past),
        })
        for key, row in thresholds.dropna().iterrows():
            cutoffs.append({
                "role": role, "scenario": scenario, "innings": key[0],
                "phase": key[1], "competition": key[2],
                "high": row[PERCENTILES[0]], "ordinary": row[PERCENTILES[1]],
                "warmup_rows": int(grouped.size().loc[key]),
            })
    assert frame.notna().all().all()
    return frame, cutoffs

In [ ]:
def libraries_and_devices():
    import catboost
    from catboost.utils import get_gpu_device_count
    import lightgbm
    import xgboost

    assert GPU_MODE in ("auto", "cpu")
    has_cuda = GPU_MODE == "auto" and get_gpu_device_count() > 0
    settings = {
        "catboost": (catboost, "GPU" if has_cuda else "CPU"),
        "lightgbm": (lightgbm, "CPU"),
        "xgboost": (
            xgboost, "GPU" if has_cuda and xgboost.build_info().get("USE_CUDA") else "CPU",
        ),
    }
    return settings


def encoded_fold(train, test, cats):
    from sklearn.preprocessing import OneHotEncoder

    numeric = [c for c in train if c not in cats]
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
    train_encoded = encoder.fit_transform(train[cats])
    test_encoded = encoder.transform(test[cats])
    return (
        pd.DataFrame(np.column_stack([train[numeric].to_numpy(dtype=np.float32), train_encoded])),
        pd.DataFrame(np.column_stack([test[numeric].to_numpy(dtype=np.float32), test_encoded])),
    )


def benchmark_estimator(name, lib, device, classifier, cats):
    if name == "catboost":
        cls = lib.CatBoostClassifier if classifier else lib.CatBoostRegressor
        return cls(
            iterations=MODEL_ROUNDS, depth=5, learning_rate=0.09, random_seed=SEED,
            thread_count=THREADS, verbose=False, allow_writing_files=False,
            task_type=device, cat_features=cats,
            **({"devices": "0"} if device == "GPU" else {}),
        )
    if name == "lightgbm":
        cls = lib.LGBMClassifier if classifier else lib.LGBMRegressor
        return cls(
            n_estimators=MODEL_ROUNDS, max_depth=5, num_leaves=31, learning_rate=0.09,
            random_state=SEED, n_jobs=THREADS, verbosity=-1, deterministic=True,
            force_col_wise=True,
        )
    cls = lib.XGBClassifier if classifier else lib.XGBRegressor
    return cls(
        n_estimators=MODEL_ROUNDS, max_depth=5, learning_rate=0.09,
        random_state=SEED, n_jobs=THREADS, tree_method="hist", verbosity=0,
        device="cuda:0" if device == "GPU" else "cpu",
        eval_metric="logloss" if classifier else "rmse",
    )


def predict_heads(model, lib, device, train_x, test_x, targets, train, cats):
    result = {}
    for head, truth in targets.items():
        classifier = head != "expected_runs"
        y = truth.loc[train]
        assert len(y) and (not classifier or y.nunique() == 2), (model, head)
        estimator = benchmark_estimator(model, lib, device, classifier, cats)
        estimator.fit(train_x, y)
        prediction = estimator.predict_proba(test_x)[:, 1] if classifier else estimator.predict(test_x)
        prediction = np.asarray(prediction, dtype=float)
        assert prediction.shape == (len(test_x),) and np.isfinite(prediction).all(), (model, head)
        result[head] = prediction.clip(0, 1) if classifier else prediction.clip(0)
    return pd.DataFrame(result)


def evaluate_heads(name, role, fold, frame, targets, test, pressure, device, train_end, test_start):
    result = {
        "model": name, "role": role, "fold": fold, "device": device,
        "train_end": str(train_end.date()), "test_start": str(test_start.date()),
        "eval_attempts": len(test), "pressure_attempts": int(pressure.sum()),
    }
    pressure = pressure.to_numpy()
    for head, actual in targets.items():
        y = actual.loc[test].to_numpy()
        pred = frame[head].to_numpy()
        error = np.square(y - pred)
        suffix = "rmse" if head == "expected_runs" else "brier"
        result[f"{head}_{suffix}"] = float(np.sqrt(error.mean()) if suffix == "rmse" else error.mean())
        if pressure.any():
            pressure_error = error[pressure].mean()
            result[f"pressure_{head}_{suffix}"] = float(
                np.sqrt(pressure_error) if suffix == "rmse" else pressure_error
            )
    return result


def predict_all_models(game, bands, blocks, libs, role):
    fields = CUMULATIVE_CONTEXT_FIELDS + CUMULATIVE_OPPOSITION_FIELDS[role]
    features = game[fields].copy()
    cats = features.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    for c in cats:
        features[c] = features[c].astype("string").fillna("<unknown>").astype(str)
    outcomes = CUMULATIVE_OUTCOMES[role]
    target = {
        "expected_runs": game[outcomes[0]],
        "expected_wickets": game[outcomes[1]],
        "expected_illegal": 1 - game["y_legal_delivery_candidate"],
    }
    available = role_available(game, role)
    rates = chase_rate_outcomes(game)
    stack = {model: [] for model in RANKING_MODELS}
    audits = []
    for fold, (start, end) in enumerate(blocks):
        if fold == 0:
            continue
        train = game.index[available & game["date"].lt(start)]
        test = game.index[available & game["date"].between(start, end)]
        train_end = game.loc[train, "date"].max()
        assert len(test) and train_end < start
        assert not set(game.loc[train, "match_id"]) & set(game.loc[test, "match_id"])
        train_x, test_x = features.loc[train], features.loc[test]
        encoded = encoded_fold(train_x, test_x, cats)
        high = bands.loc[test].eq("pressure").any(axis=1)
        forecast = {}
        for name, (lib, device) in libs.items():
            inputs = (train_x, test_x) if name == "catboost" else encoded
            frame = predict_heads(name, lib, device, *inputs, target, train, cats)
            frame.index = test
            forecast[name] = frame
            audits.append(evaluate_heads(name, role, fold, frame, target, test, high, device, train_end, start))
        assert set(ENSEMBLE_WEIGHTS) == set(BASE_MODELS)
        assert np.isclose(sum(ENSEMBLE_WEIGHTS.values()), 1) and min(ENSEMBLE_WEIGHTS.values()) >= 0
        ensemble = sum(forecast[name] * ENSEMBLE_WEIGHTS[name] for name in BASE_MODELS)
        forecast["ensemble"] = ensemble
        audits.append(evaluate_heads("ensemble", role, fold, ensemble, target, test, high, "blend", train_end, start))
        ids = [ROLE_IDS[role], ROLE_NAMES[role]]
        needed = sorted(set([
            *fields, *outcomes, *ids, "row_id", "date", "match_id", "innings_ball_num",
            "x_wickets_remaining", "y_legal_delivery_candidate",
        ]))
        base = game.loc[test, needed].rename(columns={
            ids[0]: "player_id", ids[1]: "player_name",
        })
        base = base.assign(
            fold=fold, runs_observed=target["expected_runs"].loc[test],
            wickets_observed=target["expected_wickets"].loc[test],
            legal_delivery=game.loc[test, "y_legal_delivery_candidate"],
        ).join(rates.loc[test, ["required_rate_before", "required_rate_after", "required_rate_change"]])
        tags = bands.loc[test].add_prefix("band_")
        base = pd.concat([base, tags], axis=1)
        for name, frame in forecast.items():
            stack[name].append(base.join(frame))
        print(f"{role} fold {fold}: all three models and ensemble scored {len(test):,} attempts.", flush=True)
    return {name: pd.concat(frames) for name, frames in stack.items()}, audits

In [ ]:
def round01(data):
    return data.clip(0.0, 1.0)


def role_context(game, role):
    live = game.innings.eq(2) & game.x_runs_required.gt(0) & game.x_balls_remaining.gt(0)
    pace = (game.x_required_run_rate / 6).where(live)
    probability = game.x_historical_chase_success_probability.clip(0, 1).fillna(0.5)
    contest = 4 * probability * (1 - probability)
    late = round01((game.x_legal_balls_used - 36) / 84)
    scarcity = round01((5 - game.x_wickets_remaining) / 4)
    threat = round01((game[f"x_{role}_opponent_ability10"].fillna(5) - 5) / 3)
    if role == "batter":
        urgency = round01((game.x_required_run_rate - 7) / 6).fillna(0)
        support = game.x_remaining_top3_strength_score100_proxy
        thin_support = round01((60 - support.fillna(60)) / 25)
        wicket_cost = 1 + 0.55 * scarcity + 0.35 * thin_support
        first_demand = round01(-game.x_score_minus_trajectory_proxy / 25).fillna(0)
    else:
        urgency = (1 - round01((game.x_required_run_rate - 6) / 7)).fillna(0)
        set_batter = round01(game.x_striker_nonwide_deliveries_before / 24)
        strong_stand = round01(game.x_partnership_legal_balls_before / 36)
        wicket_cost = 1 + 0.40 * threat + 0.25 * set_batter + 0.20 * strong_stand + 0.35 * game.x_wickets_remaining.eq(1)
        first_demand = round01(game.x_score_minus_trajectory_proxy / 25).fillna(0)
    chase_severity = 0.45 * contest + 0.25 * late + 0.20 * urgency + 0.10 * (scarcity if role == "batter" else threat)
    first_severity = 0.45 * first_demand + 0.25 * late + 0.20 * threat + 0.10 * game.x_knockout
    severity = 0.25 + 0.75 * chase_severity.where(live, first_severity)
    return live, pace, wicket_cost, severity


def response_mask(data, trigger):
    selected = np.zeros(len(data), dtype=bool)
    groups = data.groupby(["match_id", "innings", "player_id"], sort=False).indices
    legal = data.x_legal_balls_used.to_numpy()
    qualifying = trigger.to_numpy(dtype=bool)
    for indices in groups.values():
        position = legal[indices]
        active = np.flatnonzero(qualifying[indices])
        end_position = -1
        for i in active:
            if position[i] <= end_position:
                continue
            end_position = position[i] + 5
            stop = np.searchsorted(position, end_position, side="right")
            selected[indices[i:stop]] = True
    return pd.Series(selected, index=data.index)


def per_scenario_scores(data, recipe, mask, divisor, role, context, keep_deliveries):
    live, pace, wicket_cost, severity = context
    runs_field, wickets_field = CUMULATIVE_OUTCOMES[role]
    runs = data[runs_field]
    observed = data[wickets_field]
    legal = data.y_legal_delivery_candidate
    illegal = 1 - legal
    predicted_illegal = data.expected_illegal
    predicted_runs = data.expected_runs
    predicted_wickets = data.expected_wickets
    bat = role == "batter"
    uses_discipline = recipe["discipline"] > 0
    if uses_discipline:
        runs = runs - illegal
        predicted_runs = predicted_runs - predicted_illegal
    if bat:
        contextual = runs - predicted_runs
        achieved = runs - pace * legal
        expected_chase = predicted_runs - pace * (1 - predicted_illegal)
        versus_context = achieved - expected_chase
        chase = 0.5 * versus_context + 0.5 * achieved
        if recipe["number"] == 19:
            chase = achieved
        wicket = (predicted_wickets - observed) * wicket_cost
    else:
        contextual = predicted_runs - runs
        achieved = pace * legal - runs
        expected_chase = pace * (1 - predicted_illegal) - predicted_runs
        chase = 0.5 * (achieved - expected_chase) + 0.5 * achieved
        wicket = (observed - predicted_wickets) * wicket_cost
    contextual_or_chase = chase.where(live, contextual)
    if recipe["metric"] in ("Chase progress", "Chase squeeze"):
        run_signal = chase.where(live, contextual)
    elif recipe["metric"] in ("Context-adjusted bat runs", "Run suppression"):
        run_signal = contextual
    else:
        run_signal = contextual_or_chase
    discipline = predicted_illegal - illegal
    idx = mask[mask].index
    evidence_weight = severity.loc[idx].div(divisor.loc[idx])
    events = pd.DataFrame({
        "player_id": data.loc[idx, "player_id"],
        "match_id": data.loc[idx, "match_id"],
        "evidence": evidence_weight,
        "runs": run_signal.loc[idx] * evidence_weight,
        "wickets": wicket.loc[idx] * evidence_weight,
        "discipline": discipline.loc[idx] * evidence_weight,
    })
    assert np.isfinite(events[["evidence", "runs", "wickets", "discipline"]]).all().all()
    matches = events.groupby(["player_id", "match_id"]).agg(
        runs=("runs", "sum"), wickets=("wickets", "sum"), discipline=("discipline", "sum"),
        evidence=("evidence", "sum"),
        attempts=("evidence", "size"),
    )
    matches["severity"] = matches.evidence.div(matches.attempts)
    for col in ("runs", "wickets", "discipline"):
        matches[col] = matches[col] / matches.evidence
        matches[col] *= matches.severity
    player = matches.groupby("player_id").agg(
        runs=("runs", "sum"), wickets=("wickets", "sum"), discipline=("discipline", "sum"),
        effective=("severity", "sum"), attempts=("attempts", "sum"), matches=("severity", "size"),
    )
    confidence = player.effective.div(player.effective + 6) * (player.attempts / 18).clip(upper=1)
    score = np.zeros(len(player))
    recipe_mix = {
        "runs": recipe["runs"], "wickets": recipe["wicket"],
        "discipline": recipe["discipline"],
    }
    for metric, pct in recipe_mix.items():
        measure = player[metric].div(player.effective)
        score += (50 + 50 * np.tanh(measure / CUMULATIVE_SCALES[metric])) * pct / 100
    player["pressure_score100"] = (50 + confidence * (score - 50)).round(2)
    player["pressure_attempts"] = player["attempts"].astype(int)
    player["pressure_matches"] = player["matches"].astype(int)
    if not keep_deliveries:
        return player[["pressure_score100", "pressure_attempts", "pressure_matches"]], None
    standardized = (
        run_signal * recipe_mix["runs"] / CUMULATIVE_SCALES["runs"]
        + wicket * recipe_mix["wickets"] / CUMULATIVE_SCALES["wickets"]
        + discipline * recipe_mix["discipline"] / CUMULATIVE_SCALES["discipline"]
    ) / 100
    measured = data.loc[idx, [
        "row_id", "player_id", "runs_observed", "wickets_observed", "legal_delivery",
        "required_rate_change",
    ]].copy()
    measured["cumulative_run_signal"] = run_signal.loc[idx]
    measured["cumulative_wicket_signal"] = wicket.loc[idx]
    measured["cumulative_discipline_signal"] = discipline.loc[idx]
    measured["cumulative_evidence_weight"] = evidence_weight
    measured["above_reference"] = standardized.loc[idx].gt(0)
    observed = measured.groupby("player_id").agg(
        runs_observed=("runs_observed", "sum"), wickets_observed=("wickets_observed", "sum"),
        legal_deliveries=("legal_delivery", "sum"),
        above_reference_pct=("above_reference", "mean"),
        mean_required_rate_change=("required_rate_change", "mean"),
    )
    observed["above_reference_pct"] *= 100
    result = player[["pressure_score100", "pressure_attempts", "pressure_matches"]].join(observed)
    return result, measured


def cumulative_score_role(data, role, keep_deliveries=False):
    data = data.sort_values(["date", "match_id", "innings", "innings_ball_num"]).reset_index(drop=True)
    recipes = CUMULATIVE_RECIPES[role]
    catalog = list(SCENARIO_CATALOG[role])
    assert [r["key"] for r in recipes] == catalog
    assert all(r["runs"] + r["wicket"] + r["discipline"] == 100 for r in recipes)
    families = CUMULATIVE_FAMILIES[role]
    family_by_number = {n: f for f, numbers in families.items() for n in numbers}
    assert sorted(n for numbers in families.values() for n in numbers) == list(range(1, 51))
    direct = {r["number"]: data[f"band_{r['key']}"].eq("pressure") for r in recipes}
    masks = {
        r["number"]: (
            response_mask(data, direct[r["number"]])
            if r["metric"] in ("Scoring recovery", "Control after a setback")
            else direct[r["number"]]
        ) for r in recipes
    }
    divisors = {
        name: pd.DataFrame({n: masks[n] for n in numbers}).sum(axis=1).clip(lower=1)
        for name, numbers in families.items()
    }
    latest = data.drop_duplicates("player_id", keep="last").set_index("player_id")
    pressure = pd.DataFrame(direct).any(axis=1)
    counts = data.loc[pressure].groupby("player_id").agg(
        pressure_attempts=("match_id", "size"), pressure_matches=("match_id", "nunique"),
    )
    candidates = counts.loc[
        counts["pressure_attempts"].ge(MIN_OVERALL_ATTEMPTS)
        & counts["pressure_matches"].ge(MIN_OVERALL_MATCHES)
    ].join(latest[["player_name"]])
    context = role_context(data, role)
    values, attempts, matches, scenarios = {}, {}, {}, {}
    evidence = SCENARIO_MINIMUM
    all_attempts = data.groupby("player_id").size() if keep_deliveries else None
    for recipe in recipes:
        number = recipe["number"]
        name = recipe["key"]
        family = family_by_number[number]
        code = f"{'BAT' if role == 'batter' else 'BOWL'}{number:02}"
        result, deliveries = per_scenario_scores(
            data, recipe, masks[number], divisors[family], role, context, keep_deliveries,
        )
        scores = result.reindex(candidates.index)
        values[code] = scores["pressure_score100"].fillna(50)
        attempts[code] = scores["pressure_attempts"].fillna(0).astype(int)
        matches[code] = scores["pressure_matches"].fillna(0).astype(int)
        supported = (
            result["pressure_attempts"].ge(evidence[0])
            & result["pressure_matches"].ge(evidence[1])
        )
        scenarios[name] = {
            "supported": result.loc[supported, "pressure_score100"],
            "family": family, "code": code,
        }
        if keep_deliveries:
            table_ids = result.index[supported].union(candidates.index, sort=False)
            table = result.reindex(table_ids)
            table["pressure_score100"] = table["pressure_score100"].fillna(50)
            for field in ("pressure_attempts", "pressure_matches"):
                table[field] = table[field].fillna(0).astype(int)
            table["has_scenario_evidence"] = table["pressure_attempts"].gt(0)
            table["scenario_rank_eligible"] = table.index.isin(result.index[supported])
            table["overall_eligible"] = table.index.isin(candidates.index)
            table["player_name"] = latest["player_name"].reindex(table.index)
            table["pressure_exposure_pct"] = 100 * table["pressure_attempts"].div(all_attempts)
            table = table.reset_index().sort_values(
                ["scenario_rank_eligible", "pressure_score100", "pressure_matches",
                 "pressure_attempts", "player_id"],
                ascending=[False, False, False, False, True], kind="stable",
            ).reset_index(drop=True)
            eligible_count = int(table["scenario_rank_eligible"].sum())
            ranks = pd.array(
                list(range(1, eligible_count + 1)) + [None] * (len(table) - eligible_count),
                dtype="Int64",
            )
            table.insert(0, "rank", ranks)
            scenarios[name].update(table=table, deliveries=deliveries)
    points = pd.DataFrame(values, index=candidates.index)
    measured = pd.DataFrame(attempts, index=candidates.index)
    repeated = pd.DataFrame(matches, index=candidates.index)
    assert points.shape[1] == ROLE_SCENARIO_COUNTS[role]
    assert points.ge(0).all().all() and points.le(100).all().all()
    overall = candidates.copy()
    overall["scenarios_observed"] = measured.gt(0).sum(axis=1)
    overall["scenarios_supported"] = (
        measured.ge(evidence[0]) & repeated.ge(evidence[1])
    ).sum(axis=1)
    overall["total_score5000"] = points.sum(axis=1).round(2)
    overall["pressure_score100"] = overall["total_score5000"] / ROLE_SCENARIO_COUNTS[role]
    for family, numbers in families.items():
        prefix = "BAT" if role == "batter" else "BOWL"
        cols = [f"{prefix}{number:02}" for number in numbers]
        overall[family] = points[cols].sum(axis=1).round(2)
    overall = overall.join(points).reset_index().sort_values(
        ["total_score5000", "scenarios_supported", "pressure_matches", "player_id"],
        ascending=[False, False, False, True], kind="stable",
    ).reset_index(drop=True)
    overall.insert(0, "rank", np.arange(1, len(overall) + 1))
    overall.insert(0, "role", role)
    return overall, scenarios


def paired_rank_ndcg(earlier, later):
    paired = earlier.rename("earlier").to_frame().join(later.rename("later"), how="inner")
    score = None
    if len(paired) >= TOP_K:
        truth = paired["later"].rank(pct=True).to_numpy()
        estimates = paired["earlier"].to_numpy()
        score = float(ndcg_score(truth[None], estimates[None], k=TOP_K))
    return {"matched_players": len(paired), "ndcg_at_10": score}

In [ ]:
def score_combinations(data, role, overall, verbose):
    resolutions = CUMULATIVE_RECIPES[role]
    recipe_key = {r["number"]: r["key"] for r in resolutions}
    setup = COMBINATIONS[role]
    masks = {}
    for key, _, indices, _, _, _, _, _ in setup:
        assert len(set(indices)) == len(indices) >= 2
        columns = [f"band_{recipe_key[n]}" for n in indices]
        masks[key] = data[columns].eq("pressure").all(axis=1)
    divisor = pd.DataFrame(masks).sum(axis=1).clip(lower=1)
    context = role_context(data, role)
    candidates = pd.Index(overall["player_id"], name="player_id")
    metric, pairs, definitions = {}, [], []
    latest = data.sort_values(["date", "match_id"]).drop_duplicates("player_id", keep="last")
    names = latest.set_index("player_id")["player_name"]
    weights = {row[0]: row[-1] for row in setup}
    for key, label, indices, measure, runs, wicket, discipline, _ in setup:
        assert min(runs, wicket, discipline) >= 0 and runs + wicket + discipline == 100
        recipe = dict(number=0, metric=measure, runs=runs, wicket=wicket, discipline=discipline)
        result, _ = per_scenario_scores(
            data, recipe, masks[key], divisor, role, context, False,
        )
        metric[key] = result["pressure_score100"].reindex(candidates).fillna(50)
        if not verbose:
            continue
        supported = result["pressure_attempts"].ge(SCENARIO_MINIMUM[0]) & result["pressure_matches"].ge(SCENARIO_MINIMUM[1])
        entries = result.reindex(result.index[supported].union(candidates, sort=False)).copy()
        entries["pressure_score100"] = entries["pressure_score100"].fillna(50)
        for col in ("pressure_attempts", "pressure_matches"):
            entries[col] = entries[col].fillna(0).astype(int)
        entries["has_scenario_evidence"] = entries["pressure_attempts"].gt(0)
        entries["scenario_rank_eligible"] = entries.index.isin(result.index[supported])
        entries["overall_eligible"] = entries.index.isin(candidates)
        entries["player_name"] = names.reindex(entries.index)
        entries = entries.reset_index().sort_values(
            ["scenario_rank_eligible", "pressure_score100", "pressure_matches", "pressure_attempts", "player_id"],
            ascending=[False, False, False, False, True], kind="stable",
        ).reset_index(drop=True)
        count = int(entries["scenario_rank_eligible"].sum())
        entries.insert(0, "rank", pd.array(
            list(range(1, count + 1)) + [None] * (len(entries) - count), dtype="Int64",
        ))
        entries = entries.assign(role=role, combination_id=key, combination=label)
        pairs.append(entries)
        definitions.append({
            "role": role, "combination_id": key, "combination": label,
            "scenario_numbers": " + ".join(map(str, indices)),
            "scenario_ids": " + ".join(recipe_key[n] for n in indices),
            "measure": measure, "run_weight_pct": runs, "wicket_weight_pct": wicket,
            "discipline_weight_pct": discipline, "combination_weight": weights[key],
            "scored_attempts": int(masks[key].sum()),
            "pressure_matches": int(data.loc[masks[key], "match_id"].nunique()),
            "eligible_players": count, "top10_rows": min(count, TOP_K),
        })
    values = pd.DataFrame(metric, index=candidates)
    assert len(values.columns) == len(setup)
    combination_score = values.mul(pd.Series(weights)).sum(axis=1) / sum(weights.values())
    features = pd.DataFrame({
        "combination_score100": combination_score,
        "combination_adjustment": COMBO_CAP * (combination_score - 50) / 50,
    })
    return features, (pd.concat(pairs, ignore_index=True) if pairs else None), pd.DataFrame(definitions)


def pressure_match_records(data, role, context_wickets, context_runs):
    tags = [f"band_{r['key']}" for r in CUMULATIVE_RECIPES[role]]
    use = data[tags].eq("pressure").any(axis=1)
    _, _, wicket_cost, severity = role_context(data, role)
    runs = data["runs_observed"] - context_runs
    wickets = context_wickets - data["wickets_observed"]
    edge = (runs + WICKET_VALUE * wickets * wicket_cost) * (1 if role == "batter" else -1)
    sample = data.loc[use, ["player_id", "match_id"]].assign(
        weighted_edge=edge.loc[use] * severity.loc[use], severity=severity.loc[use],
    )
    by_match = sample.groupby(["player_id", "match_id"]).agg(
        weighted_edge=("weighted_edge", "sum"), evidence=("severity", "sum"),
        attempts=("severity", "size"),
    )
    by_match = by_match.loc[by_match["attempts"].ge(REPEAT_MATCH_MIN_ATTEMPTS)].copy()
    by_match["edge"] = by_match["weighted_edge"] / by_match["evidence"]
    return by_match


def repetition_credit(data, role, players):
    by_match = pressure_match_records(data, role, data["expected_wickets"], data["expected_runs"])
    by_match["positive"] = by_match["edge"].gt(REPEAT_MATCH_MIN_EDGE)
    counted = by_match.groupby("player_id").agg(
        repeat_positive_matches=("positive", "sum"), repeat_assessed_matches=("positive", "size"),
    ).reindex(players, fill_value=0).astype(int)
    positive = counted["repeat_positive_matches"]
    fraction = positive.div(counted["repeat_assessed_matches"].replace(0, np.nan)).fillna(0)
    counted["repeat_bonus"] = REPEAT_CAP * (1 - np.exp(-(positive - 1).clip(lower=0) / REPEAT_SATURATION)) * fraction
    return counted


def score_complete_role(data, role, detail=False):
    solo, scenarios = cumulative_score_role(data, role, keep_deliveries=detail)
    combination, combinations, definitions = score_combinations(data, role, solo, detail)
    result = solo.drop(columns="rank").set_index("player_id").join(combination)
    result = result.join(repetition_credit(data, role, result.index)).reset_index()
    result = result.rename(columns={"pressure_score100": "solo_score100"})
    result["pressure_score100"] = (
        result["solo_score100"] + result["combination_adjustment"] + result["repeat_bonus"]
    ).clip(0, 100)
    result = result.sort_values(
        ["pressure_score100", "total_score5000", "scenarios_supported", "pressure_matches", "player_id"],
        ascending=[False, False, False, False, True], kind="stable",
    ).reset_index(drop=True)
    result.insert(1, "rank", np.arange(1, len(result) + 1))
    return result, scenarios, combinations, definitions


def shared_holdout_reference(game, later, role):
    names = (
        ["x_context_expected_striker_wicket", "x_context_expected_batter_runs"]
        if role == "batter" else
        ["x_context_expected_bowler_wicket", "x_context_expected_bowler_runs"]
    )
    baseline = game[["row_id", *names]].set_index("row_id").loc[later["row_id"]].set_axis(later.index)
    valid = baseline.notna().all(axis=1)
    matches = pressure_match_records(later.loc[valid], role, baseline.loc[valid, names[0]], baseline.loc[valid, names[1]])
    reference = matches.groupby("player_id").agg(
        performance=("edge", "mean"), match_count=("edge", "size"), attempts=("attempts", "sum"),
    )
    valid = reference["match_count"].ge(3) & reference["attempts"].ge(18)
    return reference.loc[valid, "performance"] * reference.loc[valid, "match_count"].div(
        reference.loc[valid, "match_count"] + 6
    )

In [ ]:
def write_csv_gzip(path, data):
    temporary = path.with_suffix(".tmp")
    data.to_csv(temporary, index=False, compression="gzip")
    temporary.replace(path)


def root_for(model):
    return OUTPUT_DIR if model == SUBMISSION_MODEL else COMPARISON_DIR / model


def scenario_exports(root, role, model, overall, scenarios, data):
    index, definitions, shortlists = [], [], []
    score = overall.set_index("player_id")
    for recipe in CUMULATIVE_RECIPES[role]:
        number, key = recipe["number"], recipe["key"]
        condition = scenarios[key]
        spec = SCENARIO_CATALOG[role][key]
        path = Path("scenarios") / role / f"{number:02d}_{key}"
        destination = root / path
        destination.mkdir(parents=True, exist_ok=True)
        table = condition["table"].assign(
            role=role, scenario_id=key, pressure_family=condition["family"],
        )
        candidates = table.loc[table["overall_eligible"]].set_index("player_id")
        assert set(candidates.index) == set(score.index)
        np.testing.assert_allclose(
            candidates["pressure_score100"].reindex(score.index), score[condition["code"]],
        )
        shortlist = table.loc[table["scenario_rank_eligible"]].head(TOP_K).copy()
        assert shortlist["rank"].tolist() == list(range(1, len(shortlist) + 1))
        write_csv(destination / "rankings.csv", table)
        if model == SUBMISSION_MODEL:
            write_csv(destination / "top10.csv", shortlist)
            evidence = condition["deliveries"].merge(
                data[[
                    "row_id", "date", "match_id", "innings", "player_name",
                    "expected_runs", "expected_wickets", "expected_illegal", f"band_{key}",
                ]],
                on="row_id", how="left", validate="one_to_one",
            ).rename(columns={f"band_{key}": "pressure_band"})
            evidence["cumulative_scored_attempt"] = True
            write_csv_gzip(destination / "delivery_predictions.csv.gz", evidence)
        shortlists.append(shortlist.assign(scenario=spec["scenario"]))
        match_counts = condition["deliveries"]["row_id"].size
        index.append({
            "role": role, "scenario_number": number, "scenario_id": key,
            "score_column": condition["code"], "scenario": spec["scenario"],
            "pressure_family": condition["family"], "origin": spec["source"],
            "eligible_players": len(condition["supported"]),
            "top10_rows": len(shortlist), "unfilled_slots": TOP_K - len(shortlist),
            "forecast_rows": match_counts, "cumulative_scored_rows": match_counts,
            "relative_path": str(path),
        })
        definitions.append({
            "role": role, "scenario_number": number, "scenario_id": key,
            "scenario": spec["scenario"], "family": condition["family"],
            "score_column": condition["code"], "pressure_rule": spec["rule"],
            "pressure_mechanism": spec["pressure"], "cumulative_measure": recipe["metric"],
            "run_weight_pct": recipe["runs"], "wicket_weight_pct": recipe["wicket"],
            "discipline_weight_pct": recipe["discipline"],
        })
    return pd.DataFrame(index), pd.DataFrame(definitions), pd.concat(shortlists, ignore_index=True)


def verify_sums(rows, role):
    prefix = "BAT" if role == "batter" else "BOWL"
    columns = [f"{prefix}{i:02d}" for i in range(1, 51)]
    assert rows[columns].notna().all().all()
    assert np.isfinite(rows[columns].to_numpy()).all()
    assert rows[columns].ge(0).all().all() and rows[columns].le(100).all().all()
    np.testing.assert_allclose(rows["total_score5000"], rows[columns].sum(axis=1).round(2), atol=1e-9, rtol=0)
    np.testing.assert_allclose(rows["solo_score100"], rows["total_score5000"] / 50, atol=1e-9, rtol=0)
    adjustment = rows["combination_adjustment"]
    assert adjustment.between(-COMBO_CAP, COMBO_CAP).all()
    assert rows["repeat_bonus"].between(0, REPEAT_CAP).all()
    assert rows.loc[rows["repeat_positive_matches"].lt(2), "repeat_bonus"].eq(0).all()
    np.testing.assert_allclose(
        rows["pressure_score100"],
        (rows["solo_score100"] + adjustment + rows["repeat_bonus"]).clip(0, 100),
        atol=1e-9, rtol=0,
    )


def export_rankings(game, frames, all_audits):
    all_metrics = []
    all_top = []
    id_columns = ["role", "rank", "player_id", "player_name"]
    scoring_columns = [
        "total_score5000", "solo_score100", "combination_score100", "combination_adjustment",
        "repeat_positive_matches", "repeat_assessed_matches", "repeat_bonus", "pressure_score100",
        "scenarios_observed", "scenarios_supported", "pressure_matches", "pressure_attempts",
    ]
    holdout_ref = {}
    for role in ROLE_IDS:
        sample = frames[role][BASE_MODELS[0]]
        later = sample.loc[sample["fold"].eq(sample["fold"].max())]
        holdout_ref[role] = shared_holdout_reference(game, later, role)
    for model in RANKING_MODELS:
        root = root_for(model)
        folder = root / "overall"
        folder.mkdir(parents=True, exist_ok=True)
        parts = defaultdict(list)
        model_audit = all_audits.loc[all_audits["model"].eq(model)]
        for role in ROLE_IDS:
            data = frames[role][model]
            overall, scenes, combinations, combo_index = score_complete_role(data, role, detail=True)
            assert len(overall) >= TOP_K and overall["player_id"].is_unique
            verify_sums(overall, role)
            indexed = overall.set_index("player_id")
            combo = combinations.loc[combinations["overall_eligible"]].pivot(
                index="player_id", columns="combination_id", values="pressure_score100",
            )
            weights = combo_index.set_index("combination_id")["combination_weight"]
            assert combo.notna().all().all() and set(combo.columns) == set(weights.index)
            expected_combo = combo.mul(weights).sum(axis=1) / weights.sum()
            np.testing.assert_allclose(
                indexed["combination_score100"], expected_combo.reindex(indexed.index),
                atol=1e-9, rtol=0,
            )
            last_fold = data["fold"].max()
            earlier = data.loc[data["fold"].lt(last_fold)]
            later = data.loc[data["fold"].eq(last_fold)]
            assert earlier["date"].max() < later["date"].min()
            earlier_scores, early_scenes, _, _ = score_complete_role(earlier, role)
            _, late_scenes = cumulative_score_role(later, role)
            common_holdout = paired_rank_ndcg(
                earlier_scores.set_index("player_id")["pressure_score100"], holdout_ref[role],
            )
            validation = {
                "role": role, "model": model, "holdout_from": str(later["date"].min().date()),
                "holdout_to": str(later["date"].max().date()), **common_holdout,
            }
            parts["validation"].append(validation)
            latest_audit = model_audit.loc[
                model_audit["role"].eq(role) & model_audit["fold"].eq(last_fold)
            ].iloc[0].to_dict()
            all_metrics.append({**latest_audit, **validation})
            index, definitions, scenario_top10 = scenario_exports(root, role, model, overall, scenes, data)
            for key in index["scenario_id"]:
                mask = index["scenario_id"].eq(key)
                metric = paired_rank_ndcg(early_scenes[key]["supported"], late_scenes[key]["supported"])
                index.loc[mask, "matched_players"] = metric["matched_players"]
                index.loc[mask, "ndcg_at_10"] = metric["ndcg_at_10"]
            parts["scenario_index"].append(index)
            parts["scenario_definitions"].append(definitions)
            parts["scenario_top10"].append(scenario_top10)
            combo_top10 = combinations.loc[
                combinations["scenario_rank_eligible"] & combinations["rank"].le(TOP_K)
            ].copy()
            parts["combination_index"].append(combo_index)
            parts["combination_rankings"].append(combinations)
            parts["combination_top10"].append(combo_top10)
            top = overall.head(TOP_K).copy()
            write_csv(folder / f"{role}_rankings.csv", overall)
            write_csv(folder / f"{role}_top10.csv", top)
            parts["overall"].append(overall[id_columns + scoring_columns])
            parts["top"].append(top[id_columns + scoring_columns])
            print(f"{model} {role}: 50 scenarios and {len(combo_index)} combinations.", flush=True)
        for filename in ("scenario_index", "scenario_definitions", "scenario_top10",
                         "combination_index", "combination_rankings", "combination_top10"):
            write_csv(root / f"{filename}.csv", pd.concat(parts[filename], ignore_index=True))
        write_json(folder / "holdout_validation.json", parts["validation"])
        write_csv(root / "model_validation.csv", model_audit)
        scoring = pd.concat(parts["overall"], ignore_index=True)
        top = pd.concat(parts["top"], ignore_index=True)
        write_csv(folder / "combined_rankings.csv", scoring)
        write_csv(folder / "combined_top10.csv", top)
        export = top[id_columns]
        write_csv(WORKING / "submission.csv" if model == SUBMISSION_MODEL else root / "submission.csv", export)
        verified = pd.read_csv(WORKING / "submission.csv" if model == SUBMISSION_MODEL else root / "submission.csv", dtype={"player_id": str})
        pd.testing.assert_frame_equal(export.reset_index(drop=True), verified)
        assert verified.groupby("role")["rank"].agg(list).to_dict() == {
            role: list(range(1, TOP_K + 1)) for role in ROLE_IDS
        }
        all_top.append(top.assign(model=model))
    matrix = pd.concat(all_top, ignore_index=True)
    write_csv(COMPARISON_DIR / "all_models_top10.csv", matrix)
    write_csv(COMPARISON_DIR / "model_comparison.csv", pd.DataFrame(all_metrics))
    overlap = []
    for role in ROLE_IDS:
        pairs = matrix.loc[matrix["role"].eq(role)].groupby("model")["player_id"].agg(set)
        for i, left in enumerate(RANKING_MODELS):
            for right in RANKING_MODELS[i + 1:]:
                overlap.append({
                    "role": role, "model_1": left, "model_2": right,
                    "top10_overlap": len(pairs[left] & pairs[right]),
                })
    write_csv(COMPARISON_DIR / "top10_overlap.csv", pd.DataFrame(overlap))
    assert len(matrix) == len(RANKING_MODELS) * len(ROLE_IDS) * TOP_K


def run_comparison():
    assert SUBMISSION_MODEL in RANKING_MODELS
    run = compile_from_inputs()
    COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
    libs = libraries_and_devices()
    print("Models: " + ", ".join(f"{n} {lib.__version__} ({device})" for n, (lib, device) in libs.items()), flush=True)
    results, audits = {}, []
    for role in ROLE_IDS:
        results[role], trail = predict_all_models(
            run["game"], run["bands"][role], run["blocks"], libs, role,
        )
        audits.extend(trail)
    checkpoint_checklist("forecasts")
    trail = pd.DataFrame(audits)
    assert len(trail) == len(RANKING_MODELS) * len(ROLE_IDS) * (len(run["blocks"]) - 1)
    export_rankings(run["game"], results, trail)
    write_json(OUTPUT_DIR / "score_definitions.json", {
        "solo_score100": "Sum the 50 scenario scores out of 5000 and divide by 50.",
        "combination": "Each listed combination requires every component pressure flag before the same delivery. Scores use batting or bowling recipes; configurations define the combination weights.",
        "combination_adjustment": f"{COMBO_CAP} times the weighted combination score's deviation from neutral 50 divided by 50.",
        "repeat_bonus": f"At most {REPEAT_CAP} points. Count distinct matches with at least {REPEAT_MATCH_MIN_ATTEMPTS} pressure attempts and performance greater than {REPEAT_MATCH_MIN_EDGE} over expectation. The bonus starts after the first qualifying positive match, saturates, and is scaled by the positive fraction of assessed matches.",
        "final_score": "Clip solo_score100 + combination_adjustment + repeat_bonus to 0–100.",
        "model_selection": SUBMISSION_MODEL,
        "ensemble": ENSEMBLE_WEIGHTS,
        "validation": "Matching chronological folds for each engine. Earlier rankings use a shared held-out observed-results reference for comparison.",
        "unobserved_score": 50,
    })
    write_json(COMPARISON_DIR / "dashboard_manifest.json", {
        "models": {
            model: str(root_for(model).relative_to(WORKING)) for model in RANKING_MODELS
        },
        "submission_model": SUBMISSION_MODEL,
        "outputs_per_model": {
            "overall_top10": "overall/combined_top10.csv",
            "solo_index": "scenario_index.csv",
            "solo_top10": "scenario_top10.csv",
            "combination_index": "combination_index.csv",
            "combination_rankings": "combination_rankings.csv",
            "combination_top10": "combination_top10.csv",
        },
        "final_score_column": "pressure_score100",
        "solo_total_column": "total_score5000",
        "solo_normalized_column": "solo_score100",
        "combination_adjustment_column": "combination_adjustment",
        "repeat_bonus_column": "repeat_bonus",
    })
    checkpoint_checklist("exports")
    print("Verified four top-10 lists per role and a twenty-row submission.", flush=True)

In [ ]:
CHECKLIST_PROGRESS = dict.fromkeys(CHECKLIST, "PENDING")


def load_feature_game(paths, data_key):
    feature_cache = eligible_cache("features", data_key)
    if feature_cache is not None:
        game = pd.read_parquet(feature_cache / "derived_deliveries.parquet")
        print(f"Reused {len(game):,} verified derived deliveries.")
        return game
    game = construct_dataset(paths)
    write_parquet(OUTPUT_DIR / "derived_deliveries.parquet", game)
    save_receipt("features", data_key, [
        "derived_deliveries.parquet", "batter_ability_by_date.parquet",
        "bowler_ability_by_date.parquet", "recovery_episodes.parquet",
        "canonical_roster.parquet", "feature_manifest.csv",
        "micro_manifest.json", "match_audit.csv",
    ], {"rows": len(game), "matches": int(game["match_id"].nunique())})
    print(f"Built {len(game):,} eligible deliveries from the six raw files.")
    return game


def checkpoint_checklist(*items, status="PASS"):
    for key in items:
        CHECKLIST_PROGRESS[key] = status
    write_csv(OUTPUT_DIR / "checklist.csv", pd.DataFrame([
        {"item": key, "requirement": label, "status": CHECKLIST_PROGRESS[key]}
        for key, label in CHECKLIST.items()
    ]))


def compile_from_inputs():
    np.random.seed(SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    paths = discover_raw(INPUT_ROOT)
    raw = raw_manifest(paths)
    write_json(OUTPUT_DIR / "raw_manifest.json", raw)
    checkpoint_checklist()
    data_key = signature({
        "raw": {name: entry["sha256"] for name, entry in raw.items()},
        "recipe": DATA_CODE, "derivation": DERIVATION, "recent": RECENT_MATCHES,
        "scope": [ACCEPT_INFERRED_OTHER_AS_NO_BALL, INCLUDE_OTHER_T20_COMPETITIONS],
    })
    game = load_feature_game(paths, data_key)
    checkpoint_checklist("raw_inputs")
    game, bands, blocks = compile_game_scenarios(game)
    checkpoint_checklist("scenario_data")
    print(f"Compiled 50 batting and 50 bowling scenarios from {len(game):,} deliveries.")
    return {"game": game, "bands": bands, "blocks": blocks}

In [ ]:
run_comparison()